# Project Mythos Standalone Kaggle Pipeline

Upload this `.ipynb` by itself. It embeds the current `src/mythos` package, writes it into `/kaggle/working/project_mythos_embedded/src`, then runs the plan-aligned ARC pipeline and writes `/kaggle/working/submission.json`.

Default mode is `pipeline` + `fallback`, with internet downloads disabled. It autodiscovers pre-staged Kaggle inputs when present and otherwise uses explicit fallback adapters for missing model stages.

## 1. Bootstrap Embedded Project Mythos Code

In [ ]:
from pathlib import Path
import os
import sys

EMBEDDED_FILES = {'src/mythos/__init__.py': '"""Project Mythos ARC testing harness."""\n\nfrom mythos.arc import ArcExample, ArcTask, ArcValidationError, load_challenges\nfrom mythos.submission import Prediction, TestPrediction\n\n__all__ = [\n    "ArcExample",\n    "ArcTask",\n    "ArcValidationError",\n    "Prediction",\n    "TestPrediction",\n    "load_challenges",\n]\n', 'src/mythos/__main__.py': '"""Top-level module help for `python -m mythos`."""\n\nfrom __future__ import annotations\n\n\ndef main() -> int:\n    print(\n        "Project Mythos commands:\\n"\n        "  python -m mythos.validate data/toy/challenges.json\\n"\n        "  python -m mythos.solve --solver fixture --challenges data/toy/challenges.json --out runs/submission.json\\n"\n        "  python -m mythos.score --pred runs/submission.json --solutions data/toy/solutions.json\\n"\n        "  python -m mythos.hrm_smoke --task data/toy/challenges.json"\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/arc.py': '"""ARC JSON loading and validation."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Optional, Tuple\n\nGrid = List[List[int]]\nSolutionMap = Dict[str, Tuple[Grid, ...]]\n\nMAX_GRID_ROWS = 30\nMAX_GRID_COLS = 30\nMIN_CELL_VALUE = 0\nMAX_CELL_VALUE = 9\n\n\nclass ArcValidationError(ValueError):\n    """Raised when ARC-style data is malformed."""\n\n\n@dataclass(frozen=True)\nclass ArcExample:\n    input: Grid\n    output: Optional[Grid] = None\n\n\n@dataclass(frozen=True)\nclass ArcTask:\n    id: str\n    train: Tuple[ArcExample, ...]\n    test: Tuple[ArcExample, ...]\n\n\ndef _json_load(path: str | Path) -> Any:\n    file_path = Path(path)\n    try:\n        with file_path.open("r", encoding="utf-8") as handle:\n            return json.load(handle)\n    except json.JSONDecodeError as exc:\n        raise ArcValidationError(f"{file_path} is not valid JSON: {exc}") from exc\n\n\ndef validate_grid(value: Any, *, field: str = "grid") -> Grid:\n    """Validate and copy an ARC grid."""\n\n    if not isinstance(value, list) or not value:\n        raise ArcValidationError(f"{field} must be a non-empty list of rows")\n    if len(value) > MAX_GRID_ROWS:\n        raise ArcValidationError(f"{field} has {len(value)} rows; max is {MAX_GRID_ROWS}")\n\n    rows: Grid = []\n    expected_width: int | None = None\n    for row_idx, row in enumerate(value):\n        if not isinstance(row, list) or not row:\n            raise ArcValidationError(f"{field}[{row_idx}] must be a non-empty list")\n        if expected_width is None:\n            expected_width = len(row)\n            if expected_width > MAX_GRID_COLS:\n                raise ArcValidationError(\n                    f"{field} has {expected_width} columns; max is {MAX_GRID_COLS}"\n                )\n        elif len(row) != expected_width:\n            raise ArcValidationError(\n                f"{field} must be rectangular; row 0 has {expected_width} columns "\n                f"but row {row_idx} has {len(row)}"\n            )\n\n        copied_row: List[int] = []\n        for col_idx, cell in enumerate(row):\n            if isinstance(cell, bool) or not isinstance(cell, int):\n                raise ArcValidationError(f"{field}[{row_idx}][{col_idx}] must be an integer")\n            if cell < MIN_CELL_VALUE or cell > MAX_CELL_VALUE:\n                raise ArcValidationError(\n                    f"{field}[{row_idx}][{col_idx}]={cell}; expected 0..9"\n                )\n            copied_row.append(cell)\n        rows.append(copied_row)\n    return rows\n\n\ndef _parse_example(raw: Any, *, task_id: str, split: str, index: int, require_output: bool) -> ArcExample:\n    if not isinstance(raw, Mapping):\n        raise ArcValidationError(f"{task_id}.{split}[{index}] must be an object")\n    if "input" not in raw:\n        raise ArcValidationError(f"{task_id}.{split}[{index}] is missing input")\n    output = raw.get("output")\n    if require_output and output is None:\n        raise ArcValidationError(f"{task_id}.{split}[{index}] is missing output")\n    return ArcExample(\n        input=validate_grid(raw["input"], field=f"{task_id}.{split}[{index}].input"),\n        output=validate_grid(output, field=f"{task_id}.{split}[{index}].output")\n        if output is not None\n        else None,\n    )\n\n\ndef _parse_examples(\n    raw_examples: Any, *, task_id: str, split: str, require_output: bool\n) -> Tuple[ArcExample, ...]:\n    if not isinstance(raw_examples, list) or not raw_examples:\n        raise ArcValidationError(f"{task_id}.{split} must be a non-empty list")\n    return tuple(\n        _parse_example(\n            raw_example,\n            task_id=task_id,\n            split=split,\n            index=index,\n            require_output=require_output,\n        )\n        for index, raw_example in enumerate(raw_examples)\n    )\n\n\ndef parse_task(task_id: str, raw_task: Any) -> ArcTask:\n    if not isinstance(raw_task, Mapping):\n        raise ArcValidationError(f"{task_id} must be an object")\n    if "train" not in raw_task or "test" not in raw_task:\n        raise ArcValidationError(f"{task_id} must contain train and test splits")\n    return ArcTask(\n        id=task_id,\n        train=_parse_examples(raw_task["train"], task_id=task_id, split="train", require_output=True),\n        test=_parse_examples(raw_task["test"], task_id=task_id, split="test", require_output=False),\n    )\n\n\ndef load_challenges(path: str | Path) -> Dict[str, ArcTask]:\n    """Load an ARC challenge JSON file keyed by task id."""\n\n    raw = _json_load(path)\n    if not isinstance(raw, Mapping) or not raw:\n        raise ArcValidationError("challenge file must be a non-empty object keyed by task id")\n    return {str(task_id): parse_task(str(task_id), raw_task) for task_id, raw_task in raw.items()}\n\n\ndef grid_shape(grid: Grid) -> Tuple[int, int]:\n    return len(grid), len(grid[0])\n\n\ndef grid_equal(left: Grid, right: Grid) -> bool:\n    return left == right\n\n\ndef copy_grid(grid: Grid) -> Grid:\n    return [row[:] for row in grid]\n\n\ndef _looks_like_grid(value: Any) -> bool:\n    try:\n        validate_grid(value)\n    except ArcValidationError:\n        return False\n    return True\n\n\ndef _solution_grids_from_value(task_id: str, value: Any) -> Tuple[Grid, ...]:\n    if isinstance(value, Mapping):\n        if "test" in value:\n            return tuple(\n                validate_grid(item["output"], field=f"{task_id}.test[{idx}].output")\n                for idx, item in enumerate(value["test"])\n                if isinstance(item, Mapping) and "output" in item\n            )\n        if "output" in value:\n            return (validate_grid(value["output"], field=f"{task_id}.output"),)\n    if _looks_like_grid(value):\n        return (validate_grid(value, field=f"{task_id}.output"),)\n    if isinstance(value, list):\n        grids: List[Grid] = []\n        for idx, item in enumerate(value):\n            if isinstance(item, Mapping) and "output" in item:\n                grids.append(validate_grid(item["output"], field=f"{task_id}[{idx}].output"))\n            elif _looks_like_grid(item):\n                grids.append(validate_grid(item, field=f"{task_id}[{idx}]"))\n            else:\n                raise ArcValidationError(f"{task_id}[{idx}] is not a solution grid")\n        if grids:\n            return tuple(grids)\n    raise ArcValidationError(f"{task_id} does not contain solution outputs")\n\n\ndef load_solutions(path: str | Path) -> SolutionMap:\n    raw = _json_load(path)\n    if not isinstance(raw, Mapping) or not raw:\n        raise ArcValidationError("solution file must be a non-empty object keyed by task id")\n    return {\n        str(task_id): _solution_grids_from_value(str(task_id), value)\n        for task_id, value in raw.items()\n    }\n\n\ndef attach_solutions(tasks: Mapping[str, ArcTask], solutions: SolutionMap) -> Dict[str, ArcTask]:\n    """Return tasks with test outputs filled from a solution map."""\n\n    attached: Dict[str, ArcTask] = {}\n    for task_id, task in tasks.items():\n        if task_id not in solutions:\n            raise ArcValidationError(f"missing solutions for task {task_id}")\n        if len(solutions[task_id]) != len(task.test):\n            raise ArcValidationError(\n                f"{task_id} has {len(task.test)} test items but "\n                f"{len(solutions[task_id])} solution outputs"\n            )\n        attached[task_id] = ArcTask(\n            id=task.id,\n            train=task.train,\n            test=tuple(\n                ArcExample(input=example.input, output=solutions[task_id][index])\n                for index, example in enumerate(task.test)\n            ),\n        )\n    return attached\n\n\ndef require_test_outputs(tasks: Iterable[ArcTask]) -> None:\n    for task in tasks:\n        for index, example in enumerate(task.test):\n            if example.output is None:\n                raise ArcValidationError(f"{task.id}.test[{index}] is missing output")\n', 'src/mythos/augment.py': '"""Grid geometric augmentation: the 8 dihedral-group transforms and their inverses.\n\nUsed for inference-time ensembling: solving a task in several different\norientations and voting across the un-augmented predictions squeezes more\naccuracy out of an already-adapted model, independent of TTT quality.\nTransforming demo pairs and the test input *together* (not just the test\ninput alone) keeps any orientation-dependent rule internally consistent\nwithin its own augmented frame -- TTT refits on the transformed demo pairs,\nso it learns the same relative rule, just viewed from a different frame.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Callable\n\nfrom mythos.arc import ArcExample, ArcTask, Grid\n\nGridTransform = Callable[[Grid], Grid]\n\n\ndef _identity(grid: Grid) -> Grid:\n    return [row[:] for row in grid]\n\n\ndef _rotate90(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid[::-1])]\n\n\ndef _rotate180(grid: Grid) -> Grid:\n    return [row[::-1] for row in grid[::-1]]\n\n\ndef _rotate270(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid)][::-1]\n\n\ndef _mirror_horizontal(grid: Grid) -> Grid:\n    return [row[::-1] for row in grid]\n\n\ndef _mirror_vertical(grid: Grid) -> Grid:\n    return [row[:] for row in grid[::-1]]\n\n\ndef _transpose(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid)]\n\n\ndef _anti_transpose(grid: Grid) -> Grid:\n    return _rotate180(_transpose(grid))\n\n\n# index -> (name, forward, inverse). rotate90/rotate270 invert each other;\n# every other transform is its own inverse (an involution).\n_TRANSFORMS: tuple[tuple[str, GridTransform, GridTransform], ...] = (\n    ("identity", _identity, _identity),\n    ("rotate90", _rotate90, _rotate270),\n    ("rotate180", _rotate180, _rotate180),\n    ("rotate270", _rotate270, _rotate90),\n    ("mirror_horizontal", _mirror_horizontal, _mirror_horizontal),\n    ("mirror_vertical", _mirror_vertical, _mirror_vertical),\n    ("transpose", _transpose, _transpose),\n    ("anti_transpose", _anti_transpose, _anti_transpose),\n)\n\nNUM_TRANSFORMS = len(_TRANSFORMS)\n\n\ndef transform_name(index: int) -> str:\n    return _TRANSFORMS[index][0]\n\n\ndef forward_transform(index: int, grid: Grid) -> Grid:\n    return _TRANSFORMS[index][1](grid)\n\n\ndef inverse_transform(index: int, grid: Grid) -> Grid:\n    return _TRANSFORMS[index][2](grid)\n\n\ndef transform_task(task: ArcTask, index: int, *, id_suffix: str) -> ArcTask:\n    """Apply one dihedral transform to every grid in a task (demos and test).\n\n    Both the input and output of every train example are transformed the\n    same way, so any transformation rule the task encodes remains valid\n    (and re-learnable via TTT) within the transformed frame.\n    """\n\n    forward = _TRANSFORMS[index][1]\n\n    def transform_example(example: ArcExample) -> ArcExample:\n        return ArcExample(\n            input=forward(example.input),\n            output=forward(example.output) if example.output is not None else None,\n        )\n\n    return ArcTask(\n        id=f"{task.id}{id_suffix}",\n        train=tuple(transform_example(example) for example in task.train),\n        test=tuple(transform_example(example) for example in task.test),\n    )\n', 'src/mythos/features.py': '"""Deterministic ARC feature encoders shared by training and pipeline stages."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom math import sqrt\nfrom typing import Iterable, Iterator, Sequence\n\nfrom mythos.arc import ArcTask, Grid\n\nARC_MAX_SIZE = 30\nARC_NUM_COLORS = 10\nDEFAULT_JEPA_FEATURE_DIM = 1280\nDEFAULT_HRM_FEATURE_DIM = 768\nDEFAULT_RULE_DIM = 4\n\n\n@dataclass(frozen=True)\nclass GridPair:\n    task_id: str\n    index: int\n    input: Grid\n    output: Grid\n\n\ndef iter_train_grid_pairs(tasks: Iterable[ArcTask]) -> Iterator[GridPair]:\n    """Yield supervised input/output pairs from ARC train examples."""\n\n    for task in tasks:\n        for index, example in enumerate(task.train):\n            if example.output is None:\n                continue\n            yield GridPair(\n                task_id=task.id,\n                index=index,\n                input=example.input,\n                output=example.output,\n            )\n\n\ndef iter_all_supervised_grid_pairs(tasks: Iterable[ArcTask]) -> Iterator[GridPair]:\n    """Yield train pairs plus test pairs whose labels are attached."""\n\n    for task in tasks:\n        yield from iter_train_grid_pairs((task,))\n        for index, example in enumerate(task.test):\n            if example.output is None:\n                continue\n            yield GridPair(\n                task_id=task.id,\n                index=index,\n                input=example.input,\n                output=example.output,\n            )\n\n\ndef grid_to_feature_vector(grid: Grid, dim: int = DEFAULT_JEPA_FEATURE_DIM) -> tuple[float, ...]:\n    """Encode a grid as a fixed-length numeric vector.\n\n    This is intentionally deterministic and dependency-free. On Kaggle it acts\n    as the local ARC grid-to-token bridge when the external I-JEPA model cannot\n    be invoked directly, and as a stable target for projection/world-model smoke\n    training.\n    """\n\n    if dim <= 0:\n        raise ValueError("feature dimension must be positive")\n\n    values = [0.0 for _ in range(dim)]\n    height = len(grid)\n    width = len(grid[0])\n    area = float(height * width)\n    flat = [cell for row in grid for cell in row]\n    counts = Counter(flat)\n\n    def add(index: int, value: float) -> None:\n        if 0 <= index < dim:\n            values[index] += float(value)\n\n    add(0, height / ARC_MAX_SIZE)\n    add(1, width / ARC_MAX_SIZE)\n    add(2, area / float(ARC_MAX_SIZE * ARC_MAX_SIZE))\n    add(3, sum(1 for cell in flat if cell != 0) / area)\n    add(4, sum(flat) / (9.0 * area))\n\n    for color in range(ARC_NUM_COLORS):\n        add(8 + color, counts.get(color, 0) / area)\n\n    row_offset = 32\n    for row_index in range(ARC_MAX_SIZE):\n        if row_index < height:\n            row = grid[row_index]\n            row_area = float(width)\n            add(row_offset + row_index, sum(row) / (9.0 * row_area))\n            add(row_offset + ARC_MAX_SIZE + row_index, sum(1 for cell in row if cell != 0) / row_area)\n\n    col_offset = row_offset + 2 * ARC_MAX_SIZE\n    for col_index in range(ARC_MAX_SIZE):\n        if col_index < width:\n            column = [grid[row_index][col_index] for row_index in range(height)]\n            col_area = float(height)\n            add(col_offset + col_index, sum(column) / (9.0 * col_area))\n            add(col_offset + ARC_MAX_SIZE + col_index, sum(1 for cell in column if cell != 0) / col_area)\n\n    flat_offset = col_offset + 2 * ARC_MAX_SIZE\n    hash_offset = max(flat_offset, int(dim * 0.875))\n    flat_capacity = max(0, min(hash_offset, dim) - flat_offset)\n    for row_index in range(ARC_MAX_SIZE):\n        for col_index in range(ARC_MAX_SIZE):\n            flat_index = row_index * ARC_MAX_SIZE + col_index\n            if flat_index >= flat_capacity:\n                break\n            target_index = flat_offset + flat_index\n            if row_index < height and col_index < width:\n                add(target_index, (grid[row_index][col_index] + 1) / 10.0)\n            else:\n                add(target_index, 0.0)\n\n    hash_dim = dim - hash_offset\n    if hash_dim > 0:\n        for row_index, row in enumerate(grid):\n            for col_index, color in enumerate(row):\n                base = row_index * 1009 + col_index * 917 + color * 613\n                magnitude = ((color + 1) / 10.0) * (1.0 + (row_index + col_index) / 60.0)\n                for salt in range(4):\n                    slot = hash_offset + ((base + salt * 193) % hash_dim)\n                    sign = 1.0 if ((base + salt * 389) % 2 == 0) else -1.0\n                    add(slot, sign * magnitude)\n\n    return _l2_normalize(values)\n\n\ndef task_rule_vector(task: ArcTask, dim: int = DEFAULT_RULE_DIM) -> tuple[float, ...]:\n    """Summarize the demonstrated transformation as a fixed-length rule vector."""\n\n    if dim <= 0:\n        raise ValueError("rule dimension must be positive")\n\n    shape_deltas: list[tuple[int, int]] = []\n    density_deltas: list[float] = []\n    color_overlaps: list[float] = []\n    color_shift_votes: list[float] = []\n\n    for example in task.train:\n        if example.output is None:\n            continue\n        in_h, in_w = len(example.input), len(example.input[0])\n        out_h, out_w = len(example.output), len(example.output[0])\n        shape_deltas.append((out_h - in_h, out_w - in_w))\n        density_deltas.append(_density(example.output) - _density(example.input))\n        color_overlaps.append(_color_jaccard(example.input, example.output))\n        color_shift_votes.append(_dominant_color(example.output) - _dominant_color(example.input))\n\n    base = [\n        _average(delta[0] for delta in shape_deltas) / ARC_MAX_SIZE,\n        _average(delta[1] for delta in shape_deltas) / ARC_MAX_SIZE,\n        _average(density_deltas),\n        _average(color_overlaps),\n        _average(color_shift_votes) / 9.0,\n    ]\n    if dim <= len(base):\n        return tuple(round(value, 6) for value in base[:dim])\n\n    values = [0.0 for _ in range(dim)]\n    for index, value in enumerate(base):\n        values[index] = value\n    for pair_index, example in enumerate(task.train):\n        if example.output is None:\n            continue\n        for grid_index, grid in enumerate((example.input, example.output)):\n            for color, count in Counter(cell for row in grid for cell in row).items():\n                slot = len(base) + ((pair_index * 97 + grid_index * 31 + color * 17) % (dim - len(base)))\n                values[slot] += count / 900.0\n    return tuple(round(value, 6) for value in values)\n\n\ndef grid_to_hrm_sequence(grid: Grid, *, max_size: int = ARC_MAX_SIZE) -> tuple[int, ...]:\n    """Encode a grid using HRM ARC token conventions: PAD=0, EOS=1, colors=2..11."""\n\n    height = len(grid)\n    width = len(grid[0])\n    if height > max_size or width > max_size:\n        raise ValueError(f"grid shape {height}x{width} exceeds {max_size}x{max_size}")\n    tokens = [[0 for _ in range(max_size)] for _ in range(max_size)]\n    for row_index, row in enumerate(grid):\n        for col_index, color in enumerate(row):\n            tokens[row_index][col_index] = color + 2\n    if height < max_size:\n        for col_index in range(width):\n            tokens[height][col_index] = 1\n    if width < max_size:\n        for row_index in range(height):\n            tokens[row_index][width] = 1\n    return tuple(token for row in tokens for token in row)\n\n\ndef hrm_sequence_to_grid(\n    sequence: Sequence[int],\n    *,\n    shape_hint: tuple[int, int] | None = None,\n    max_size: int = ARC_MAX_SIZE,\n) -> Grid:\n    """Decode a HRM ARC token sequence into a valid 0..9 grid."""\n\n    if len(sequence) != max_size * max_size:\n        raise ValueError(f"expected {max_size * max_size} HRM tokens, got {len(sequence)}")\n    matrix = [\n        [int(sequence[row * max_size + col]) for col in range(max_size)]\n        for row in range(max_size)\n    ]\n    if shape_hint is None:\n        shape_hint = _infer_shape_from_eos(matrix, max_size=max_size)\n    height = min(max(1, shape_hint[0]), max_size)\n    width = min(max(1, shape_hint[1]), max_size)\n    return [\n        [_token_to_color(matrix[row][col]) for col in range(width)]\n        for row in range(height)\n    ]\n\n\ndef output_shape_hint(task: ArcTask, input_grid: Grid) -> tuple[int, int] | None:\n    """Choose an output shape only when train examples agree on one.\n\n    When train output shapes disagree (common for crop/symmetry-repair\n    tasks, where the output size is the bounding box of some occluded\n    region and varies per example), there is no safe static guess -- return\n    None so the caller falls back to the model\'s own predicted EOS boundary\n    markers (see `_infer_shape_from_eos`) instead of forcing the full input\n    grid\'s shape, which is virtually never correct for this task family and\n    makes exact-match scoring impossible regardless of prediction quality.\n    """\n\n    shapes = {\n        (len(example.output), len(example.output[0]))\n        for example in task.train\n        if example.output is not None\n    }\n    if len(shapes) == 1:\n        return next(iter(shapes))\n    return None\n\n\ndef embedding_cosine_similarity(left: Sequence[float], right: Sequence[float]) -> float:\n    numerator = sum(a * b for a, b in zip(left, right))\n    left_norm = sqrt(sum(a * a for a in left))\n    right_norm = sqrt(sum(b * b for b in right))\n    if left_norm == 0.0 or right_norm == 0.0:\n        return 0.0\n    return numerator / (left_norm * right_norm)\n\n\ndef max_pairwise_cosine(vectors: Sequence[Sequence[float]]) -> float:\n    if len(vectors) < 2:\n        return 0.0\n    max_value = -1.0\n    for left_index, left in enumerate(vectors):\n        for right in vectors[left_index + 1 :]:\n            max_value = max(max_value, embedding_cosine_similarity(left, right))\n    return max_value\n\n\ndef _infer_shape_from_eos(matrix: Sequence[Sequence[int]], *, max_size: int) -> tuple[int, int]:\n    row_scores = [\n        sum(1 for value in matrix[row_index] if value == 1)\n        for row_index in range(max_size)\n    ]\n    col_scores = [\n        sum(1 for row_index in range(max_size) if matrix[row_index][col_index] == 1)\n        for col_index in range(max_size)\n    ]\n    height = _best_boundary_index(row_scores, max_size)\n    width = _best_boundary_index(col_scores, max_size)\n    return max(1, height), max(1, width)\n\n\ndef _best_boundary_index(scores: Sequence[int], default: int, *, min_score: int = 2) -> int:\n    """Pick the index most likely to be the EOS boundary marker.\n\n    Per `grid_to_hrm_sequence`, every in-grid row/column already carries\n    exactly one legitimate stray EOS(=1) token from the *orthogonal*\n    boundary marker (the EOS column marks every content row; the EOS row\n    marks every content column). A low fixed threshold is therefore\n    trivially false-triggered by that baseline plus a single unit of\n    prediction noise on some earlier row/column. The genuine boundary\n    instead carries a full run of EOS tokens -- one per in-grid cell along\n    that axis -- so picking the argmax (not the first index crossing a low\n    threshold) is far more robust to noisy raw predictions.\n    """\n\n    best_index = default\n    best_score = min_score - 1\n    for index, score in enumerate(scores):\n        if score > best_score:\n            best_score = score\n            best_index = index\n    return best_index if best_score >= min_score else default\n\n\ndef _token_to_color(token: int) -> int:\n    if token <= 1:\n        return 0\n    return min(max(token - 2, 0), 9)\n\n\ndef _density(grid: Grid) -> float:\n    flat = [cell for row in grid for cell in row]\n    return sum(1 for cell in flat if cell != 0) / float(len(flat))\n\n\ndef _dominant_color(grid: Grid) -> int:\n    counts = Counter(cell for row in grid for cell in row)\n    return counts.most_common(1)[0][0]\n\n\ndef _color_jaccard(left: Grid, right: Grid) -> float:\n    left_colors = {cell for row in left for cell in row}\n    right_colors = {cell for row in right for cell in row}\n    union = left_colors | right_colors\n    if not union:\n        return 1.0\n    return len(left_colors & right_colors) / len(union)\n\n\ndef _average(values: Iterable[float]) -> float:\n    collected = list(values)\n    return sum(collected) / len(collected) if collected else 0.0\n\n\ndef _l2_normalize(values: Sequence[float]) -> tuple[float, ...]:\n    norm = sqrt(sum(value * value for value in values))\n    if norm == 0.0:\n        return tuple(round(value, 6) for value in values)\n    return tuple(round(value / norm, 6) for value in values)\n', 'src/mythos/hrm_dataset.py': '"""Dataset-preparation glue for the external HRM checkout."""\n\nfrom __future__ import annotations\n\nimport json\nfrom pathlib import Path\nimport shutil\nimport subprocess\nimport sys\nfrom typing import Iterable\n\nfrom mythos.arc import ArcTask, ArcValidationError, Grid, require_test_outputs\n\n\ndef default_run_dir() -> Path:\n    import os\n\n    # Must be absolute: build_hrm_dataset() invokes HRM\'s dataset builder with\n    # cwd=repo_dir (the external HRM checkout), while the caller that later reads\n    # the built dataset back (HRMInferenceRunner._run_external_evaluate, running in\n    # the notebook/CLI\'s own process) has a different cwd. A relative path here\n    # resolves to two different real locations across those processes -- confirmed\n    # by a real run: the builder wrote train/dataset.json under repo_dir, but the\n    # dataloader looked for it relative to the notebook\'s cwd and got FileNotFoundError.\n    return Path(os.environ.get("MYTHOS_RUN_DIR", "runs")).resolve()\n\n\ndef prepare_hrm_raw_dataset(\n    tasks: Iterable[ArcTask],\n    output_dir: str | Path,\n    *,\n    allow_dummy_test_outputs: bool = False,\n) -> Path:\n    """Write tasks into the directory shape HRM\'s ARC dataset builder expects."""\n\n    task_list = list(tasks)\n    if not allow_dummy_test_outputs:\n        require_test_outputs(task_list)\n\n    raw_data_dir = Path(output_dir)\n    eval_dir = raw_data_dir / "evaluation"\n    eval_dir.mkdir(parents=True, exist_ok=True)\n\n    for task in task_list:\n        raw_task = {\n            "train": [\n                {"input": example.input, "output": example.output}\n                for example in task.train\n            ],\n            "test": [\n                {\n                    "input": example.input,\n                    "output": example.output\n                    if example.output is not None\n                    else _dummy_output_like(example.input),\n                }\n                for example in task.test\n            ],\n        }\n        with (eval_dir / f"{task.id}.json").open("w", encoding="utf-8") as handle:\n            json.dump(raw_task, handle, indent=2)\n            handle.write("\\n")\n    return raw_data_dir\n\n\ndef _dummy_output_like(grid: Grid) -> Grid:\n    return [[0 for _ in row] for row in grid]\n\n\ndef build_hrm_dataset(\n    *,\n    hrm_repo_dir: str | Path,\n    raw_data_dir: str | Path,\n    output_dir: str | Path,\n    num_aug: int = 0,\n) -> subprocess.CompletedProcess[str]:\n    """Invoke HRM\'s own ARC dataset builder against a prepared raw-data directory."""\n\n    repo_dir = Path(hrm_repo_dir)\n    script = repo_dir / "dataset" / "build_arc_dataset.py"\n    if not script.exists():\n        raise ArcValidationError(f"HRM dataset builder not found: {script}")\n\n    output_path = Path(output_dir)\n    output_path.mkdir(parents=True, exist_ok=True)\n\n    # build_arc_dataset.py\'s DataProcessConfig.dataset_dirs is a List[str] Pydantic\n    # field defaulting to ["dataset/raw-data/ARC-AGI/data", "dataset/raw-data/ConceptARC/corpus"],\n    # resolved relative to the subprocess\'s cwd. Passing --dataset-dirs <path> on the CLI\n    # does not override this list -- verified against a real run: it silently kept\n    # scanning the unmodified default and crashed with FileNotFoundError. Sidestep the CLI\n    # entirely by giving the subprocess a writable cwd that already has the default paths\n    # satisfied, decoupled from repo_dir -- which is read-only when HRM_REPO_DIR points at\n    # a Kaggle Dataset mount, confirmed by a real run failing with OSError(30, \'Read-only\n    # file system\') when this used to write directly under repo_dir. The script itself is\n    # still invoked from its real (possibly read-only) location via an absolute path.\n    build_cwd = output_path.parent / "hrm_build_cwd"\n    default_arc_dir = build_cwd / "dataset" / "raw-data" / "ARC-AGI" / "data"\n    default_concept_dir = build_cwd / "dataset" / "raw-data" / "ConceptARC" / "corpus"\n    if default_arc_dir.is_symlink() or default_arc_dir.is_file():\n        default_arc_dir.unlink()\n    elif default_arc_dir.exists():\n        shutil.rmtree(default_arc_dir)\n    default_arc_dir.parent.mkdir(parents=True, exist_ok=True)\n    shutil.copytree(Path(raw_data_dir), default_arc_dir)\n    default_concept_dir.mkdir(parents=True, exist_ok=True)\n\n    command = [\n        sys.executable,\n        str(script.resolve()),\n        "--output-dir",\n        str(output_path),\n        "--num-aug",\n        str(num_aug),\n    ]\n    return subprocess.run(\n        command,\n        cwd=build_cwd,\n        check=True,\n        capture_output=True,\n        text=True,\n    )\n', 'src/mythos/hrm_smoke.py': '"""CLI smoke test for the external HRM runtime."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\nimport time\nfrom pathlib import Path\n\nfrom mythos.arc import ArcValidationError, attach_solutions, load_challenges, load_solutions\nfrom mythos.hrm_dataset import build_hrm_dataset, default_run_dir, prepare_hrm_raw_dataset\nfrom mythos.solvers.hrm import HRMEnvironment, HRMEnvironmentError\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Smoke-test external HRM integration.")\n    parser.add_argument("--task", required=True, help="ARC-style challenge JSON for the smoke run.")\n    parser.add_argument("--solutions", help="Optional solution JSON if --task omits test outputs.")\n    parser.add_argument("--run-dir", default=None, help="Output directory for smoke artifacts.")\n    parser.add_argument("--num-aug", type=int, default=0, help="HRM dataset-builder augmentation count.")\n    parser.add_argument(\n        "--skip-dataset-build",\n        action="store_true",\n        help="Only write the HRM raw-data layout; do not invoke HRM\'s dataset builder.",\n    )\n    args = parser.parse_args(argv)\n\n    started = time.perf_counter()\n    try:\n        tasks = load_challenges(args.task)\n        if args.solutions:\n            tasks = attach_solutions(tasks, load_solutions(args.solutions))\n\n        env = HRMEnvironment.from_env()\n        env.validate(require_cuda=True)\n        modules = env.import_modules()\n\n        torch = HRMEnvironment._import_torch()\n        torch.cuda.reset_peak_memory_stats()\n        checkpoint = env.load_checkpoint()\n\n        run_dir = Path(args.run_dir) if args.run_dir else default_run_dir() / "hrm_smoke"\n        raw_dir = prepare_hrm_raw_dataset(tasks.values(), run_dir / "raw" / "ARC-AGI-2" / "data")\n\n        dataset_build = None\n        if not args.skip_dataset_build:\n            result = build_hrm_dataset(\n                hrm_repo_dir=env.repo_dir,\n                raw_data_dir=raw_dir,\n                output_dir=run_dir / "data" / "arc-2-smoke",\n                num_aug=args.num_aug,\n            )\n            dataset_build = {\n                "returncode": result.returncode,\n                "stdout_tail": result.stdout[-2000:],\n                "stderr_tail": result.stderr[-2000:],\n            }\n    except (ArcValidationError, HRMEnvironmentError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    summary = {\n        "tasks": len(tasks),\n        "hrm_repo_dir": str(env.repo_dir),\n        "checkpoint_path": str(env.checkpoint_path),\n        "checkpoint_type": type(checkpoint).__name__,\n        "imported_modules": sorted(modules),\n        "raw_data_dir": str(raw_dir),\n        "dataset_build": dataset_build,\n        "elapsed_seconds": round(time.perf_counter() - started, 3),\n        "cuda_peak_memory_bytes": int(torch.cuda.max_memory_allocated()),\n    }\n    print(json.dumps(summary, indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/jepa_encoder.py': '"""Optional transformers-native I-JEPA image encoder for ARC grids."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom mythos.arc import Grid\n\nARC_PALETTE: dict[int, tuple[int, int, int]] = {\n    0: (0, 0, 0),\n    1: (0, 116, 217),\n    2: (255, 65, 54),\n    3: (46, 204, 64),\n    4: (255, 220, 0),\n    5: (170, 170, 170),\n    6: (240, 18, 190),\n    7: (255, 133, 27),\n    8: (127, 219, 255),\n    9: (135, 12, 37),\n}\n\n\nclass JepaEncodingError(RuntimeError):\n    """Raised when optional I-JEPA image encoding cannot run."""\n\n\ndef grid_to_rgb_image(grid: Grid, *, cell_px: int = 8, output_size: int = 224):\n    """Rasterize an ARC grid to the RGB image shape expected by ViT-H/14."""\n\n    try:\n        from PIL import Image\n    except Exception as exc:  # pragma: no cover - optional dependency.\n        raise JepaEncodingError("Pillow is required for real I-JEPA grid rasterization") from exc\n\n    if cell_px <= 0:\n        raise ValueError("cell_px must be positive")\n    height = len(grid)\n    width = len(grid[0])\n    image = Image.new("RGB", (width * cell_px, height * cell_px))\n    for row_index, row in enumerate(grid):\n        for col_index, color in enumerate(row):\n            image.paste(\n                ARC_PALETTE[int(color)],\n                (\n                    col_index * cell_px,\n                    row_index * cell_px,\n                    (col_index + 1) * cell_px,\n                    (row_index + 1) * cell_px,\n                ),\n            )\n    resampling = getattr(Image, "Resampling", Image)\n    return image.resize((output_size, output_size), resampling.NEAREST)\n\n\n@dataclass\nclass JepaImageEncoder:\n    """Frozen transformers I-JEPA feature extractor."""\n\n    model_root: Path\n    device: str\n    processor: object\n    model: object\n\n    @classmethod\n    def from_path(cls, model_path: str | Path, *, device: str | None = None) -> "JepaImageEncoder":\n        try:\n            import torch\n            from transformers import AutoModel, AutoProcessor\n        except Exception as exc:  # pragma: no cover - optional dependency.\n            raise JepaEncodingError("transformers, torch, and Pillow are required for real I-JEPA") from exc\n\n        root = _model_root(model_path)\n        selected_device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        try:\n            processor = AutoProcessor.from_pretrained(root, local_files_only=True)\n            model = AutoModel.from_pretrained(root, local_files_only=True).to(selected_device)\n            model.eval()\n            for parameter in model.parameters():\n                parameter.requires_grad = False\n        except Exception as exc:  # pragma: no cover - depends on external model files.\n            raise JepaEncodingError(f"failed to load transformers I-JEPA model from {root}: {exc}") from exc\n        return cls(model_root=root, device=selected_device, processor=processor, model=model)\n\n    def encode_grid(self, grid: Grid) -> tuple[float, ...]:\n        try:\n            import torch\n        except Exception as exc:  # pragma: no cover - optional dependency.\n            raise JepaEncodingError("torch is required for real I-JEPA") from exc\n\n        image = grid_to_rgb_image(grid)\n        encoded = self.processor(images=image, return_tensors="pt")\n        encoded = {\n            key: value.to(self.device) if hasattr(value, "to") else value\n            for key, value in encoded.items()\n        }\n        with torch.no_grad():\n            outputs = self.model(**encoded)\n        hidden = getattr(outputs, "last_hidden_state", None)\n        if hidden is None:\n            raise JepaEncodingError("I-JEPA model output did not include last_hidden_state")\n        pooled = hidden.mean(dim=1)[0].detach().float().cpu().tolist()\n        return tuple(round(float(value), 6) for value in pooled)\n\n\ndef _model_root(model_path: str | Path) -> Path:\n    path = Path(model_path)\n    return path.parent if path.is_file() else path\n', 'src/mythos/kaggle_models.py': '"""Kaggle model input auto-discovery.\n\nKaggle submissions usually mount model code and checkpoints under\n`/kaggle/input/<dataset-name>/...`. This module scans those inputs and sets the\nenvironment variables consumed by `mythos.models.ModelRegistry`.\n"""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\nimport os\nimport subprocess\nimport urllib.request\nfrom typing import Iterable\n\n\nCHECKPOINT_SUFFIXES = (".pt", ".pth", ".ckpt", ".bin", ".safetensors", ".tar")\n\n\nMODEL_ENV_KEYS = (\n    "IJEPA_CHECKPOINT_PATH",\n    "IJEPA_PROJECTION_CHECKPOINT_PATH",\n    "HRM_TEXT_REPO_DIR",\n    "HRM_TEXT_CHECKPOINT_PATH",\n    "WORLD_MODEL_CHECKPOINT_PATH",\n    "TTT_LORA_CHECKPOINT_PATH",\n    "HRM_REPO_DIR",\n    "HRM_CHECKPOINT_PATH",\n)\n\nHF_MODEL_SPECS = (\n    {\n        "name": "jepa",\n        "repo_id_env": "IJEPA_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "IJEPA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "jepa_projection",\n        "repo_id_env": "IJEPA_PROJECTION_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "IJEPA_PROJECTION_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_text",\n        "repo_id_env": "HRM_TEXT_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "HRM_TEXT_CHECKPOINT_PATH",\n    },\n    {\n        "name": "world_model",\n        "repo_id_env": "WORLD_MODEL_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "WORLD_MODEL_CHECKPOINT_PATH",\n    },\n    {\n        "name": "ttt_lora",\n        "repo_id_env": "TTT_LORA_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "TTT_LORA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_l_module",\n        "repo_id_env": "HRM_HF_REPO_ID",\n        "repo_dir_env": None,\n        "checkpoint_env": "HRM_CHECKPOINT_PATH",\n    },\n)\n\nGIT_REPO_SPECS = (\n    {\n        "name": "hrm_text",\n        "url_env": "HRM_TEXT_GIT_REPO_URL",\n        "repo_dir_env": "HRM_TEXT_REPO_DIR",\n    },\n    {\n        "name": "hrm_l_module",\n        "url_env": "HRM_GIT_REPO_URL",\n        "repo_dir_env": "HRM_REPO_DIR",\n    },\n)\n\nDIRECT_CHECKPOINT_SPECS = (\n    {\n        "name": "jepa",\n        "url_env": "IJEPA_CHECKPOINT_URL",\n        "checkpoint_env": "IJEPA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "jepa_projection",\n        "url_env": "IJEPA_PROJECTION_CHECKPOINT_URL",\n        "checkpoint_env": "IJEPA_PROJECTION_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_text",\n        "url_env": "HRM_TEXT_CHECKPOINT_URL",\n        "checkpoint_env": "HRM_TEXT_CHECKPOINT_PATH",\n    },\n    {\n        "name": "world_model",\n        "url_env": "WORLD_MODEL_CHECKPOINT_URL",\n        "checkpoint_env": "WORLD_MODEL_CHECKPOINT_PATH",\n    },\n    {\n        "name": "ttt_lora",\n        "url_env": "TTT_LORA_CHECKPOINT_URL",\n        "checkpoint_env": "TTT_LORA_CHECKPOINT_PATH",\n    },\n    {\n        "name": "hrm_l_module",\n        "url_env": "HRM_CHECKPOINT_URL",\n        "checkpoint_env": "HRM_CHECKPOINT_PATH",\n    },\n)\n\n\ndef download_git_code_repositories(\n    output_root: str | Path = "/kaggle/working/model_code",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Clone configured Git repos for model code and export repo env vars."""\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "cloned": {},\n        "skipped": [],\n        "errors": {},\n    }\n    configured = [spec for spec in GIT_REPO_SPECS if os.environ.get(str(spec["url_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["url_env"]) for spec in GIT_REPO_SPECS]\n        return result\n\n    root = Path(output_root)\n    root.mkdir(parents=True, exist_ok=True)\n    for spec in configured:\n        name = str(spec["name"])\n        url = os.environ[str(spec["url_env"])]\n        repo_dir = root / name\n        try:\n            if not repo_dir.exists():\n                subprocess.run(\n                    ["git", "clone", "--depth", "1", url, str(repo_dir)],\n                    check=True,\n                    capture_output=True,\n                    text=True,\n                )\n            if apply:\n                os.environ.setdefault(str(spec["repo_dir_env"]), str(repo_dir))\n            result["cloned"][name] = {\n                "url": url,\n                "repo_dir": str(repo_dir),\n                "repo_dir_env": spec["repo_dir_env"],\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n    return result\n\n\ndef download_direct_checkpoint_inputs(\n    output_root: str | Path = "/kaggle/working/model_inputs/direct",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Download configured direct checkpoint URLs and export checkpoint env vars."""\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "downloaded": {},\n        "skipped": [],\n        "errors": {},\n    }\n    configured = [spec for spec in DIRECT_CHECKPOINT_SPECS if os.environ.get(str(spec["url_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["url_env"]) for spec in DIRECT_CHECKPOINT_SPECS]\n        return result\n\n    root = Path(output_root)\n    root.mkdir(parents=True, exist_ok=True)\n    for spec in configured:\n        name = str(spec["name"])\n        url = os.environ[str(spec["url_env"])]\n        target = root / name / _filename_from_url(url)\n        try:\n            target.parent.mkdir(parents=True, exist_ok=True)\n            if not target.exists():\n                urllib.request.urlretrieve(url, target)\n            checkpoint_env = str(spec["checkpoint_env"])\n            if apply:\n                os.environ.setdefault(checkpoint_env, str(target))\n            result["downloaded"][name] = {\n                "url": url,\n                "checkpoint": str(target),\n                "checkpoint_env": checkpoint_env,\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n    return result\n\n\ndef download_huggingface_model_inputs(\n    output_root: str | Path = "/kaggle/working/model_inputs",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Download configured Hugging Face model repos before model loading.\n\n    Configure with env vars such as `HRM_HF_REPO_ID`. Optional env vars named\n    `<PREFIX>_HF_CHECKPOINT_GLOB` can narrow checkpoint selection, for example\n    `HRM_HF_CHECKPOINT_GLOB="*.pt"`.\n    """\n\n    result: dict[str, object] = {\n        "output_root": str(output_root),\n        "downloaded": {},\n        "skipped": [],\n        "errors": {},\n    }\n\n    configured = [spec for spec in HF_MODEL_SPECS if os.environ.get(str(spec["repo_id_env"]))]\n    if not configured:\n        result["skipped"] = [str(spec["repo_id_env"]) for spec in HF_MODEL_SPECS]\n        return result\n\n    try:\n        from huggingface_hub import snapshot_download\n    except Exception as exc:\n        result["errors"]["huggingface_hub"] = (\n            "huggingface_hub is not installed or cannot be imported: " + str(exc)\n        )\n        return result\n\n    output_dir = Path(output_root)\n    output_dir.mkdir(parents=True, exist_ok=True)\n\n    for spec in configured:\n        name = str(spec["name"])\n        repo_id_env = str(spec["repo_id_env"])\n        repo_id = os.environ[repo_id_env]\n        local_dir = output_dir / name\n        glob_env = f"{repo_id_env.removesuffix(\'_REPO_ID\')}_CHECKPOINT_GLOB"\n        checkpoint_glob = os.environ.get(glob_env)\n\n        try:\n            downloaded = Path(\n                snapshot_download(\n                    repo_id=repo_id,\n                    local_dir=local_dir,\n                    local_dir_use_symlinks=False,\n                )\n            )\n            checkpoint = _find_checkpoint_by_glob(downloaded, checkpoint_glob)\n            if checkpoint is None:\n                result["errors"][name] = (\n                    f"downloaded {repo_id} to {downloaded}, but found no checkpoint "\n                    f"matching {checkpoint_glob or CHECKPOINT_SUFFIXES}"\n                )\n                continue\n\n            repo_dir_env = spec["repo_dir_env"]\n            checkpoint_env = str(spec["checkpoint_env"])\n            if apply:\n                if repo_dir_env is not None:\n                    os.environ.setdefault(str(repo_dir_env), str(downloaded))\n                os.environ.setdefault(checkpoint_env, str(checkpoint))\n\n            result["downloaded"][name] = {\n                "repo_id": repo_id,\n                "local_dir": str(downloaded),\n                "checkpoint": str(checkpoint),\n                "repo_dir_env": repo_dir_env,\n                "checkpoint_env": checkpoint_env,\n            }\n        except Exception as exc:\n            result["errors"][name] = str(exc)\n\n    return result\n\n\ndef autodiscover_model_inputs(\n    input_root: str | Path = "/kaggle/input",\n    *,\n    apply: bool = True,\n) -> dict[str, object]:\n    """Find likely model repos/checkpoints and optionally export env vars."""\n\n    root = Path(input_root)\n    result: dict[str, object] = {\n        "input_root": str(root),\n        "exists": root.exists(),\n        "set": {},\n        "missing": [],\n    }\n    if not root.exists():\n        result["missing"] = list(MODEL_ENV_KEYS)\n        return result\n\n    discovered: dict[str, Path] = {}\n\n    hrm_repo = _find_hrm_repo(root)\n    hrm_checkpoint = _find_checkpoint(root, include=("hrm",), exclude=("text", "lora", "adapter"))\n    if hrm_repo is not None and hrm_checkpoint is not None:\n        discovered["HRM_REPO_DIR"] = hrm_repo\n        discovered["HRM_CHECKPOINT_PATH"] = hrm_checkpoint\n\n    ijepa_checkpoint = _find_checkpoint(root, include=("ijepa", "i-jepa", "jepa"), exclude=("projection",))\n    if ijepa_checkpoint is not None:\n        discovered["IJEPA_CHECKPOINT_PATH"] = ijepa_checkpoint\n\n    ijepa_projection_checkpoint = _find_checkpoint(root, include=("ijepa", "projection"), require_all=True)\n    if ijepa_projection_checkpoint is not None:\n        discovered["IJEPA_PROJECTION_CHECKPOINT_PATH"] = ijepa_projection_checkpoint\n\n    hrm_text_repo = _find_named_repo(root, names=("hrm-text", "hrm_text", "hrmtext"))\n    hrm_text_checkpoint = _find_checkpoint(root, include=("hrm", "text"), require_all=True)\n    if hrm_text_repo is not None and hrm_text_checkpoint is not None:\n        discovered["HRM_TEXT_REPO_DIR"] = hrm_text_repo\n        discovered["HRM_TEXT_CHECKPOINT_PATH"] = hrm_text_checkpoint\n\n    world_model_checkpoint = _find_checkpoint(root, include=("world", "transition"))\n    if world_model_checkpoint is not None:\n        discovered["WORLD_MODEL_CHECKPOINT_PATH"] = world_model_checkpoint\n\n    lora_checkpoint = _find_checkpoint(root, include=("lora", "adapter"))\n    if lora_checkpoint is not None:\n        discovered["TTT_LORA_CHECKPOINT_PATH"] = lora_checkpoint\n\n    set_values: dict[str, str] = {}\n    for key, path in discovered.items():\n        if key in os.environ:\n            set_values[key] = os.environ[key]\n            continue\n        if apply:\n            os.environ[key] = str(path)\n        set_values[key] = str(path)\n\n    result["set"] = set_values\n    result["missing"] = [key for key in MODEL_ENV_KEYS if key not in set_values and key not in os.environ]\n    return result\n\n\ndef _find_hrm_repo(root: Path) -> Path | None:\n    for path in _iter_dirs(root):\n        if (path / "evaluate.py").exists() and (path / "dataset" / "build_arc_dataset.py").exists():\n            return path\n    return None\n\n\ndef _find_named_repo(root: Path, *, names: Iterable[str]) -> Path | None:\n    lowered_names = tuple(name.lower() for name in names)\n    for path in _iter_dirs(root):\n        path_text = path.as_posix().lower()\n        if any(name in path_text for name in lowered_names):\n            return path\n    return None\n\n\ndef _find_checkpoint(\n    root: Path,\n    *,\n    include: Iterable[str],\n    exclude: Iterable[str] = (),\n    require_all: bool = False,\n) -> Path | None:\n    include_terms = tuple(term.lower() for term in include)\n    exclude_terms = tuple(term.lower() for term in exclude)\n    candidates = []\n    for path in root.rglob("*"):\n        if not path.is_file() or path.suffix.lower() not in CHECKPOINT_SUFFIXES:\n            continue\n        path_text = path.as_posix().lower()\n        if require_all and not all(term in path_text for term in include_terms):\n            continue\n        if not require_all and not any(term in path_text for term in include_terms):\n            continue\n        if any(term in path_text for term in exclude_terms):\n            continue\n        candidates.append(path)\n    if not candidates:\n        return None\n    return sorted(candidates, key=lambda item: (len(item.as_posix()), item.as_posix()))[0]\n\n\ndef _find_checkpoint_by_glob(root: Path, checkpoint_glob: str | None) -> Path | None:\n    if checkpoint_glob:\n        candidates = [path for path in root.rglob(checkpoint_glob) if path.is_file()]\n    else:\n        candidates = [\n            path\n            for path in root.rglob("*")\n            if path.is_file() and path.suffix.lower() in CHECKPOINT_SUFFIXES\n        ]\n    if not candidates:\n        return None\n    return sorted(candidates, key=lambda item: (len(item.as_posix()), item.as_posix()))[0]\n\n\ndef _filename_from_url(url: str) -> str:\n    filename = url.rstrip("/").split("/")[-1]\n    return filename or "checkpoint.pt"\n\n\ndef _iter_dirs(root: Path):\n    for path in root.rglob("*"):\n        if path.is_dir():\n            yield path\n', 'src/mythos/kaggle_run.py': '"""Kaggle-oriented runner for producing /kaggle/working/submission.json."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nimport traceback\n\nfrom mythos.arc import ArcTask, ArcValidationError, copy_grid, load_challenges\nfrom mythos.metrics import score_files\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.factory import make_solver\nfrom mythos.solvers.hrm import HRMEnvironment, HRMInferenceRunner, HRMTTTRunner, TTTConfig\nfrom mythos.solvers.symbolic import SymbolicSolver\nfrom mythos.submission import Prediction, write_submission\n\nDEFAULT_KAGGLE_DATA_DIR = Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2")\nDEFAULT_KAGGLE_OUTPUT = Path("/kaggle/working/submission.json")\n\nCHALLENGE_FILES = {\n    "training": ("arc-agi_training-challenges.json", "arc-agi_training_challenges.json"),\n    "evaluation": ("arc-agi_evaluation-challenges.json", "arc-agi_evaluation_challenges.json"),\n    "test": ("arc-agi_test-challenges.json", "arc-agi_test_challenges.json"),\n}\n\nSOLUTION_FILES = {\n    "training": ("arc-agi_training-solutions.json", "arc-agi_training_solutions.json"),\n    "evaluation": ("arc-agi_evaluation-solutions.json", "arc-agi_evaluation_solutions.json"),\n}\n\n\ndef resolve_challenge_path(data_dir: str | Path, split: str) -> Path:\n    return _first_existing(Path(data_dir), CHALLENGE_FILES[split])\n\n\ndef resolve_solution_path(data_dir: str | Path, split: str) -> Path | None:\n    candidates = SOLUTION_FILES.get(split)\n    if not candidates:\n        return None\n    try:\n        return _first_existing(Path(data_dir), candidates)\n    except FileNotFoundError:\n        return None\n\n\ndef _first_existing(data_dir: Path, names: tuple[str, ...]) -> Path:\n    for name in names:\n        path = data_dir / name\n        if path.exists():\n            return path\n    joined = ", ".join(names)\n    raise FileNotFoundError(f"none of these files exist in {data_dir}: {joined}")\n\n\ndef solve_with_fallback(solver, fallback_solver, task: ArcTask) -> Prediction:\n    """Solve one task, degrading to guaranteed-output fallbacks on any failure.\n\n    A Kaggle rerun must always write a submission.json even if some tasks\n    crash their primary solver (model errors, OOM, malformed grids, etc.);\n    letting one bad task abort the whole loop would zero out every other\n    already-solved task too.\n    """\n    try:\n        return solver.solve(task)\n    except Exception as exc:  # noqa: BLE001 - any solver failure must not abort the run\n        print(f"WARNING: {task.id} failed with {solver.__class__.__name__}: {exc!r}; using baseline fallback", file=sys.stderr)\n    try:\n        return fallback_solver.solve(task)\n    except Exception as exc:  # noqa: BLE001 - last-resort guarantee of a valid prediction\n        print(f"WARNING: {task.id} baseline fallback also failed: {exc!r}; using trivial prediction", file=sys.stderr)\n    attempts = [(copy_grid(example.input), [[0]]) for example in task.test]\n    return make_prediction(task, attempts)\n\n\ndef parse_ensemble_transforms(value: str) -> tuple[int, ...]:\n    """Parse a comma-separated list of dihedral transform indices, e.g. \'0,2,5\'."""\n\n    indices = tuple(int(part) for part in value.split(",") if part.strip())\n    return indices or (0,)\n\n\ndef solve_symbolic_first(tasks: list[ArcTask]) -> tuple[dict[str, Prediction], list[ArcTask]]:\n    """Try the verified symbolic solver on every task before spending neural compute.\n\n    SymbolicSolver only ever returns a prediction that exactly reproduces\n    every train pair -- it never guesses -- so this is a strictly free win\n    layered ahead of any other solver: whatever it solves is solved for\n    certain, and everything else falls through unchanged to the caller.\n    """\n\n    symbolic_solver = SymbolicSolver()\n    solved: dict[str, Prediction] = {}\n    remaining: list[ArcTask] = []\n    for task in tasks:\n        try:\n            solved[task.id] = symbolic_solver.solve(task)\n        except SolverError:\n            remaining.append(task)\n    return solved, remaining\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run Mythos against Kaggle ARC-AGI data.")\n    parser.add_argument("--data-dir", default=str(DEFAULT_KAGGLE_DATA_DIR))\n    parser.add_argument("--split", choices=sorted(CHALLENGE_FILES), default="test")\n    parser.add_argument("--challenges", help="Explicit challenge JSON path; overrides --data-dir/--split.")\n    parser.add_argument("--solutions", help="Explicit solutions JSON path for local scoring.")\n    parser.add_argument("--solver", choices=["pipeline", "baseline", "fixture", "hrm"], default="pipeline")\n    parser.add_argument("--model-mode", choices=["fallback", "strict"], default=None)\n    parser.add_argument("--out", default=str(DEFAULT_KAGGLE_OUTPUT))\n    parser.add_argument("--score", action="store_true", help="Score output when solutions are available.")\n    args = parser.parse_args(argv)\n\n    try:\n        challenge_path = Path(args.challenges) if args.challenges else resolve_challenge_path(args.data_dir, args.split)\n        solution_path = Path(args.solutions) if args.solutions else resolve_solution_path(args.data_dir, args.split)\n\n        tasks = load_challenges(challenge_path)\n        solver = make_solver(args.solver, model_mode=args.model_mode)\n        fallback_solver = solver if isinstance(solver, BaselineSolver) else BaselineSolver()\n        if args.solver == "hrm":\n            symbolic_predictions, remaining_tasks = solve_symbolic_first(list(tasks.values()))\n            print(\n                f"symbolic solver: {len(symbolic_predictions)}/{len(tasks)} tasks solved with a "\n                f"train-verified rule; {len(remaining_tasks)} sent to HRM",\n                file=sys.stderr,\n            )\n            try:\n                env = HRMEnvironment.from_env()\n                env.validate(require_cuda=True)\n                if os.environ.get("MYTHOS_ENABLE_TTT") == "1":\n                    runner = HRMTTTRunner(\n                        env,\n                        ttt=TTTConfig(\n                            rank=int(os.environ.get("MYTHOS_TTT_RANK", "16")),\n                            steps=int(os.environ.get("MYTHOS_TTT_STEPS", "20")),\n                            lr=float(os.environ.get("MYTHOS_TTT_LR", "1e-3")),\n                            batch_size=int(os.environ.get("MYTHOS_TTT_BATCH_SIZE", "2")),\n                            genie_weight=float(os.environ.get("MYTHOS_TTT_GENIE_WEIGHT", "0.1")),\n                            ensemble_transforms=parse_ensemble_transforms(\n                                os.environ.get("MYTHOS_TTT_ENSEMBLE", "0")\n                            ),\n                        ),\n                        num_aug=int(os.environ.get("MYTHOS_TTT_NUM_AUG", "0")),\n                    )\n                else:\n                    runner = HRMInferenceRunner(env)\n                hrm_predictions = runner.solve_tasks(remaining_tasks) if remaining_tasks else []\n            except Exception as exc:  # noqa: BLE001 - HRM batch failure must not abort the run\n                print(f"WARNING: HRM batch run failed: {exc!r}; using baseline fallback for all tasks", file=sys.stderr)\n                traceback.print_exc(file=sys.stderr)\n                if getattr(exc, "stdout", None):\n                    print("--- subprocess stdout (tail) ---", file=sys.stderr)\n                    print(exc.stdout[-4000:], file=sys.stderr)\n                if getattr(exc, "stderr", None):\n                    print("--- subprocess stderr (tail) ---", file=sys.stderr)\n                    print(exc.stderr[-4000:], file=sys.stderr)\n                hrm_predictions = [fallback_solver.solve(task) for task in remaining_tasks]\n            hrm_predictions_by_id = {prediction.task_id: prediction for prediction in hrm_predictions}\n            predictions = [\n                symbolic_predictions.get(task.id) or hrm_predictions_by_id[task.id] for task in tasks.values()\n            ]\n        else:\n            predictions = [solve_with_fallback(solver, fallback_solver, task) for task in tasks.values()]\n        write_submission(predictions, args.out)\n\n        summary: dict[str, object] = {\n            "challenge_path": str(challenge_path),\n            "output_path": str(args.out),\n            "solver": args.solver,\n            "model_mode": args.model_mode or "fallback",\n            "tasks": len(tasks),\n        }\n        if hasattr(solver, "pipeline"):\n            summary["models"] = solver.pipeline.model_registry.summary()\n        if args.score and solution_path is not None:\n            summary["score"] = score_files(args.out, str(solution_path)).to_dict()\n        elif args.score:\n            summary["score"] = "skipped: no solutions file for this split"\n    except (ArcValidationError, SolverError, FileNotFoundError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(summary, indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/lora.py': '"""Minimal LoRA utilities for HRM/TTT adapter experiments."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport importlib\nimport math\nfrom pathlib import Path\nfrom typing import Iterable, Sequence\n\n\ntry:  # Keep this module importable until a LoRA function is actually used.\n    from torch import nn as _OPTIONAL_NN\nexcept Exception:  # pragma: no cover - depends on optional torch install.\n    _OPTIONAL_NN = None\n\nDEFAULT_LORA_TARGET_PATTERNS = (\n    "attn",\n    "attention",\n    "qkv_proj",\n    "q_proj",\n    "k_proj",\n    "v_proj",\n    "o_proj",\n    "out_proj",\n)\n\n\n@dataclass(frozen=True)\nclass LoRAInjectionReport:\n    injected_modules: tuple[str, ...]\n    trainable_parameters: int\n    frozen_parameters: int\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "injected_modules": list(self.injected_modules),\n            "trainable_parameters": self.trainable_parameters,\n            "frozen_parameters": self.frozen_parameters,\n        }\n\n\ndef _torch():\n    try:\n        import torch\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for LoRA utilities") from exc\n    return torch\n\n\ndef _nn():\n    try:\n        from torch import nn\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for LoRA utilities") from exc\n    return nn\n\n\n_BASE_MODULE = _OPTIONAL_NN.Module if _OPTIONAL_NN is not None else object\n\n\nclass LoRALinear(_BASE_MODULE):\n    """Wrap a Linear layer with trainable low-rank adapter weights."""\n\n    def __init__(\n        self,\n        base_layer,\n        *,\n        rank: int = 16,\n        alpha: float | None = None,\n        dropout: float = 0.0,\n    ) -> None:\n        nn = _nn()\n        torch = _torch()\n        super().__init__()\n        if not isinstance(base_layer, nn.Linear):\n            raise TypeError("LoRALinear can only wrap torch.nn.Linear")\n        if rank <= 0:\n            raise ValueError("LoRA rank must be positive")\n\n        self.base_layer = base_layer\n        self.rank = rank\n        self.alpha = float(alpha if alpha is not None else rank)\n        self.scaling = self.alpha / float(rank)\n        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()\n        # Match the base layer\'s device/dtype: creating these as plain CPU/fp32\n        # tensors breaks torch.compile\'d models (HRM runs compiled on CUDA in\n        # bfloat16) -- confirmed by a real run: Dynamo\'s tracer rejected the\n        # LoRA matmul with "Unhandled FakeTensor Device Propagation ... found\n        # two different devices cuda:0, cpu".\n        base_device = base_layer.weight.device\n        base_dtype = base_layer.weight.dtype\n        self.lora_a = nn.Parameter(torch.empty(rank, base_layer.in_features, device=base_device, dtype=base_dtype))\n        self.lora_b = nn.Parameter(torch.zeros(base_layer.out_features, rank, device=base_device, dtype=base_dtype))\n        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))\n\n        for parameter in self.base_layer.parameters():\n            parameter.requires_grad = False\n\n    def forward(self, inputs):  # type: ignore[no-untyped-def]\n        torch = _torch()\n        base = self.base_layer(inputs)\n        # CastedLinear (and some plain Linear layers under mixed precision) stores\n        # its weight in one dtype (fp32) but receives activations in another\n        # (bf16) -- cast the activation to lora_a\'s dtype before this matmul, not\n        # base_layer.weight\'s dtype, since those two can legitimately differ.\n        # Confirmed by a real run: "expected mat1 and mat2 to have the same\n        # dtype, but got: c10::BFloat16 != float" once the device mismatch (the\n        # earlier bug) was fixed.\n        update = self.dropout(inputs).to(dtype=self.lora_a.dtype).matmul(self.lora_a.transpose(0, 1))\n        update = update.matmul(self.lora_b.transpose(0, 1))\n        return base + update.to(dtype=base.dtype) * torch.as_tensor(self.scaling, dtype=base.dtype, device=base.device)\n\n\nclass LoRACastedLinear(_BASE_MODULE):\n    """Wrap HRM\'s custom CastedLinear layer with trainable LoRA weights."""\n\n    def __init__(\n        self,\n        base_layer,\n        *,\n        rank: int = 16,\n        alpha: float | None = None,\n        dropout: float = 0.0,\n    ) -> None:\n        nn = _nn()\n        torch = _torch()\n        super().__init__()\n        if not _is_casted_linear_like(base_layer):\n            raise TypeError("LoRACastedLinear can only wrap CastedLinear-like modules")\n        if rank <= 0:\n            raise ValueError("LoRA rank must be positive")\n\n        out_features, in_features = base_layer.weight.shape\n        self.base_layer = base_layer\n        self.rank = rank\n        self.alpha = float(alpha if alpha is not None else rank)\n        self.scaling = self.alpha / float(rank)\n        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()\n        # See LoRALinear\'s identical comment: must match the base layer\'s\n        # device/dtype or torch.compile\'d forward passes fail to trace.\n        base_device = base_layer.weight.device\n        base_dtype = base_layer.weight.dtype\n        self.lora_a = nn.Parameter(torch.empty(rank, in_features, device=base_device, dtype=base_dtype))\n        self.lora_b = nn.Parameter(torch.zeros(out_features, rank, device=base_device, dtype=base_dtype))\n        nn.init.kaiming_uniform_(self.lora_a, a=math.sqrt(5))\n\n        for parameter in self.base_layer.parameters():\n            parameter.requires_grad = False\n\n    def forward(self, inputs):  # type: ignore[no-untyped-def]\n        torch = _torch()\n        base = self.base_layer(inputs)\n        # CastedLinear (and some plain Linear layers under mixed precision) stores\n        # its weight in one dtype (fp32) but receives activations in another\n        # (bf16) -- cast the activation to lora_a\'s dtype before this matmul, not\n        # base_layer.weight\'s dtype, since those two can legitimately differ.\n        # Confirmed by a real run: "expected mat1 and mat2 to have the same\n        # dtype, but got: c10::BFloat16 != float" once the device mismatch (the\n        # earlier bug) was fixed.\n        update = self.dropout(inputs).to(dtype=self.lora_a.dtype).matmul(self.lora_a.transpose(0, 1))\n        update = update.matmul(self.lora_b.transpose(0, 1))\n        return base + update.to(dtype=base.dtype) * torch.as_tensor(self.scaling, dtype=base.dtype, device=base.device)\n\n\ndef inject_lora_adapters(\n    model,\n    *,\n    rank: int = 16,\n    alpha: float | None = None,\n    dropout: float = 0.0,\n    target_patterns: Sequence[str] = DEFAULT_LORA_TARGET_PATTERNS,\n    fallback_to_all_linear: bool = False,\n    freeze_backbone: bool = True,\n) -> LoRAInjectionReport:\n    """Replace matching Linear/CastedLinear modules with LoRA wrappers."""\n\n    nn = _nn()\n    lowered_patterns = tuple(pattern.lower() for pattern in target_patterns)\n    injected: list[str] = []\n\n    if freeze_backbone:\n        for parameter in model.parameters():\n            parameter.requires_grad = False\n\n    def should_wrap(full_name: str, child) -> bool:  # type: ignore[no-untyped-def]\n        if not _is_lora_wrappable(child, nn):\n            return False\n        if any(pattern in full_name.lower() for pattern in lowered_patterns):\n            return True\n        return fallback_to_all_linear\n\n    def visit(module, prefix: str = "") -> None:  # type: ignore[no-untyped-def]\n        for child_name, child in list(module.named_children()):\n            full_name = f"{prefix}.{child_name}" if prefix else child_name\n            if should_wrap(full_name, child):\n                setattr(\n                    module,\n                    child_name,\n                    _wrap_lora_layer(child, nn, rank=rank, alpha=alpha, dropout=dropout),\n                )\n                injected.append(full_name)\n            else:\n                visit(child, full_name)\n\n    visit(model)\n    if not injected:\n        raise ValueError("no Linear or CastedLinear modules matched the LoRA target patterns")\n\n    trainable = 0\n    frozen = 0\n    for parameter in model.parameters():\n        if parameter.requires_grad:\n            trainable += parameter.numel()\n        else:\n            frozen += parameter.numel()\n    return LoRAInjectionReport(\n        injected_modules=tuple(injected),\n        trainable_parameters=trainable,\n        frozen_parameters=frozen,\n    )\n\n\ndef lora_parameters(model) -> list:  # type: ignore[no-untyped-def]\n    return [parameter for name, parameter in model.named_parameters() if "lora_" in name and parameter.requires_grad]\n\n\ndef lora_state_dict(model) -> dict[str, object]:  # type: ignore[no-untyped-def]\n    return {\n        name: parameter.detach().cpu()\n        for name, parameter in model.named_parameters()\n        if "lora_" in name\n    }\n\n\ndef save_lora_checkpoint(model, path: str | Path, *, metadata: dict[str, object] | None = None) -> Path:  # type: ignore[no-untyped-def]\n    torch = _torch()\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "kind": "mythos_lora",\n            "state_dict": lora_state_dict(model),\n            "metadata": metadata or {},\n        },\n        output_path,\n    )\n    return output_path\n\n\ndef snapshot_frozen_parameters(model) -> dict[str, object]:  # type: ignore[no-untyped-def]\n    return {\n        name: parameter.detach().clone().cpu()\n        for name, parameter in model.named_parameters()\n        if not parameter.requires_grad\n    }\n\n\ndef changed_frozen_parameters(model, snapshot: dict[str, object], *, atol: float = 0.0) -> tuple[str, ...]:  # type: ignore[no-untyped-def]\n    torch = _torch()\n    changed: list[str] = []\n    for name, before in snapshot.items():\n        current = dict(model.named_parameters())[name].detach().cpu()\n        if not torch.allclose(current, before, atol=atol, rtol=0.0):\n            changed.append(name)\n    return tuple(changed)\n\n\ndef count_parameters(parameters: Iterable) -> int:  # type: ignore[type-arg]\n    return sum(parameter.numel() for parameter in parameters)\n\n\ndef _wrap_lora_layer(child, nn, *, rank: int, alpha: float | None, dropout: float):  # type: ignore[no-untyped-def]\n    if isinstance(child, nn.Linear):\n        return LoRALinear(child, rank=rank, alpha=alpha, dropout=dropout)\n    if _is_casted_linear_like(child):\n        return LoRACastedLinear(child, rank=rank, alpha=alpha, dropout=dropout)\n    raise TypeError(f"unsupported LoRA target module: {type(child).__name__}")\n\n\ndef _is_lora_wrappable(child, nn) -> bool:  # type: ignore[no-untyped-def]\n    return isinstance(child, nn.Linear) or _is_casted_linear_like(child)\n\n\ndef _is_casted_linear_like(child) -> bool:  # type: ignore[no-untyped-def]\n    casted_types = _casted_linear_types()\n    if casted_types and isinstance(child, casted_types):\n        return True\n    if child.__class__.__name__ != "CastedLinear":\n        return False\n    weight = getattr(child, "weight", None)\n    return weight is not None and getattr(weight, "ndim", None) == 2 and callable(getattr(child, "forward", None))\n\n\ndef _casted_linear_types() -> tuple[type, ...]:\n    types: list[type] = []\n    for module_name in ("models.layers", "layers"):\n        try:\n            module = importlib.import_module(module_name)\n        except Exception:\n            continue\n        casted_linear = getattr(module, "CastedLinear", None)\n        if isinstance(casted_linear, type):\n            types.append(casted_linear)\n    return tuple(types)\n', 'src/mythos/losses.py': '"""Loss functions for Mythos adaptation experiments."""\n\nfrom __future__ import annotations\n\nfrom typing import Sequence\n\nfrom mythos.arc import Grid\n\n\ndef genie_background_consistency_loss(\n    logits,\n    input_grid: Grid,\n    *,\n    preserve_mask: Sequence[Sequence[bool]] | None = None,\n):  # type: ignore[no-untyped-def]\n    """Penalize changing cells that should remain visually consistent.\n\n    `logits` may be shaped `[H, W, 10]`, `[1, H, W, 10]`, or `[10, H, W]`.\n    The target color for preserved cells is the original input-grid color.\n    """\n\n    torch = _torch()\n    prepared = _prepare_logits(logits)\n    height = min(prepared.shape[0], len(input_grid))\n    width = min(prepared.shape[1], len(input_grid[0]))\n    mask = preserve_mask or _default_preserve_mask(input_grid)\n\n    selected_logits = []\n    selected_targets = []\n    for row in range(height):\n        for col in range(width):\n            if row < len(mask) and col < len(mask[row]) and mask[row][col]:\n                selected_logits.append(prepared[row, col])\n                selected_targets.append(int(input_grid[row][col]))\n\n    if not selected_logits:\n        return prepared.sum() * 0.0\n\n    logits_tensor = torch.stack(selected_logits, dim=0)\n    target_tensor = torch.tensor(selected_targets, dtype=torch.long, device=prepared.device)\n    return torch.nn.functional.cross_entropy(logits_tensor, target_tensor)\n\n\ndef background_preservation_mask(input_grid: Grid, output_grid: Grid | None = None) -> tuple[tuple[bool, ...], ...]:\n    """Return cells that should be preserved for consistency regularization."""\n\n    if output_grid is not None and _same_shape(input_grid, output_grid):\n        return tuple(\n            tuple(input_grid[row][col] == output_grid[row][col] for col in range(len(input_grid[0])))\n            for row in range(len(input_grid))\n        )\n    return _default_preserve_mask(input_grid)\n\n\ndef _default_preserve_mask(input_grid: Grid) -> tuple[tuple[bool, ...], ...]:\n    background = _dominant_color(input_grid)\n    return tuple(tuple(cell == background for cell in row) for row in input_grid)\n\n\ndef _prepare_logits(logits):  # type: ignore[no-untyped-def]\n    torch = _torch()\n    tensor = logits if hasattr(logits, "shape") else torch.as_tensor(logits)\n    if tensor.ndim == 4:\n        if tensor.shape[0] != 1:\n            raise ValueError("batched consistency loss expects batch size 1")\n        tensor = tensor[0]\n    if tensor.ndim != 3:\n        raise ValueError("logits must have rank 3 or rank 4")\n    if tensor.shape[0] == 10 and tensor.shape[-1] != 10:\n        tensor = tensor.permute(1, 2, 0)\n    if tensor.shape[-1] != 10:\n        raise ValueError("logits must have 10 color channels")\n    return tensor.float()\n\n\ndef _same_shape(left: Grid, right: Grid) -> bool:\n    return len(left) == len(right) and len(left[0]) == len(right[0])\n\n\ndef _dominant_color(grid: Grid) -> int:\n    counts: dict[int, int] = {}\n    for row in grid:\n        for cell in row:\n            counts[cell] = counts.get(cell, 0) + 1\n    return max(counts.items(), key=lambda item: item[1])[0]\n\n\ndef _torch():\n    try:\n        import torch\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for Mythos loss functions") from exc\n    return torch\n', 'src/mythos/metrics.py': '"""Scoring helpers for ARC-style submissions."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom typing import Mapping, Tuple\n\nfrom mythos.arc import ArcValidationError, Grid, SolutionMap, grid_equal, load_solutions\nfrom mythos.submission import SubmissionMap, TestPrediction, load_submission\n\n\n@dataclass(frozen=True)\nclass ScoreResult:\n    total_items: int\n    exact_matches: int\n    exact_attempt_1: int\n    exact_attempt_2: int\n    total_cells: int\n    matched_cells: int\n    extra_predictions: int\n\n    @property\n    def exact_accuracy(self) -> float:\n        return self.exact_matches / self.total_items if self.total_items else 0.0\n\n    @property\n    def cell_accuracy(self) -> float:\n        return self.matched_cells / self.total_cells if self.total_cells else 0.0\n\n    def to_dict(self) -> dict[str, int | float]:\n        data = asdict(self)\n        data["exact_accuracy"] = self.exact_accuracy\n        data["cell_accuracy"] = self.cell_accuracy\n        return data\n\n\ndef _cell_count(grid: Grid) -> int:\n    return sum(len(row) for row in grid)\n\n\ndef _cell_matches(prediction: Grid, truth: Grid) -> Tuple[int, int]:\n    total = _cell_count(truth)\n    if len(prediction) != len(truth) or len(prediction[0]) != len(truth[0]):\n        return 0, total\n    matches = 0\n    for pred_row, truth_row in zip(prediction, truth):\n        for pred_cell, truth_cell in zip(pred_row, truth_row):\n            if pred_cell == truth_cell:\n                matches += 1\n    return matches, total\n\n\ndef score_submission_data(predictions: SubmissionMap, solutions: SolutionMap) -> ScoreResult:\n    total_items = 0\n    exact_matches = 0\n    exact_attempt_1 = 0\n    exact_attempt_2 = 0\n    matched_cells = 0\n    total_cells = 0\n\n    extra_predictions = len(set(predictions) - set(solutions))\n\n    for task_id, truth_outputs in solutions.items():\n        if task_id not in predictions:\n            raise ArcValidationError(f"submission is missing task {task_id}")\n        task_predictions = predictions[task_id]\n        if len(task_predictions) != len(truth_outputs):\n            raise ArcValidationError(\n                f"{task_id} has {len(task_predictions)} predictions but "\n                f"{len(truth_outputs)} solution outputs"\n            )\n\n        for prediction, truth in zip(task_predictions, truth_outputs):\n            total_items += 1\n            attempt_1_exact = grid_equal(prediction.attempt_1, truth)\n            attempt_2_exact = grid_equal(prediction.attempt_2, truth)\n            exact_attempt_1 += int(attempt_1_exact)\n            exact_attempt_2 += int(attempt_2_exact)\n            exact_matches += int(attempt_1_exact or attempt_2_exact)\n\n            cells_1, cells_total = _cell_matches(prediction.attempt_1, truth)\n            cells_2, _ = _cell_matches(prediction.attempt_2, truth)\n            matched_cells += max(cells_1, cells_2)\n            total_cells += cells_total\n\n    return ScoreResult(\n        total_items=total_items,\n        exact_matches=exact_matches,\n        exact_attempt_1=exact_attempt_1,\n        exact_attempt_2=exact_attempt_2,\n        total_cells=total_cells,\n        matched_cells=matched_cells,\n        extra_predictions=extra_predictions,\n    )\n\n\ndef score_files(prediction_path: str, solution_path: str) -> ScoreResult:\n    return score_submission_data(load_submission(prediction_path), load_solutions(solution_path))\n', 'src/mythos/models.py': '"""External model loading for the Project Mythos pipeline."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass, field\nimport importlib\nimport os\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nfrom mythos.solvers.base import SolverError\n\n\nclass ModelLoadError(SolverError):\n    """Raised when a configured external model cannot be loaded."""\n\n\n@dataclass(frozen=True)\nclass ModelSpec:\n    key: str\n    label: str\n    checkpoint_env: str\n    repo_env: str | None = None\n    module_names: tuple[str, ...] = ()\n\n\nMODEL_SPECS: tuple[ModelSpec, ...] = (\n    ModelSpec(\n        key="jepa",\n        label="JEPA encoder",\n        checkpoint_env="IJEPA_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="jepa_projection",\n        label="I-JEPA projection",\n        checkpoint_env="IJEPA_PROJECTION_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="hrm_text",\n        label="HRM-Text H-module",\n        checkpoint_env="HRM_TEXT_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="world_model",\n        label="World model",\n        checkpoint_env="WORLD_MODEL_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="ttt_lora",\n        label="TTT LoRA adapters",\n        checkpoint_env="TTT_LORA_CHECKPOINT_PATH",\n    ),\n    ModelSpec(\n        key="hrm_l_module",\n        label="HRM 27M L-module",\n        repo_env="HRM_REPO_DIR",\n        checkpoint_env="HRM_CHECKPOINT_PATH",\n        module_names=("pretrain", "evaluate"),\n    ),\n)\n\n\n@dataclass(frozen=True)\nclass LoadedModel:\n    spec: ModelSpec\n    repo_dir: Path | None = None\n    checkpoint_path: Path | None = None\n    checkpoint: Any = None\n    modules: dict[str, Any] = field(default_factory=dict)\n    error: str | None = None\n\n    @property\n    def loaded(self) -> bool:\n        return self.checkpoint is not None\n\n    def describe(self) -> str:\n        if not self.loaded:\n            if self.error:\n                return f"{self.spec.label}: load failed: {self.error}"\n            return f"{self.spec.label}: not configured"\n        parts = [f"{self.spec.label}: checkpoint={self.checkpoint_path}"]\n        if self.repo_dir is not None:\n            parts.append(f"repo={self.repo_dir}")\n        if self.modules:\n            parts.append(f"modules={\',\'.join(sorted(self.modules))}")\n        return "; ".join(parts)\n\n\nclass ModelRegistry:\n    """Loads and stores external models keyed by planned pipeline component."""\n\n    def __init__(self, models: dict[str, LoadedModel], *, strict: bool) -> None:\n        self.models = models\n        self.strict = strict\n\n    @classmethod\n    def from_env(cls, *, strict: bool = False) -> "ModelRegistry":\n        models: dict[str, LoadedModel] = {}\n        for spec in MODEL_SPECS:\n            try:\n                models[spec.key] = _load_model_from_env(spec, strict=strict)\n            except ModelLoadError as exc:\n                if strict:\n                    raise\n                models[spec.key] = LoadedModel(spec=spec, error=str(exc))\n        return cls(models=models, strict=strict)\n\n    def get(self, key: str) -> LoadedModel:\n        return self.models[key]\n\n    def summary(self) -> list[dict[str, object]]:\n        return [\n            {\n                "key": key,\n                "label": model.spec.label,\n                "loaded": model.loaded,\n                "repo_dir": str(model.repo_dir) if model.repo_dir is not None else None,\n                "checkpoint_path": str(model.checkpoint_path) if model.checkpoint_path is not None else None,\n                "modules": sorted(model.modules),\n                "error": model.error,\n            }\n            for key, model in self.models.items()\n        ]\n\n\ndef _load_model_from_env(spec: ModelSpec, *, strict: bool) -> LoadedModel:\n    repo_value = os.environ.get(spec.repo_env) if spec.repo_env is not None else None\n    checkpoint_value = os.environ.get(spec.checkpoint_env)\n\n    if not repo_value and not checkpoint_value:\n        if strict:\n            missing = spec.checkpoint_env\n            if spec.repo_env is not None:\n                missing = f"{spec.repo_env} and {spec.checkpoint_env}"\n            raise ModelLoadError(f"{missing} are required for strict model loading")\n        return LoadedModel(spec=spec)\n\n    if spec.repo_env is not None and not repo_value:\n        raise ModelLoadError(f"{spec.repo_env} is required when loading {spec.label}")\n    if not checkpoint_value:\n        raise ModelLoadError(f"{spec.checkpoint_env} is required when loading {spec.label}")\n\n    repo_dir = Path(repo_value) if repo_value else None\n    checkpoint_path = Path(checkpoint_value)\n\n    if repo_dir is not None:\n        if not repo_dir.exists():\n            raise ModelLoadError(f"{spec.repo_env} does not exist: {repo_dir}")\n        _add_repo_to_path(repo_dir)\n\n    if not checkpoint_path.exists():\n        raise ModelLoadError(f"{spec.checkpoint_env} does not exist: {checkpoint_path}")\n\n    modules = _import_modules(spec)\n    checkpoint = _load_torch_checkpoint(checkpoint_path)\n    return LoadedModel(\n        spec=spec,\n        repo_dir=repo_dir,\n        checkpoint_path=checkpoint_path,\n        checkpoint=checkpoint,\n        modules=modules,\n    )\n\n\ndef _add_repo_to_path(repo_dir: Path) -> None:\n    repo = str(repo_dir.resolve())\n    if repo not in sys.path:\n        sys.path.insert(0, repo)\n\n\ndef _import_modules(spec: ModelSpec) -> dict[str, Any]:\n    modules: dict[str, Any] = {}\n    for module_name in spec.module_names:\n        try:\n            modules[module_name] = importlib.import_module(module_name)\n        except Exception as exc:\n            raise ModelLoadError(\n                f"failed to import {module_name!r} for {spec.label}: {exc}"\n            ) from exc\n    return modules\n\n\ndef _load_torch_checkpoint(checkpoint_path: Path) -> Any:\n    if checkpoint_path.is_dir() or checkpoint_path.suffix.lower() == ".safetensors":\n        return {\n            "path": str(checkpoint_path),\n            "format": checkpoint_path.suffix.lower().lstrip(".") or "directory",\n            "lazy": True,\n        }\n\n    try:\n        torch = importlib.import_module("torch")\n    except Exception as exc:\n        raise ModelLoadError("PyTorch is required to load model checkpoints") from exc\n\n    map_location = "cuda" if torch.cuda.is_available() else "cpu"\n    try:\n        return torch.load(checkpoint_path, map_location=map_location, weights_only=False)\n    except TypeError:\n        return torch.load(checkpoint_path, map_location=map_location)\n', 'src/mythos/object_ops.py': '"""Object-level transform candidates for verified program search.\n\nEach `find_*_transform` function takes a whole ArcTask and tries to build\nONE transform (Grid -> Grid) that reproduces every train pair exactly; if\nit can, it returns that transform ready to apply to test inputs, else None.\nThis mirrors FixtureSolver\'s verify-before-trust pattern, just operating\nover segmented objects instead of raw grid transforms -- the substrate a\none-line rule like "move this fragment into its matching slot" actually\nneeds, instead of reasoning over 900 flat pixels.\n"""\n\nfrom __future__ import annotations\n\nfrom typing import Callable\n\nfrom mythos.arc import ArcTask, Grid, grid_equal\nfrom mythos.objects import ArcObject, crop_to_object, d4_signature_variants, segment_objects\n\nTransform = Callable[[Grid], Grid]\n\n_BACKGROUND = 0\n\n\ndef find_fragment_to_slot_transform(task: ArcTask) -> Transform | None:\n    """Move/copy scattered fragments into shape-matching occluded slots.\n\n    Targets the "jigsaw" ARC pattern: one or more small colored objects sit\n    apart from a solid-colored placeholder region shaped like a rotated or\n    mirrored copy of one of them; the true output pastes each fragment\'s\n    colors into its matching placeholder, possibly leaving the original\n    fragment in place or removing it. Both variants are tried and verified\n    against every train pair before being trusted; an ambiguous match (two\n    donors equally fit one slot) makes a variant refuse rather than guess.\n    """\n\n    if any(example.output is None for example in task.train):\n        return None\n\n    # Slot colors are not restricted to solid rectangles here (unlike\n    # mythos.symmetry\'s occlusion detector): a slot mirrors a donor\n    # fragment\'s own silhouette, which can be any shape. Verification\n    # against every train pair is what filters out the false positives a\n    # broad candidate set like this necessarily includes.\n    slot_colors: set[int] = set()\n    for example in task.train:\n        slot_colors.update(color for row in example.input for color in row if color != _BACKGROUND)\n\n    for slot_color in slot_colors:\n        for keep_donor in (False, True):\n            transform = _fragment_transform(slot_color, keep_donor)\n            if _verifies_on_train(transform, task):\n                return transform\n    return None\n\n\ndef _fragment_transform(slot_color: int, keep_donor: bool) -> Transform:\n    def transform(grid: Grid) -> Grid:\n        result = _place_fragments(grid, slot_color, keep_donor)\n        if result is None:\n            raise _TransformFailed(f"no unambiguous fragment placement for slot color {slot_color}")\n        return result\n\n    return transform\n\n\ndef _place_fragments(grid: Grid, slot_color: int, keep_donor: bool) -> Grid | None:\n    all_objects = segment_objects(grid, background=_BACKGROUND, connectivity=8, univalued=False)\n    slots = [obj for obj in all_objects if obj.dominant_color == slot_color and obj.is_univalued]\n    donors = [obj for obj in all_objects if obj.dominant_color != slot_color]\n    if not slots or not donors:\n        return None\n\n    result = [row[:] for row in grid]\n    matched_donor_indices: set[int] = set()\n    for slot in slots:\n        matches = [\n            index\n            for index, donor in enumerate(donors)\n            if index not in matched_donor_indices and _donor_fits_slot(donor, slot)\n        ]\n        if len(matches) != 1:\n            return None  # no fit, or an ambiguous multi-way fit -- refuse\n        donor_index = matches[0]\n        matched_donor_indices.add(donor_index)\n        if not _paste_donor_into_slot(result, donors[donor_index], slot):\n            return None\n        if not keep_donor:\n            for r, c in donors[donor_index].cells:\n                result[r][c] = _BACKGROUND\n    return result\n\n\ndef _donor_fits_slot(donor: ArcObject, slot: ArcObject) -> bool:\n    return slot.shape_signature in d4_signature_variants(donor.shape_signature)\n\n\ndef _paste_donor_into_slot(result: Grid, donor: ArcObject, slot: ArcObject) -> bool:\n    target_shape = slot.shape_signature\n    top, left, _, _ = slot.bbox\n    for variant in _colored_d4_variants(donor.colored_signature):\n        variant_shape = frozenset((r, c) for r, c, _ in variant)\n        if variant_shape != target_shape:\n            continue\n        for r, c, color in variant:\n            result[top + r][left + c] = color\n        return True\n    return False\n\n\ndef _colored_d4_variants(colored_signature: frozenset[tuple[int, int, int]]) -> list[frozenset[tuple[int, int, int]]]:\n    variants: list[frozenset[tuple[int, int, int]]] = []\n    current = colored_signature\n    for _ in range(4):\n        variants.append(_normalize_colored(current))\n        variants.append(_normalize_colored(frozenset((r, -c, color) for r, c, color in current)))\n        current = frozenset((c, -r, color) for r, c, color in current)  # rotate 90 degrees\n    return variants\n\n\ndef _normalize_colored(cells: frozenset[tuple[int, int, int]]) -> frozenset[tuple[int, int, int]]:\n    min_r = min(r for r, _, _ in cells)\n    min_c = min(c for _, c, _ in cells)\n    return frozenset((r - min_r, c - min_c, color) for r, c, color in cells)\n\n\ndef find_crop_to_selected_object_transform(task: ArcTask) -> Transform | None:\n    """Output is one train-consistently-selected object, cropped to its bounding box.\n\n    Common ARC pattern family: "find the largest/smallest/uniquely-colored/\n    uniquely-shaped object and output just that." Each selection rule is\n    tried and verified against every train pair.\n    """\n\n    if any(example.output is None for example in task.train):\n        return None\n\n    for connectivity in (4, 8):\n        for univalued in (True, False):\n            for selector in _OBJECT_SELECTORS:\n                transform = _crop_to_selected_transform(connectivity, univalued, selector)\n                if _verifies_on_train(transform, task):\n                    return transform\n    return None\n\n\ndef _crop_to_selected_transform(connectivity: int, univalued: bool, selector: Callable[[list[ArcObject]], ArcObject | None]) -> Transform:\n    def transform(grid: Grid) -> Grid:\n        objects = segment_objects(grid, background=_BACKGROUND, connectivity=connectivity, univalued=univalued)\n        selected = selector(objects)\n        if selected is None:\n            raise _TransformFailed("no object matched the selection rule")\n        return crop_to_object(grid, selected, background=_BACKGROUND)\n\n    return transform\n\n\ndef _largest_object(objects: list[ArcObject]) -> ArcObject | None:\n    return max(objects, key=lambda obj: obj.size) if objects else None\n\n\ndef _smallest_object(objects: list[ArcObject]) -> ArcObject | None:\n    return min(objects, key=lambda obj: obj.size) if objects else None\n\n\ndef _uniquely_colored_object(objects: list[ArcObject]) -> ArcObject | None:\n    counts: dict[int, int] = {}\n    for obj in objects:\n        counts[obj.dominant_color] = counts.get(obj.dominant_color, 0) + 1\n    unique = [obj for obj in objects if counts[obj.dominant_color] == 1]\n    return unique[0] if len(unique) == 1 else None\n\n\ndef _uniquely_shaped_object(objects: list[ArcObject]) -> ArcObject | None:\n    counts: dict[frozenset, int] = {}\n    for obj in objects:\n        counts[obj.shape_signature] = counts.get(obj.shape_signature, 0) + 1\n    unique = [obj for obj in objects if counts[obj.shape_signature] == 1]\n    return unique[0] if len(unique) == 1 else None\n\n\n_OBJECT_SELECTORS = (_largest_object, _smallest_object, _uniquely_colored_object, _uniquely_shaped_object)\n\n\ndef find_fill_enclosed_regions_transform(task: ArcTask) -> Transform | None:\n    """Fill background pockets fully enclosed by non-background cells.\n\n    Common ARC pattern: closed outlines get their interior flood-filled\n    with a consistent color. Tries a handful of train-derived fill-color\n    rules (fixed color, bordering object\'s own color) and verifies each.\n    """\n\n    if any(example.output is None for example in task.train):\n        return None\n\n    palette: set[int] = set()\n    for example in task.train:\n        for row in example.output:\n            palette.update(row)\n\n    for fill_color in sorted(palette):\n        transform = _fill_enclosed_transform(fill_color)\n        if _verifies_on_train(transform, task):\n            return transform\n\n    transform = _fill_enclosed_transform(None)  # fill with the enclosing object\'s own color\n    if _verifies_on_train(transform, task):\n        return transform\n    return None\n\n\ndef _fill_enclosed_transform(fill_color: int | None) -> Transform:\n    def transform(grid: Grid) -> Grid:\n        return _fill_enclosed_regions(grid, fill_color)\n\n    return transform\n\n\ndef _fill_enclosed_regions(grid: Grid, fill_color: int | None) -> Grid:\n    height = len(grid)\n    width = len(grid[0]) if height else 0\n    background_components = segment_objects(grid, background=None, connectivity=4, univalued=True)\n    result = [row[:] for row in grid]\n    for component in background_components:\n        if component.dominant_color != _BACKGROUND:\n            continue\n        if _touches_border(component.cells, height, width):\n            continue\n        color = fill_color if fill_color is not None else _surrounding_color(grid, component.cells, height, width)\n        if color is None:\n            continue\n        for r, c in component.cells:\n            result[r][c] = color\n    return result\n\n\ndef _touches_border(cells: frozenset[tuple[int, int]], height: int, width: int) -> bool:\n    return any(r in (0, height - 1) or c in (0, width - 1) for r, c in cells)\n\n\ndef _surrounding_color(grid: Grid, cells: frozenset[tuple[int, int]], height: int, width: int) -> int | None:\n    neighbor_colors: set[int] = set()\n    for r, c in cells:\n        for dr, dc in ((-1, 0), (1, 0), (0, -1), (0, 1)):\n            nr, nc = r + dr, c + dc\n            if 0 <= nr < height and 0 <= nc < width and (nr, nc) not in cells:\n                neighbor_colors.add(grid[nr][nc])\n    return neighbor_colors.pop() if len(neighbor_colors) == 1 else None\n\n\nclass _TransformFailed(Exception):\n    pass\n\n\ndef _verifies_on_train(transform: Transform, task: ArcTask) -> bool:\n    for example in task.train:\n        if example.output is None:\n            return False\n        try:\n            predicted = transform(example.input)\n        except _TransformFailed:\n            return False\n        if not grid_equal(predicted, example.output):\n            return False\n    return True\n\n\nALL_OBJECT_TRANSFORM_FINDERS = (\n    find_fragment_to_slot_transform,\n    find_crop_to_selected_object_transform,\n    find_fill_enclosed_regions_transform,\n)\n', 'src/mythos/objects.py': '"""Object-centric grid decomposition: connected-component segmentation.\n\nRaw grid tokens are a poor substrate for compositional rules ("move this\nfragment into its matching slot" is a one-line rule over objects and a\nnear-impossible one over 900 flat pixels). This module segments a grid into\nconnected-component objects with shape/color/position attributes, so\ndownstream primitives (mythos.object_ops) can operate over objects instead\nof raw cells.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\n\nfrom mythos.arc import Grid\n\nCell = tuple[int, int]\n\n\n@dataclass(frozen=True)\nclass ArcObject:\n    """A connected group of non-background cells, in original grid coordinates."""\n\n    cells: frozenset[Cell]\n    colors: tuple[tuple[Cell, int], ...]  # (cell, color) pairs, sorted by cell\n\n    @property\n    def color_map(self) -> dict[Cell, int]:\n        return dict(self.colors)\n\n    @property\n    def size(self) -> int:\n        return len(self.cells)\n\n    @property\n    def bbox(self) -> tuple[int, int, int, int]:\n        """(top, left, height, width)."""\n        rows = [r for r, _ in self.cells]\n        cols = [c for _, c in self.cells]\n        top, bottom = min(rows), max(rows)\n        left, right = min(cols), max(cols)\n        return top, left, bottom - top + 1, right - left + 1\n\n    @property\n    def dominant_color(self) -> int:\n        counts = Counter(color for _, color in self.colors)\n        return counts.most_common(1)[0][0]\n\n    @property\n    def is_univalued(self) -> bool:\n        return len({color for _, color in self.colors}) == 1\n\n    @property\n    def shape_signature(self) -> frozenset[Cell]:\n        """Cell offsets relative to the bounding-box origin, ignoring color.\n\n        Two objects with the same signature have the same silhouette,\n        independent of position -- the basis for shape-matching primitives\n        like "find where this fragment\'s outline fits".\n        """\n        top, left, _, _ = self.bbox\n        return frozenset((r - top, c - left) for r, c in self.cells)\n\n    @property\n    def colored_signature(self) -> frozenset[tuple[Cell, int]]:\n        """Like shape_signature, but including each cell\'s color."""\n        top, left, _, _ = self.bbox\n        color_map = self.color_map\n        return frozenset((r - top, c - left, color_map[(r, c)]) for r, c in self.cells)\n\n    def translated(self, dr: int, dc: int) -> "ArcObject":\n        return ArcObject(\n            cells=frozenset((r + dr, c + dc) for r, c in self.cells),\n            colors=tuple(sorted(((r + dr, c + dc), color) for (r, c), color in self.colors)),\n        )\n\n\ndef segment_objects(\n    grid: Grid,\n    *,\n    background: int | None = 0,\n    connectivity: int = 4,\n    univalued: bool = True,\n) -> list[ArcObject]:\n    """Segment a grid into connected-component objects.\n\n    background: color treated as empty space (excluded from all objects).\n        None means every cell (including the majority/background color) is\n        segmented -- rarely useful, but supported for completeness.\n    connectivity: 4 (edge-adjacent) or 8 (edge+diagonal-adjacent).\n    univalued: if True, a component only connects cells of the *same* color\n        (mirrors arc-dsl\'s univalued objects()); if False, any adjacent\n        non-background cells join one object regardless of color (useful for\n        multicolor fragments like the ones in task 16b78196).\n    """\n\n    if connectivity not in (4, 8):\n        raise ValueError(f"connectivity must be 4 or 8, got {connectivity}")\n\n    height = len(grid)\n    width = len(grid[0]) if height else 0\n    visited = [[False] * width for _ in range(height)]\n    offsets = _NEIGHBOR_OFFSETS[connectivity]\n\n    objects: list[ArcObject] = []\n    for start_r in range(height):\n        for start_c in range(width):\n            if visited[start_r][start_c]:\n                continue\n            start_color = grid[start_r][start_c]\n            if background is not None and start_color == background:\n                visited[start_r][start_c] = True\n                continue\n\n            stack = [(start_r, start_c)]\n            visited[start_r][start_c] = True\n            component: list[Cell] = []\n            while stack:\n                r, c = stack.pop()\n                component.append((r, c))\n                cell_color = grid[r][c]\n                for dr, dc in offsets:\n                    nr, nc = r + dr, c + dc\n                    if not (0 <= nr < height and 0 <= nc < width) or visited[nr][nc]:\n                        continue\n                    neighbor_color = grid[nr][nc]\n                    if background is not None and neighbor_color == background:\n                        continue\n                    if univalued and neighbor_color != cell_color:\n                        continue\n                    visited[nr][nc] = True\n                    stack.append((nr, nc))\n\n            colors = tuple(sorted((cell, grid[cell[0]][cell[1]]) for cell in component))\n            objects.append(ArcObject(cells=frozenset(component), colors=colors))\n    return objects\n\n\n_NEIGHBOR_OFFSETS = {\n    4: ((-1, 0), (1, 0), (0, -1), (0, 1)),\n    8: ((-1, 0), (1, 0), (0, -1), (0, 1), (-1, -1), (-1, 1), (1, -1), (1, 1)),\n}\n\n\ndef crop_to_object(grid: Grid, obj: ArcObject, *, background: int = 0) -> Grid:\n    """Return the object\'s bounding box as a standalone grid."""\n\n    top, left, height, width = obj.bbox\n    color_map = obj.color_map\n    return [\n        [color_map.get((top + r, left + c), background) for c in range(width)]\n        for r in range(height)\n    ]\n\n\ndef dominant_grid_color(grid: Grid) -> int:\n    counts = Counter(cell for row in grid for cell in row)\n    return counts.most_common(1)[0][0]\n\n\ndef d4_signature_variants(signature: frozenset[Cell]) -> list[frozenset[Cell]]:\n    """All 8 dihedral-group transforms of a shape signature, each renormalized to (0,0).\n\n    Used to match a donor fragment\'s silhouette against a target slot\'s\n    silhouette allowing for the donor having been rotated/reflected before\n    being placed -- a common ARC pattern ("fit this piece into its outline,\n    turned whichever way makes it fit").\n    """\n\n    variants: list[frozenset[Cell]] = []\n    current = signature\n    for _ in range(4):\n        variants.append(_normalize_signature(current))\n        variants.append(_normalize_signature(frozenset((r, -c) for r, c in current)))\n        current = frozenset((c, -r) for r, c in current)  # rotate 90 degrees\n    return variants\n\n\ndef _normalize_signature(cells: frozenset[Cell]) -> frozenset[Cell]:\n    min_r = min(r for r, _ in cells)\n    min_c = min(c for _, c in cells)\n    return frozenset((r - min_r, c - min_c) for r, c in cells)\n', 'src/mythos/pipeline.py': '"""Plan-aligned Project Mythos inference pipeline.\n\nThe real research components are still adapters here. The important point for\nthe base implementation is that data flows through the same stage boundaries as\nthe master plan, so each placeholder has an obvious replacement point.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nimport os\nfrom typing import Iterable, Tuple\n\nfrom mythos.arc import ArcTask, Grid, copy_grid\nfrom mythos.features import (\n    DEFAULT_HRM_FEATURE_DIM,\n    DEFAULT_JEPA_FEATURE_DIM,\n    embedding_cosine_similarity,\n    grid_to_feature_vector,\n    task_rule_vector,\n)\nfrom mythos.jepa_encoder import JepaEncodingError, JepaImageEncoder\nfrom mythos.models import ModelRegistry\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.base import SolverError\nfrom mythos.solvers.hrm import HRMSolver\nfrom mythos.submission import Prediction, TestPrediction, prediction_to_json, validate_submission_data\nfrom mythos.text_reasoning import HRMTextError, generate_hrm_text_rule\nfrom mythos.training import load_projection_checkpoint, load_world_model_checkpoint\n\nPLAN_STAGE_ORDER = (\n    "ingest",\n    "encode_jepa",\n    "plan_hrm_text",\n    "simulate_world_model",\n    "adapt_ttt_lora",\n    "execute_hrm_l_module",\n    "decode_output",\n)\n\n\n@dataclass(frozen=True)\nclass StageRecord:\n    name: str\n    status: str\n    detail: str\n\n\n@dataclass(frozen=True)\nclass GridEmbedding:\n    source: str\n    shape: tuple[int, int]\n    vector: tuple[float, ...]\n\n\n@dataclass(frozen=True)\nclass RuleVector:\n    source: str\n    description: str\n    vector: tuple[float, ...]\n\n\n@dataclass\nclass PipelineTrace:\n    task_id: str\n    stages: list[StageRecord] = field(default_factory=list)\n\n    @property\n    def stage_names(self) -> list[str]:\n        return [stage.name for stage in self.stages]\n\n    def add(self, name: str, status: str, detail: str) -> None:\n        self.stages.append(StageRecord(name=name, status=status, detail=detail))\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "task_id": self.task_id,\n            "stages": [\n                {"name": stage.name, "status": stage.status, "detail": stage.detail}\n                for stage in self.stages\n            ],\n        }\n\n\n@dataclass(frozen=True)\nclass PipelineResult:\n    prediction: Prediction\n    trace: PipelineTrace\n\n\n@dataclass\nclass PipelineState:\n    task: ArcTask\n    trace: PipelineTrace\n    embeddings: tuple[GridEmbedding, ...] = ()\n    rule_vector: RuleVector | None = None\n    world_rollouts: tuple[GridEmbedding, ...] = ()\n    prediction: Prediction | None = None\n\n\nclass PlannedPipeline:\n    """Runs the ARC task through the Project Mythos master-plan stages."""\n\n    def __init__(\n        self,\n        executor: BaselineSolver | None = None,\n        model_registry: ModelRegistry | None = None,\n        *,\n        strict_models: bool = False,\n    ) -> None:\n        self.executor = executor or BaselineSolver()\n        self.model_registry = model_registry or ModelRegistry.from_env(strict=strict_models)\n        self.strict_models = strict_models\n        self._jepa_encoder = None\n        self._projection_model = None\n        self._world_model = None\n\n    def run(self, task: ArcTask) -> PipelineResult:\n        state = PipelineState(task=task, trace=PipelineTrace(task_id=task.id))\n\n        self._ingest(state)\n        self._encode_jepa(state)\n        self._plan_hrm_text(state)\n        self._simulate_world_model(state)\n        self._adapt_ttt_lora(state)\n        self._execute_hrm_l_module(state)\n        self._decode_output(state)\n\n        assert state.prediction is not None\n        return PipelineResult(prediction=state.prediction, trace=state.trace)\n\n    def _ingest(self, state: PipelineState) -> None:\n        train_count = len(state.task.train)\n        test_count = len(state.task.test)\n        state.trace.add(\n            "ingest",\n            "ok",\n            f"loaded task with {train_count} train pairs and {test_count} test inputs",\n        )\n\n    def _encode_jepa(self, state: PipelineState) -> None:\n        jepa_model = self.model_registry.get("jepa")\n        projection = self.model_registry.get("jepa_projection")\n        embeddings: list[GridEmbedding] = []\n        projection_detail = "projection checkpoint not configured"\n        projection_loaded = False\n        real_jepa_enabled = os.environ.get("MYTHOS_ENABLE_REAL_JEPA", "0") == "1"\n        real_jepa_loaded = False\n        try:\n            jepa_encoder = None\n            if real_jepa_enabled and jepa_model.loaded and jepa_model.checkpoint_path is not None:\n                jepa_encoder = self._get_jepa_encoder(jepa_model.checkpoint_path)\n                real_jepa_loaded = True\n            if projection.loaded and projection.checkpoint_path is not None:\n                projection_model = self._get_projection_model(projection.checkpoint_path)\n                projection_loaded = True\n                projection_detail = f"projection={projection.checkpoint_path}"\n                for split, grids in _iter_task_grids(state.task):\n                    for index, grid in enumerate(grids):\n                        embeddings.append(\n                            _encode_jepa_grid(\n                                f"{split}[{index}]",\n                                grid,\n                                jepa_encoder=jepa_encoder,\n                                projection_model=projection_model,\n                            )\n                        )\n            else:\n                for split, grids in _iter_task_grids(state.task):\n                    for index, grid in enumerate(grids):\n                        embeddings.append(\n                            _encode_jepa_grid(\n                                f"{split}[{index}]",\n                                grid,\n                                jepa_encoder=jepa_encoder,\n                                projection_model=None,\n                            )\n                        )\n        except Exception as exc:\n            if self.strict_models:\n                raise\n            projection_detail = f"projection failed: {exc}; deterministic ARC-feature fallback used"\n            embeddings = [\n                _feature_grid(f"{split}[{index}]", grid, dim=DEFAULT_JEPA_FEATURE_DIM)\n                for split, grids in _iter_task_grids(state.task)\n                for index, grid in enumerate(grids)\n            ]\n        state.embeddings = tuple(embeddings)\n        if real_jepa_loaded and projection_loaded:\n            status = "jepa_forward_projection_loaded"\n        elif real_jepa_loaded:\n            status = "jepa_forward"\n        elif projection_loaded:\n            status = "projection_loaded"\n        else:\n            status = "fallback"\n\n        if real_jepa_loaded:\n            detail = (\n                f"{jepa_model.describe()}; {projection.describe()}; "\n                f"MYTHOS_ENABLE_REAL_JEPA=1; transformers I-JEPA forward ran; "\n                f"{projection_detail}; produced {len(embeddings)} embeddings"\n            )\n        elif projection_loaded:\n            detail = (\n                f"{jepa_model.describe()}; {projection.describe()}; {projection_detail}; "\n                "no I-JEPA forward pass is run without MYTHOS_ENABLE_REAL_JEPA=1; "\n                f"projected {len(embeddings)} deterministic ARC-feature embeddings"\n            )\n        else:\n            detail = (\n                f"{jepa_model.describe()}; {projection.describe()}; {projection_detail}; "\n                f"MYTHOS_ENABLE_REAL_JEPA={int(real_jepa_enabled)}; no I-JEPA forward pass ran; produced "\n                f"{len(embeddings)} deterministic ARC-feature embeddings"\n            )\n        state.trace.add(\n            "encode_jepa",\n            status,\n            detail,\n        )\n\n    def _plan_hrm_text(self, state: PipelineState) -> None:\n        model = self.model_registry.get("hrm_text")\n        enabled = os.environ.get("MYTHOS_ENABLE_HRM_TEXT", "0") == "1"\n        state.rule_vector = _make_rule_vector(state.task)\n        status = "fallback"\n        detail = (\n            f"{model.describe()}; MYTHOS_ENABLE_HRM_TEXT={int(enabled)}; "\n            "deterministic fallback rule vector derived from train-pair deltas"\n        )\n        if enabled and model.loaded and model.checkpoint_path is not None:\n            try:\n                result = generate_hrm_text_rule(state.task, model.checkpoint_path)\n                state.rule_vector = RuleVector(\n                    source="hrm_text_forward",\n                    description=result.description,\n                    vector=result.vector,\n                )\n                status = "model_forward"\n                detail = (\n                    f"{model.describe()}; HRM-Text forward/generation ran from "\n                    f"{result.model_root}; rule={result.description[:160]!r}"\n                )\n            except HRMTextError as exc:\n                if self.strict_models:\n                    raise\n                status = "fallback"\n                detail = (\n                    f"{model.describe()}; HRM-Text forward failed: {exc}; "\n                    "deterministic fallback rule vector used"\n                )\n        elif model.loaded and not enabled:\n            status = "disabled_checkpoint_loaded"\n        state.trace.add(\n            "plan_hrm_text",\n            status,\n            detail,\n        )\n\n    def _simulate_world_model(self, state: PipelineState) -> None:\n        if state.rule_vector is None:\n            raise RuntimeError("rule vector must exist before world-model simulation")\n        model = self.model_registry.get("world_model")\n        status = "model_loaded" if model.loaded else "fallback"\n        detail = f"{model.describe()}; no world-model checkpoint, so no rollout guidance was produced"\n        if model.loaded and model.checkpoint_path is not None:\n            try:\n                world_model = self._get_world_model(model.checkpoint_path)\n                state.world_rollouts = tuple(_simulate_rollouts(state.task, world_model))\n                detail = (\n                    f"{model.describe()}; generated {len(state.world_rollouts)} "\n                    "test-input latent rollouts"\n                )\n            except Exception as exc:\n                if self.strict_models:\n                    raise\n                status = "fallback"\n                detail = f"{model.describe()}; world-model rollout failed: {exc}"\n        state.trace.add(\n            "simulate_world_model",\n            status,\n            detail,\n        )\n\n    def _adapt_ttt_lora(self, state: PipelineState) -> None:\n        model = self.model_registry.get("ttt_lora")\n        hrm = self.model_registry.get("hrm_l_module")\n        enabled = os.environ.get("MYTHOS_ENABLE_TTT", "0") == "1"\n        status = "model_loaded" if model.loaded else "fallback"\n        if enabled and hrm.loaded:\n            detail = (\n                f"{model.describe()}; MYTHOS_ENABLE_TTT=1, but live LoRA-on-HRM "\n                "is not connected because the external HRM train-step hook is not "\n                "wrapped by this pipeline yet"\n            )\n            status = "not_connected"\n        else:\n            detail = (\n                f"{model.describe()}; per-task LoRA update skipped "\n                f"(MYTHOS_ENABLE_TTT={int(enabled)}, hrm_loaded={hrm.loaded})"\n            )\n        state.trace.add(\n            "adapt_ttt_lora",\n            status,\n            detail,\n        )\n\n    def _execute_hrm_l_module(self, state: PipelineState) -> None:\n        model = self.model_registry.get("hrm_l_module")\n        enable_real_hrm = os.environ.get("MYTHOS_ENABLE_REAL_HRM", "0") == "1"\n        status = "model_loaded_fallback_executor" if model.loaded else "fallback"\n        if enable_real_hrm:\n            try:\n                state.prediction = HRMSolver().solve(state.task)\n                status = "model_loaded"\n                detail = f"{model.describe()}; external HRM prediction path returned output"\n            except SolverError as exc:\n                if self.strict_models:\n                    raise\n                state.prediction = self.executor.solve(state.task)\n                status = "fallback"\n                detail = f"{model.describe()}; HRM execution failed: {exc}; baseline executor used"\n        else:\n            state.prediction = self.executor.solve(state.task)\n            detail = (\n                f"{model.describe()}; baseline executor produced valid output "\n                "because MYTHOS_ENABLE_REAL_HRM is not set"\n            )\n        if state.prediction is not None and state.world_rollouts:\n            state.prediction, guided_count = _apply_world_rollout_attempts(\n                state.task,\n                state.prediction,\n                state.world_rollouts,\n            )\n            detail += f"; world-model rollout guidance replaced attempt_2 for {guided_count} test item(s)"\n        state.trace.add(\n            "execute_hrm_l_module",\n            status,\n            detail,\n        )\n\n    def _decode_output(self, state: PipelineState) -> None:\n        if state.prediction is None:\n            raise RuntimeError("prediction must exist before decode/output")\n        validate_submission_data({state.prediction.task_id: prediction_to_json(state.prediction)})\n        state.trace.add(\n            "decode_output",\n            "ok",\n            f"validated {len(state.prediction.outputs)} two-attempt test predictions",\n        )\n\n    def _get_projection_model(self, checkpoint_path):  # type: ignore[no-untyped-def]\n        if self._projection_model is None:\n            self._projection_model = load_projection_checkpoint(checkpoint_path)\n        return self._projection_model\n\n    def _get_jepa_encoder(self, checkpoint_path):  # type: ignore[no-untyped-def]\n        if self._jepa_encoder is None:\n            self._jepa_encoder = JepaImageEncoder.from_path(checkpoint_path)\n        return self._jepa_encoder\n\n    def _get_world_model(self, checkpoint_path):  # type: ignore[no-untyped-def]\n        if self._world_model is None:\n            self._world_model = load_world_model_checkpoint(checkpoint_path)\n        return self._world_model\n\n\ndef _iter_task_grids(task: ArcTask) -> Iterable[tuple[str, tuple[Grid, ...]]]:\n    yield "train_input", tuple(example.input for example in task.train)\n    yield "train_output", tuple(example.output for example in task.train if example.output is not None)\n    yield "test_input", tuple(example.input for example in task.test)\n\n\ndef _encode_grid(source: str, grid: Grid) -> GridEmbedding:\n    height = len(grid)\n    width = len(grid[0])\n    flat = [cell for row in grid for cell in row]\n    counts = Counter(flat)\n    dominant_color = counts.most_common(1)[0][0]\n    nonzero = sum(1 for cell in flat if cell != 0)\n    total = len(flat)\n    vector = (\n        height / 30.0,\n        width / 30.0,\n        dominant_color / 9.0,\n        nonzero / total,\n        sum(flat) / (9.0 * total),\n    )\n    return GridEmbedding(source=source, shape=(height, width), vector=_rounded(vector))\n\n\ndef _feature_grid(source: str, grid: Grid, *, dim: int) -> GridEmbedding:\n    return GridEmbedding(\n        source=source,\n        shape=(len(grid), len(grid[0])),\n        vector=grid_to_feature_vector(grid, dim),\n    )\n\n\ndef _encode_jepa_grid(\n    source: str,\n    grid: Grid,\n    *,\n    jepa_encoder,\n    projection_model,\n) -> GridEmbedding:  # type: ignore[no-untyped-def]\n    if jepa_encoder is not None:\n        vector = jepa_encoder.encode_grid(grid)\n        vector_source = "i_jepa_forward"\n    else:\n        feature_dim = projection_model.config.input_dim if projection_model is not None else DEFAULT_JEPA_FEATURE_DIM\n        vector = grid_to_feature_vector(grid, feature_dim)\n        vector_source = "deterministic_arc_features"\n\n    if projection_model is not None:\n        vector = _project_vector(vector, projection_model)\n        vector_source += "_projected"\n\n    return GridEmbedding(\n        source=f"{source}.{vector_source}",\n        shape=(len(grid), len(grid[0])),\n        vector=_rounded(float(value) for value in vector),\n    )\n\n\ndef _project_grid(source: str, grid: Grid, projection_model) -> GridEmbedding:  # type: ignore[no-untyped-def]\n    vector = grid_to_feature_vector(grid, projection_model.config.input_dim)\n    return GridEmbedding(\n        source=source,\n        shape=(len(grid), len(grid[0])),\n        vector=_project_vector(vector, projection_model),\n    )\n\n\ndef _project_vector(vector: tuple[float, ...], projection_model) -> tuple[float, ...]:  # type: ignore[no-untyped-def]\n    import torch\n\n    if len(vector) != projection_model.config.input_dim:\n        raise JepaEncodingError(\n            f"projection expects {projection_model.config.input_dim} features, got {len(vector)}"\n        )\n    device = next(projection_model.parameters()).device\n    tensor = torch.tensor([vector], dtype=torch.float32, device=device)\n    with torch.no_grad():\n        projected = projection_model(tensor)[0].detach().cpu().tolist()\n    return _rounded(float(value) for value in projected)\n\n\ndef _simulate_rollouts(task: ArcTask, world_model) -> Iterable[GridEmbedding]:  # type: ignore[no-untyped-def]\n    import torch\n\n    device = next(world_model.parameters()).device\n    rule = torch.tensor(\n        [task_rule_vector(task, world_model.config.rule_dim)],\n        dtype=torch.float32,\n        device=device,\n    )\n    for index, example in enumerate(task.test):\n        z_input = torch.tensor(\n            [grid_to_feature_vector(example.input, world_model.config.z_dim)],\n            dtype=torch.float32,\n            device=device,\n        )\n        with torch.no_grad():\n            rollout = world_model(z_input, rule)[0].detach().cpu().tolist()\n        yield GridEmbedding(\n            source=f"test_input[{index}].world_rollout",\n            shape=(len(example.input), len(example.input[0])),\n            vector=_rounded(float(value) for value in rollout),\n        )\n\n\ndef _apply_world_rollout_attempts(\n    task: ArcTask,\n    prediction: Prediction,\n    rollouts: tuple[GridEmbedding, ...],\n) -> tuple[Prediction, int]:\n    train_outputs = [\n        (grid_to_feature_vector(example.output, len(rollouts[0].vector)), example.output)\n        for example in task.train\n        if example.output is not None\n    ]\n    if not train_outputs:\n        return prediction, 0\n\n    guided_outputs: list[TestPrediction] = []\n    guided_count = 0\n    for index, item in enumerate(prediction.outputs):\n        if index >= len(rollouts):\n            guided_outputs.append(item)\n            continue\n        rollout = rollouts[index]\n        nearest_grid = max(\n            train_outputs,\n            key=lambda candidate: embedding_cosine_similarity(rollout.vector, candidate[0]),\n        )[1]\n        guided_outputs.append(\n            TestPrediction(\n                attempt_1=item.attempt_1,\n                attempt_2=copy_grid(nearest_grid),\n            )\n        )\n        guided_count += 1\n    return Prediction(task_id=prediction.task_id, outputs=tuple(guided_outputs)), guided_count\n\n\ndef _make_rule_vector(task: ArcTask) -> RuleVector:\n    vector = task_rule_vector(task)\n    return RuleVector(\n        source="deterministic_train_pair_delta",\n        description="shape, density, and color-overlap summary from demonstration pairs",\n        vector=vector,\n    )\n\n\ndef _nonzero_count(grid: Grid) -> int:\n    return sum(1 for row in grid for cell in row if cell != 0)\n\n\ndef _color_jaccard(left: Grid, right: Grid) -> float:\n    left_colors = {cell for row in left for cell in row}\n    right_colors = {cell for row in right for cell in row}\n    union = left_colors | right_colors\n    if not union:\n        return 1.0\n    return len(left_colors & right_colors) / len(union)\n\n\ndef _average(values: Iterable[float]) -> float:\n    collected = list(values)\n    return sum(collected) / len(collected) if collected else 0.0\n\n\ndef _rounded(values: Iterable[float]) -> Tuple[float, ...]:\n    return tuple(round(value, 6) for value in values)\n', 'src/mythos/score.py': '"""CLI for scoring a submission against solution JSON."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.arc import ArcValidationError\nfrom mythos.metrics import score_files\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Score an ARC submission JSON file.")\n    parser.add_argument("--pred", required=True, help="Path to submission JSON.")\n    parser.add_argument("--solutions", required=True, help="Path to solution JSON.")\n    args = parser.parse_args(argv)\n\n    try:\n        result = score_files(args.pred, args.solutions)\n    except ArcValidationError as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/solve.py': '"""CLI for running a solver and writing submission JSON."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nfrom mythos.arc import ArcValidationError, load_challenges\nfrom mythos.solvers.base import SolverError\nfrom mythos.solvers.factory import make_solver\nfrom mythos.submission import write_submission\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run a Mythos solver.")\n    parser.add_argument(\n        "--solver",\n        choices=["pipeline", "baseline", "fixture", "symbolic", "hrm", "agentic_repl"],\n        default="pipeline",\n    )\n    parser.add_argument("--model-mode", choices=["fallback", "strict"], default=None)\n    parser.add_argument("--challenges", required=True, help="Path to ARC-style challenges JSON.")\n    parser.add_argument("--out", required=True, help="Output submission JSON path.")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.challenges)\n        solver = make_solver(args.solver, model_mode=args.model_mode)\n        predictions = [solver.solve(task) for task in tasks.values()]\n        write_submission(predictions, args.out)\n    except (ArcValidationError, SolverError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    print(f"Wrote {len(predictions)} predictions to {args.out}")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/solvers/__init__.py': '"""Solver implementations."""\n\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.base import Solver, SolverError\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.solvers.hrm import HRMEnvironmentError, HRMSolver\nfrom mythos.solvers.pipeline import PlannedPipelineSolver\n\n__all__ = [\n    "BaselineSolver",\n    "FixtureSolver",\n    "HRMEnvironmentError",\n    "HRMSolver",\n    "PlannedPipelineSolver",\n    "Solver",\n    "SolverError",\n]\n', 'src/mythos/solvers/base.py': '"""Common solver types."""\n\nfrom __future__ import annotations\n\nfrom typing import Protocol\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.submission import Prediction, TestPrediction\n\n\nclass SolverError(RuntimeError):\n    """Raised when a solver cannot produce a prediction."""\n\n\nclass Solver(Protocol):\n    def solve(self, task: ArcTask) -> Prediction:\n        """Return a two-attempt prediction for every test item in a task."""\n\n\ndef make_prediction(task: ArcTask, attempts: list[tuple[Grid, Grid]]) -> Prediction:\n    if len(attempts) != len(task.test):\n        raise SolverError(\n            f"{task.id}: expected {len(task.test)} test predictions, got {len(attempts)}"\n        )\n    return Prediction(\n        task_id=task.id,\n        outputs=tuple(TestPrediction(attempt_1=a1, attempt_2=a2) for a1, a2 in attempts),\n    )\n', 'src/mythos/solvers/baseline.py': '"""Guaranteed-output baseline solver for smoke runs and Kaggle plumbing tests."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\n\nfrom mythos.arc import ArcTask, Grid, copy_grid\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.submission import Prediction\n\n\nclass BaselineSolver:\n    """Try simple fixture rules, then fall back to valid low-skill predictions."""\n\n    def __init__(self) -> None:\n        self.fixture_solver = FixtureSolver()\n\n    def solve(self, task: ArcTask) -> Prediction:\n        try:\n            return self.fixture_solver.solve(task)\n        except SolverError:\n            attempts = [\n                (copy_grid(example.input), _blank_output_for_task(task, example.input))\n                for example in task.test\n            ]\n            return make_prediction(task, attempts)\n\n\ndef _blank_output_for_task(task: ArcTask, input_grid: Grid) -> Grid:\n    height, width = _fallback_shape(task, input_grid)\n    color = _dominant_output_color(task)\n    return [[color for _ in range(width)] for _ in range(height)]\n\n\ndef _fallback_shape(task: ArcTask, input_grid: Grid) -> tuple[int, int]:\n    output_shapes = {\n        (len(example.output), len(example.output[0]))\n        for example in task.train\n        if example.output is not None\n    }\n    if len(output_shapes) == 1:\n        return next(iter(output_shapes))\n    return len(input_grid), len(input_grid[0])\n\n\ndef _dominant_output_color(task: ArcTask) -> int:\n    counts: Counter[int] = Counter()\n    for example in task.train:\n        if example.output is None:\n            continue\n        for row in example.output:\n            counts.update(row)\n    if not counts:\n        return 0\n    return counts.most_common(1)[0][0]\n', 'src/mythos/solvers/compress_arc.py': '"""Integration wrapper around the vendored CompressARC solver.\n\nCompressARC (see third_party/compress_arc/NOTICE.md for provenance) trains\na small, randomly-initialized network from scratch on each task\'s own demo\npairs via a compression (MDL/VAE) objective -- unlike HRM, it needs no\npretrained checkpoint at all. This file is our own glue code around the\nvendored, unmodified upstream implementation.\n\nNote: `arc_compressor.py` calls `torch.set_default_device(\'cuda\')` at\n*import time*, unconditionally -- there is no CPU fallback, and importing\nit changes torch\'s global default device for the rest of the process.\nRun this solver in an isolated subprocess when combining it with any other\nPyTorch-using code (e.g. the HRM solver) in the same pipeline, the same way\nHRM\'s own dataset builder is already run out-of-process for isolation.\n"""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nimport sys\nimport time\n\nfrom mythos.arc import ArcTask\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.submission import Prediction\n\n_THIRD_PARTY_DIR = Path(__file__).resolve().parents[3] / "third_party" / "compress_arc"\n\n\ndef _ensure_on_path() -> None:\n    path_str = str(_THIRD_PARTY_DIR)\n    if _THIRD_PARTY_DIR.is_dir() and path_str not in sys.path:\n        sys.path.insert(0, path_str)\n\n\n@dataclass(frozen=True)\nclass CompressARCConfig:\n    steps: int = 2000\n    # Wall-clock cutoff per task; solve_task-style early exit keeps whatever\n    # the best-so-far tracked solution is. None = no limit (run all steps).\n    time_limit_seconds: float | None = None\n\n\nclass CompressARCSolver:\n    """Per-task from-scratch compression-based solver (no pretraining)."""\n\n    def __init__(self, config: CompressARCConfig | None = None) -> None:\n        self.config = config or CompressARCConfig()\n\n    def solve(self, task: ArcTask) -> Prediction:\n        _ensure_on_path()\n        import torch  # local: only needed once the vendored deps are on sys.path\n        if not torch.cuda.is_available():\n            raise SolverError("CompressARC requires CUDA: arc_compressor.py hard-sets the default device at import time")\n        import arc_compressor  # import-time side effect: torch.set_default_device(\'cuda\') for the whole process\n        import preprocessing\n        import solution_selection\n        import train as compress_arc_train\n\n        problem = {\n            "train": [{"input": example.input, "output": example.output} for example in task.train],\n            "test": [{"input": example.input} for example in task.test],\n        }\n        torch.manual_seed(0)\n        try:\n            compress_task = preprocessing.Task(task.id, problem, None)\n        except Exception as exc:  # noqa: BLE001 - a malformed/unsupported task must not abort the run\n            raise SolverError(f"{task.id}: CompressARC preprocessing failed: {exc!r}") from exc\n\n        model = arc_compressor.ARCCompressor(compress_task)\n        optimizer = torch.optim.Adam(model.weights_list, lr=0.01, betas=(0.5, 0.9))\n        logger = solution_selection.Logger(compress_task)\n        logger.solution_most_frequent = tuple(((0, 0), (0, 0)) for _ in range(compress_task.n_test))\n        logger.solution_second_most_frequent = tuple(((0, 0), (0, 0)) for _ in range(compress_task.n_test))\n\n        deadline = time.time() + self.config.time_limit_seconds if self.config.time_limit_seconds else None\n        for step in range(self.config.steps):\n            compress_arc_train.take_step(compress_task, model, optimizer, step, logger)\n            if deadline is not None and time.time() > deadline:\n                break\n\n        attempts: list[tuple] = []\n        for example_index in range(compress_task.n_test):\n            attempt_1 = [list(row) for row in logger.solution_most_frequent[example_index]]\n            attempt_2 = [list(row) for row in logger.solution_second_most_frequent[example_index]]\n            attempts.append((attempt_1, attempt_2))\n        return make_prediction(task, attempts)\n', 'src/mythos/solvers/factory.py': '"""Solver factory shared by CLIs."""\n\nfrom __future__ import annotations\n\nimport os\n\nfrom mythos.solvers.baseline import BaselineSolver\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.solvers.hrm import HRMSolver\nfrom mythos.solvers.pipeline import PlannedPipelineSolver\nfrom mythos.solvers.symbolic import SymbolicSolver\n\n\ndef make_solver(name: str, *, model_mode: str | None = None):\n    selected_mode = model_mode or os.environ.get("MYTHOS_MODEL_MODE", "fallback")\n    if selected_mode not in {"fallback", "strict"}:\n        raise ValueError(f"unknown model mode: {selected_mode}")\n    strict_models = selected_mode == "strict"\n    if name == "pipeline":\n        return PlannedPipelineSolver(strict_models=strict_models)\n    if name == "baseline":\n        return BaselineSolver()\n    if name == "fixture":\n        return FixtureSolver()\n    if name == "symbolic":\n        return SymbolicSolver()\n    if name == "hrm":\n        return HRMSolver()\n    if name == "agentic_repl":\n        # Local import: agentic_repl/ is a separate top-level package (kept\n        # apart from the shelved neural path here in src/mythos), and its\n        # real LLM backend needs llama-cpp-python -- selecting any other\n        # solver should never require that dependency to be installed.\n        from agentic_repl.llm.client import LlamaCppClient\n        from agentic_repl.solver import AgenticReplSolver\n\n        return AgenticReplSolver(LlamaCppClient())\n    raise ValueError(f"unknown solver: {name}")\n', 'src/mythos/solvers/fixture.py': '"""Small deterministic solver for the committed toy fixtures."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom typing import Callable, Iterable, List, Optional, Tuple\n\nfrom mythos.arc import ArcTask, Grid, copy_grid, grid_equal\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.submission import Prediction\n\nTransform = Callable[[Grid], Grid]\n\n\n@dataclass(frozen=True)\nclass Candidate:\n    name: str\n    transform: Transform\n\n\nclass FixtureSolver:\n    """Infer one simple transformation from train examples and apply it to tests."""\n\n    def solve(self, task: ArcTask) -> Prediction:\n        candidates = _matching_candidates(task)\n        if not candidates:\n            raise SolverError(f"{task.id}: no fixture transformation matched train examples")\n\n        primary = candidates[0].transform\n        secondary = candidates[1].transform if len(candidates) > 1 else primary\n        attempts = [(primary(example.input), secondary(example.input)) for example in task.test]\n        return make_prediction(task, attempts)\n\n\ndef _matching_candidates(task: ArcTask) -> List[Candidate]:\n    candidates = _base_candidates()\n    candidates.extend(_translation_candidates(task))\n    recolor = _recolor_candidate(task)\n    if recolor is not None:\n        candidates.append(recolor)\n\n    matched: List[Candidate] = []\n    seen_outputs: set[str] = set()\n    for candidate in candidates:\n        if _fits(task, candidate.transform):\n            signature = _candidate_signature(task, candidate.transform)\n            if signature not in seen_outputs:\n                matched.append(candidate)\n                seen_outputs.add(signature)\n    return matched\n\n\ndef _candidate_signature(task: ArcTask, transform: Transform) -> str:\n    return repr([transform(example.input) for example in task.test])\n\n\ndef _fits(task: ArcTask, transform: Transform) -> bool:\n    for example in task.train:\n        if example.output is None:\n            return False\n        try:\n            predicted = transform(example.input)\n        except ValueError:\n            return False\n        if not grid_equal(predicted, example.output):\n            return False\n    return True\n\n\ndef _base_candidates() -> List[Candidate]:\n    return [\n        Candidate("identity", copy_grid),\n        Candidate("mirror_horizontal", _mirror_horizontal),\n        Candidate("mirror_vertical", _mirror_vertical),\n        Candidate("rotate_clockwise", _rotate_clockwise),\n        Candidate("rotate_180", lambda grid: _rotate_clockwise(_rotate_clockwise(grid))),\n        Candidate("rotate_counterclockwise", _rotate_counterclockwise),\n    ]\n\n\ndef _mirror_horizontal(grid: Grid) -> Grid:\n    return [list(reversed(row)) for row in grid]\n\n\ndef _mirror_vertical(grid: Grid) -> Grid:\n    return [row[:] for row in reversed(grid)]\n\n\ndef _rotate_clockwise(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid[::-1])]\n\n\ndef _rotate_counterclockwise(grid: Grid) -> Grid:\n    return [list(row) for row in zip(*grid)][::-1]\n\n\ndef _recolor_candidate(task: ArcTask) -> Optional[Candidate]:\n    mapping: dict[int, int] = {}\n    for example in task.train:\n        if example.output is None:\n            return None\n        if len(example.input) != len(example.output) or len(example.input[0]) != len(example.output[0]):\n            return None\n        for in_row, out_row in zip(example.input, example.output):\n            for in_cell, out_cell in zip(in_row, out_row):\n                previous = mapping.setdefault(in_cell, out_cell)\n                if previous != out_cell:\n                    return None\n\n    def transform(grid: Grid) -> Grid:\n        return [[mapping.get(cell, cell) for cell in row] for row in grid]\n\n    return Candidate("recolor", transform)\n\n\ndef _translation_candidates(task: ArcTask) -> List[Candidate]:\n    offsets: Optional[set[tuple[int, int]]] = None\n    for example in task.train:\n        if example.output is None:\n            return []\n        example_offsets = set(_valid_translation_offsets(example.input, example.output))\n        offsets = example_offsets if offsets is None else offsets & example_offsets\n    return [\n        Candidate(f"translate_{dr}_{dc}", _translate_transform(dr, dc))\n        for dr, dc in sorted(offsets or set())\n        if dr != 0 or dc != 0\n    ]\n\n\ndef _valid_translation_offsets(source: Grid, target: Grid) -> Iterable[tuple[int, int]]:\n    if len(source) != len(target) or len(source[0]) != len(target[0]):\n        return []\n\n    source_cells = _foreground_cells(source)\n    target_cells = _foreground_cells(target)\n    if len(source_cells) != len(target_cells):\n        return []\n    if not source_cells and not target_cells:\n        return [(0, 0)]\n\n    offsets = []\n    first_r, first_c, first_value = source_cells[0]\n    for target_r, target_c, target_value in target_cells:\n        if target_value != first_value:\n            continue\n        dr = target_r - first_r\n        dc = target_c - first_c\n        try:\n            translated = _translate_grid(source, dr, dc)\n        except ValueError:\n            continue\n        if translated == target:\n            offsets.append((dr, dc))\n    return offsets\n\n\ndef _foreground_cells(grid: Grid) -> List[tuple[int, int, int]]:\n    return [\n        (row_idx, col_idx, cell)\n        for row_idx, row in enumerate(grid)\n        for col_idx, cell in enumerate(row)\n        if cell != 0\n    ]\n\n\ndef _translate_transform(dr: int, dc: int) -> Transform:\n    def transform(grid: Grid) -> Grid:\n        return _translate_grid(grid, dr, dc)\n\n    return transform\n\n\ndef _translate_grid(grid: Grid, dr: int, dc: int) -> Grid:\n    height = len(grid)\n    width = len(grid[0])\n    translated = [[0 for _ in range(width)] for _ in range(height)]\n    for row_idx, row in enumerate(grid):\n        for col_idx, cell in enumerate(row):\n            if cell == 0:\n                continue\n            next_r = row_idx + dr\n            next_c = col_idx + dc\n            if next_r < 0 or next_r >= height or next_c < 0 or next_c >= width:\n                raise ValueError("translation moves cell out of bounds")\n            translated[next_r][next_c] = cell\n    return translated\n', 'src/mythos/solvers/hrm.py': '"""External HRM adapter and environment checks."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nimport importlib\nimport json\nimport os\nfrom pathlib import Path\nimport sys\nfrom typing import Any\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.augment import inverse_transform, transform_task\nfrom mythos.features import ARC_MAX_SIZE, hrm_sequence_to_grid, output_shape_hint\nfrom mythos.hrm_dataset import build_hrm_dataset, default_run_dir, prepare_hrm_raw_dataset\nfrom mythos.losses import genie_background_consistency_loss\nfrom mythos.lora import inject_lora_adapters, lora_parameters\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.submission import Prediction\n\n\nclass HRMEnvironmentError(SolverError):\n    """Raised when the external HRM runtime is not ready."""\n\n\n@dataclass(frozen=True)\nclass HRMEnvironment:\n    repo_dir: Path\n    checkpoint_path: Path\n\n    @classmethod\n    def from_env(cls) -> "HRMEnvironment":\n        repo_value = os.environ.get("HRM_REPO_DIR")\n        checkpoint_value = os.environ.get("HRM_CHECKPOINT_PATH")\n        if not repo_value:\n            raise HRMEnvironmentError("HRM_REPO_DIR is required for HRM execution")\n        if not checkpoint_value:\n            raise HRMEnvironmentError("HRM_CHECKPOINT_PATH is required for HRM execution")\n        return cls(repo_dir=Path(repo_value), checkpoint_path=Path(checkpoint_value))\n\n    def validate(self, *, require_cuda: bool = True) -> None:\n        if not self.repo_dir.exists():\n            raise HRMEnvironmentError(f"HRM_REPO_DIR does not exist: {self.repo_dir}")\n        if not (self.repo_dir / "evaluate.py").exists():\n            raise HRMEnvironmentError(f"HRM checkout is missing evaluate.py: {self.repo_dir}")\n        if not (self.repo_dir / "dataset" / "build_arc_dataset.py").exists():\n            raise HRMEnvironmentError(\n                f"HRM checkout is missing dataset/build_arc_dataset.py: {self.repo_dir}"\n            )\n        if not self.checkpoint_path.exists():\n            raise HRMEnvironmentError(f"HRM_CHECKPOINT_PATH does not exist: {self.checkpoint_path}")\n\n        torch = self._import_torch()\n        if require_cuda and not torch.cuda.is_available():\n            raise HRMEnvironmentError("HRM execution requires CUDA; torch.cuda.is_available() is false")\n\n    def import_modules(self) -> dict[str, Any]:\n        self._add_repo_to_path()\n        modules = {}\n        for module_name in ("pretrain", "evaluate"):\n            try:\n                modules[module_name] = importlib.import_module(module_name)\n            except Exception as exc:  # pragma: no cover - depends on external HRM deps.\n                raise HRMEnvironmentError(\n                    f"failed to import HRM module {module_name!r} from {self.repo_dir}: {exc}"\n                ) from exc\n        return modules\n\n    def load_checkpoint(self) -> Any:\n        torch = self._import_torch()\n        map_location = "cuda" if torch.cuda.is_available() else "cpu"\n        try:\n            return torch.load(\n                self.checkpoint_path,\n                map_location=map_location,\n                weights_only=False,\n            )\n        except TypeError:\n            try:\n                return torch.load(self.checkpoint_path, map_location=map_location)\n            except Exception as exc:  # pragma: no cover - depends on checkpoint format.\n                raise HRMEnvironmentError(\n                    f"failed to load HRM checkpoint {self.checkpoint_path}: {exc}"\n                ) from exc\n        except Exception as exc:  # pragma: no cover - depends on checkpoint format.\n            raise HRMEnvironmentError(f"failed to load HRM checkpoint {self.checkpoint_path}: {exc}") from exc\n\n    def _add_repo_to_path(self) -> None:\n        repo = str(self.repo_dir.resolve())\n        if repo not in sys.path:\n            sys.path.insert(0, repo)\n\n    @staticmethod\n    def _import_torch() -> Any:\n        try:\n            return importlib.import_module("torch")\n        except Exception as exc:  # pragma: no cover - torch is optional locally.\n            raise HRMEnvironmentError("PyTorch is required for HRM execution") from exc\n\n\nclass HRMSolver:\n    """External HRM solver for CUDA/Kaggle smoke inference."""\n\n    def __init__(self, env: HRMEnvironment | None = None) -> None:\n        self.env = env\n\n    def solve(self, task: ArcTask) -> Prediction:\n        env = self.env or HRMEnvironment.from_env()\n        env.validate(require_cuda=True)\n        runner = HRMInferenceRunner(env)\n        return runner.solve_task(task)\n\n\n@dataclass(frozen=True)\nclass HRMInferenceRunner:\n    env: HRMEnvironment\n    num_aug: int = 0\n\n    def solve_task(self, task: ArcTask) -> Prediction:\n        run_dir = default_run_dir() / "hrm_inference" / task.id\n        raw_dir = prepare_hrm_raw_dataset(\n            (task,),\n            run_dir / "raw" / "ARC-AGI-2" / "data",\n            allow_dummy_test_outputs=True,\n        )\n        dataset_dir = run_dir / "data" / "arc-2-one-task"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir,\n            raw_data_dir=raw_dir,\n            output_dir=dataset_dir,\n            num_aug=self.num_aug,\n        )\n        prediction_tokens = self._run_external_evaluate(dataset_dir, run_dir / "outputs")\n        return self._tokens_to_prediction(task, prediction_tokens)\n\n    def solve_tasks(self, tasks: list[ArcTask] | tuple[ArcTask, ...]) -> list[Prediction]:\n        task_list = list(tasks)\n        if not task_list:\n            return []\n        run_dir = default_run_dir() / "hrm_inference_batch"\n        raw_dir = prepare_hrm_raw_dataset(\n            task_list,\n            run_dir / "raw" / "ARC-AGI-2" / "data",\n            allow_dummy_test_outputs=True,\n        )\n        dataset_dir = run_dir / "data" / "arc-2-batch"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir,\n            raw_data_dir=raw_dir,\n            output_dir=dataset_dir,\n            num_aug=self.num_aug,\n        )\n        prediction_tokens = self._run_external_evaluate(dataset_dir, run_dir / "outputs")\n        predictions: list[Prediction] = []\n        cursor = 0\n        for task in task_list:\n            count = len(task.test)\n            predictions.append(self._tokens_to_prediction(task, prediction_tokens[cursor : cursor + count]))\n            cursor += count\n        if cursor > len(prediction_tokens):\n            raise HRMEnvironmentError(\n                f"HRM returned {len(prediction_tokens)} predictions for {cursor} requested test inputs"\n            )\n        return predictions\n\n    def _tokens_to_prediction(\n        self,\n        task: ArcTask,\n        prediction_tokens: list[tuple[list[int], list[int]]],\n    ) -> Prediction:\n        if len(prediction_tokens) < len(task.test):\n            raise HRMEnvironmentError(\n                f"{task.id}: HRM returned {len(prediction_tokens)} predictions for "\n                f"{len(task.test)} test inputs"\n            )\n\n        attempts = []\n        for index, example in enumerate(task.test):\n            shape_hint = output_shape_hint(task, example.input)\n            top1, top2 = prediction_tokens[index]\n            attempts.append(\n                (\n                    hrm_sequence_to_grid(top1, shape_hint=shape_hint),\n                    hrm_sequence_to_grid(top2, shape_hint=shape_hint),\n                )\n            )\n        return make_prediction(task, attempts)\n\n    def _run_external_evaluate(self, dataset_dir: Path, output_dir: Path) -> list[tuple[list[int], list[int]]]:\n        torch = HRMEnvironment._import_torch()\n        modules = self.env.import_modules()\n        pretrain = modules["pretrain"]\n\n        output_dir.mkdir(parents=True, exist_ok=True)\n        config = _load_hrm_config(self.env.checkpoint_path, dataset_dir=dataset_dir, output_dir=output_dir)\n        train_loader, train_metadata = pretrain.create_dataloader(\n            config,\n            "train",\n            test_set_mode=False,\n            epochs_per_iter=1,\n            global_batch_size=config.global_batch_size,\n            rank=0,\n            world_size=1,\n        )\n        eval_loader, eval_metadata = pretrain.create_dataloader(\n            config,\n            "test",\n            test_set_mode=True,\n            epochs_per_iter=1,\n            global_batch_size=config.global_batch_size,\n            rank=0,\n            world_size=1,\n        )\n        del train_loader\n\n        train_state = pretrain.init_train_state(config, train_metadata, world_size=1)\n        checkpoint = torch.load(self.env.checkpoint_path, map_location="cuda", weights_only=False)\n        _load_hrm_checkpoint_best_effort(train_state.model, checkpoint)\n        train_state.step = 0\n        train_state.model.eval()\n        pretrain.evaluate(config, train_state, eval_loader, eval_metadata, rank=0, world_size=1)\n        return _load_decoded_hrm_predictions(output_dir)\n\n\n@dataclass(frozen=True)\nclass TTTConfig:\n    rank: int = 16\n    steps: int = 20\n    lr: float = 1e-3\n    grad_clip_norm: float = 1.0\n    # A step whose (pre-clip) LoRA gradient norm exceeds this is treated as an\n    # explosion: skipped and rolled back rather than clipped-and-applied.\n    explosion_grad_norm: float = 20.0\n    # Must be fixed and consistent across every task\'s forward passes: the\n    # puzzle embedding\'s sparse-update buffer (local_weights) is allocated\n    # once at model-init time sized to whatever global_batch_size was used\n    # then, and cannot accept a different batch size later -- confirmed by a\n    # real run: "expand: attempting to expand a dimension of length 4 -> 32"\n    # once the sizing config (32) and a per-task batch (4) diverged. 2 is the\n    # ARC-guaranteed minimum train-pair count for any task, so it\'s never\n    # dropped as an incomplete batch regardless of augmentation settings.\n    batch_size: int = 2\n    # Weight for the Genie-style background-consistency auxiliary loss (see\n    # mythos.losses.genie_background_consistency_loss): penalizes the model\n    # for changing background cells during TTT, the master plan\'s named fix\n    # for TTT "hallucinating" -- objects vanishing, backgrounds recoloring --\n    # under pure demo-pair supervision. 0 disables it.\n    genie_weight: float = 0.1\n    # Dihedral transform indices (see mythos.augment) to ensemble across at\n    # inference: each gets its own from-scratch TTT fit (demo pairs and test\n    # input transformed together, so any orientation-dependent rule stays\n    # internally consistent) and the un-augmented predictions are voted\n    # over. (0,) = identity only, i.e. ensembling disabled -- the original\n    # single-view behavior. Costs roughly len(ensemble_transforms)x the TTT\n    # compute per task, so keep this short.\n    ensemble_transforms: tuple[int, ...] = (0,)\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "rank": self.rank,\n            "steps": self.steps,\n            "lr": self.lr,\n            "grad_clip_norm": self.grad_clip_norm,\n            "explosion_grad_norm": self.explosion_grad_norm,\n            "batch_size": self.batch_size,\n            "genie_weight": self.genie_weight,\n            "ensemble_transforms": list(self.ensemble_transforms),\n        }\n\n\nclass HRMTTTRunner:\n    """Per-task test-time training on top of the loaded HRM checkpoint.\n\n    Loads the model once and injects LoRA adapters once (the backbone is\n    frozen at injection time and never receives gradients). For each task:\n    resets the LoRA adapters to their initial no-op state, runs `ttt.steps`\n    gradient-descent steps against only that task\'s own train pairs, then\n    runs inference with the now-adapted model before moving to the next task.\n\n    The model\'s puzzle-identifier embedding table is sized once from a\n    dataset build over *all* tasks (matching the batched eval-only path) so\n    it\'s safely oversized for any single task\'s tiny per-task dataset build,\n    whose own identifier indices will always fit inside it.\n    """\n\n    def __init__(self, env: HRMEnvironment, ttt: TTTConfig | None = None, num_aug: int = 0) -> None:\n        self.env = env\n        self.ttt = ttt or TTTConfig()\n        self.num_aug = num_aug\n        self._pretrain: Any = None\n        self._train_state: Any = None\n        self._lora_report: Any = None\n        self._lora_init_snapshot: dict[str, Any] = {}\n        self._backbone_verified = False\n\n    def solve_tasks(self, tasks: list[ArcTask] | tuple[ArcTask, ...]) -> list[Prediction]:\n        task_list = list(tasks)\n        if not task_list:\n            return []\n\n        self._ensure_model_loaded(task_list)\n\n        predictions: list[Prediction] = []\n        for task in task_list:\n            predictions.append(self._solve_one_task_ensembled(task))\n        return predictions\n\n    def _solve_one_task_ensembled(self, task: ArcTask) -> Prediction:\n        transform_indices = self.ttt.ensemble_transforms or (0,)\n        if tuple(transform_indices) == (0,):\n            return self._solve_one_task_with_ttt(task)  # unchanged single-view path\n\n        # Per test item, every un-augmented candidate grid seen across views\n        # (both attempts from every view all count as votes).\n        candidates_per_item: list[list[Grid]] = [[] for _ in task.test]\n        for index in transform_indices:\n            view_task = task if index == 0 else transform_task(task, index, id_suffix=f"__aug{index}")\n            try:\n                prediction = self._solve_one_task_with_ttt(view_task)\n            except Exception as exc:  # noqa: BLE001 - one bad view must not sink the whole ensemble\n                print(f"TTT: {task.id}: ensemble view {index} failed: {exc!r}; skipping this view")\n                continue\n            for item_index, output in enumerate(prediction.outputs):\n                candidates_per_item[item_index].append(inverse_transform(index, output.attempt_1))\n                candidates_per_item[item_index].append(inverse_transform(index, output.attempt_2))\n\n        attempts: list[tuple[Grid, Grid]] = []\n        for candidates in candidates_per_item:\n            attempts.append(_top_two_by_vote(candidates))\n        return make_prediction(task, attempts)\n\n    def _ensure_model_loaded(self, task_list: list[ArcTask]) -> None:\n        if self._train_state is not None:\n            return\n        torch = HRMEnvironment._import_torch()\n        modules = self.env.import_modules()\n        self._pretrain = modules["pretrain"]\n\n        # Dataset build over every task purely to size the model\'s architecture\n        # (vocab size, puzzle-identifier count) the same way the working\n        # eval-only batched path already does -- not used for training or eval.\n        sizing_run_dir = default_run_dir() / "hrm_ttt_sizing"\n        raw_dir = prepare_hrm_raw_dataset(\n            task_list, sizing_run_dir / "raw" / "ARC-AGI-2" / "data", allow_dummy_test_outputs=True\n        )\n        sizing_dataset_dir = sizing_run_dir / "data" / "arc-2-sizing"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir, raw_data_dir=raw_dir, output_dir=sizing_dataset_dir, num_aug=self.num_aug\n        )\n        sizing_config = _load_hrm_config(\n            self.env.checkpoint_path, dataset_dir=sizing_dataset_dir, output_dir=sizing_run_dir / "outputs"\n        )\n        # Must match what every per-task TTT call below uses (self.ttt.batch_size),\n        # not the eval-only path\'s larger default -- see TTTConfig.batch_size.\n        sizing_config.global_batch_size = self.ttt.batch_size\n        sizing_loader, sizing_metadata = self._pretrain.create_dataloader(\n            sizing_config, "train", test_set_mode=False, epochs_per_iter=1,\n            global_batch_size=sizing_config.global_batch_size, rank=0, world_size=1,\n        )\n        del sizing_loader\n\n        train_state = self._pretrain.init_train_state(sizing_config, sizing_metadata, world_size=1)\n        checkpoint = torch.load(self.env.checkpoint_path, map_location="cuda", weights_only=False)\n        _load_hrm_checkpoint_best_effort(train_state.model, checkpoint)\n        train_state.step = 0\n\n        report = inject_lora_adapters(\n            train_state.model,\n            rank=self.ttt.rank,\n            # Attention-only adapters cap how much the model\'s actual per-task\n            # behavior can change; the MLP layers (gate_up_proj/down_proj) are\n            # roughly half the model\'s parameters and are where most of the\n            # per-token transformation logic lives. Widening the LoRA target\n            # set gives more real capacity to adapt, instead of just pushing\n            # rank/LR higher on a narrower slice of the model (confirmed\n            # unstable: v36\'s rank=64 attention-only run diverged).\n            target_patterns=("self_attn", "attn", "qkv_proj", "o_proj", "gate_up_proj", "down_proj"),\n            freeze_backbone=True,\n        )\n        print(\n            f"TTT: injected LoRA (rank={self.ttt.rank}) into {len(report.injected_modules)} module(s); "\n            f"{report.trainable_parameters} trainable / {report.frozen_parameters} frozen parameters"\n        )\n        self._lora_report = report\n        self._lora_init_snapshot = {\n            name: parameter.detach().clone()\n            for name, parameter in train_state.model.named_parameters()\n            if "lora_" in name\n        }\n        self._train_state = train_state\n\n    def _reset_lora(self) -> None:\n        torch = HRMEnvironment._import_torch()\n        with torch.no_grad():\n            for name, parameter in self._train_state.model.named_parameters():\n                snapshot = self._lora_init_snapshot.get(name)\n                if snapshot is not None:\n                    parameter.copy_(snapshot)\n\n    def _solve_one_task_with_ttt(self, task: ArcTask) -> Prediction:\n        from mythos.lora import changed_frozen_parameters, snapshot_frozen_parameters\n\n        torch = HRMEnvironment._import_torch()\n        pretrain = self._pretrain\n        train_state = self._train_state\n\n        run_dir = default_run_dir() / "hrm_ttt" / task.id\n        raw_dir = prepare_hrm_raw_dataset(\n            (task,), run_dir / "raw" / "ARC-AGI-2" / "data", allow_dummy_test_outputs=True\n        )\n        dataset_dir = run_dir / "data" / "arc-2-ttt"\n        build_hrm_dataset(\n            hrm_repo_dir=self.env.repo_dir, raw_data_dir=raw_dir, output_dir=dataset_dir, num_aug=self.num_aug\n        )\n        output_dir = run_dir / "outputs"\n        output_dir.mkdir(parents=True, exist_ok=True)\n        config = _load_hrm_config(self.env.checkpoint_path, dataset_dir=dataset_dir, output_dir=output_dir)\n        # Must equal the sizing config\'s batch size (see TTTConfig.batch_size):\n        # the puzzle embedding\'s sparse-update buffer is allocated once at\n        # model-init time and cannot accept a different batch size per task.\n        original_batch_size = config.global_batch_size\n        config.global_batch_size = self.ttt.batch_size\n        print(f"TTT: {task.id}: global_batch_size {original_batch_size} -> {config.global_batch_size}")\n\n        train_loader, _train_metadata = pretrain.create_dataloader(\n            config, "train", test_set_mode=False, epochs_per_iter=1,\n            global_batch_size=config.global_batch_size, rank=0, world_size=1,\n        )\n        eval_loader, eval_metadata = pretrain.create_dataloader(\n            config, "test", test_set_mode=True, epochs_per_iter=1,\n            global_batch_size=config.global_batch_size, rank=0, world_size=1,\n        )\n\n        self._reset_lora()\n        lora_params = lora_parameters(train_state.model)\n        optimizer = torch.optim.AdamW(lora_params, lr=self.ttt.lr)\n\n        backbone_snapshot = None\n        if not self._backbone_verified:\n            backbone_snapshot = snapshot_frozen_parameters(train_state.model)\n\n        # Diagnostic: lora_b is zero-initialized (a fresh LoRA adapter is a\n        # mathematical no-op), so any nonzero value after training proves\n        # gradients actually flowed and the optimizer actually updated\n        # something, independent of whether the eval predictions changed.\n        lora_b_before = sum(p.detach().abs().sum().item() for name, p in train_state.model.named_parameters() if name.endswith("lora_b"))\n\n        train_state.model.train()\n        carry = None\n        skipped_steps = 0\n        first_loss = None\n        last_loss = None\n        step_index = 0\n        genie_enabled = self.ttt.genie_weight > 0\n        genie_applied = 0\n        return_keys = ["logits"] if genie_enabled else []\n        for _ in range(self.ttt.steps):\n            for _set_name, batch, global_batch_size in train_loader:\n                batch = {key: (value.to("cuda") if hasattr(value, "to") else value) for key, value in batch.items()}\n                if carry is None:\n                    # initial_carry() creates some internal state tensors (e.g. the\n                    # `halted` flag) without an explicit device argument, relying on\n                    # the ambient default-device context to land them on CUDA --\n                    # confirmed both by HRM\'s own evaluate() doing the same thing\n                    # and by a real run failing with "Unhandled FakeTensor Device\n                    # Propagation for aten.where.self, found two different devices\n                    # cpu, cuda:0" without this wrapper.\n                    with torch.device("cuda"):\n                        carry = train_state.model.initial_carry(batch)\n                carry, primary_loss, _metrics, preds, _all_finish = train_state.model(\n                    carry=carry, batch=batch, return_keys=return_keys\n                )\n                loss = primary_loss\n                # Only "logits" needs to come from the model\'s return_keys -- the\n                # input grid is already in hand as batch["inputs"] (what we\'re\n                # feeding in, not something the model needs to report back).\n                # Requiring all_finish (the model\'s own halt state) first made\n                # this never fire in a real run: 0/50 steps across every task,\n                # since a single training forward call rarely reaches full ACT\n                # convergence within the step budget. logits are populated on\n                # every call regardless of halt state, so use those directly --\n                # a consistency signal on the model\'s current best guess is\n                # still useful even before it\'s fully converged.\n                if genie_enabled:\n                    try:\n                        genie_term = _batch_genie_loss(preds, batch.get("inputs"))\n                        if genie_term is not None:\n                            loss = primary_loss + self.ttt.genie_weight * genie_term\n                            genie_applied += 1\n                    except Exception as exc:  # noqa: BLE001 - auxiliary loss must never abort real TTT\n                        print(f"TTT: {task.id}: disabling Genie loss after a failure: {exc!r}")\n                        genie_enabled = False\n                        return_keys = []\n                loss_value = float(loss.detach())\n                if first_loss is None:\n                    first_loss = loss_value\n                last_loss = loss_value\n                optimizer.zero_grad(set_to_none=True)\n                (loss / max(1, global_batch_size)).backward()\n                grad_norm = torch.nn.utils.clip_grad_norm_(lora_params, self.ttt.explosion_grad_norm)\n                if not torch.isfinite(grad_norm) or grad_norm >= self.ttt.explosion_grad_norm:\n                    skipped_steps += 1\n                    optimizer.zero_grad(set_to_none=True)\n                    continue\n                torch.nn.utils.clip_grad_norm_(lora_params, self.ttt.grad_clip_norm)\n                optimizer.step()\n                step_index += 1\n        if skipped_steps:\n            print(f"TTT: {task.id}: rolled back {skipped_steps}/{self.ttt.steps} step(s) on gradient explosion")\n        if self.ttt.genie_weight > 0:\n            print(f"TTT: {task.id}: Genie consistency loss applied on {genie_applied}/{step_index} step(s)")\n\n        lora_b_after = sum(p.detach().abs().sum().item() for name, p in train_state.model.named_parameters() if name.endswith("lora_b"))\n        print(\n            f"TTT: {task.id}: loss {first_loss!r} -> {last_loss!r} over {step_index} applied step(s); "\n            f"sum(|lora_b|) {lora_b_before:.6f} -> {lora_b_after:.6f}"\n        )\n\n        if backbone_snapshot is not None:\n            changed = changed_frozen_parameters(train_state.model, backbone_snapshot)\n            if changed:\n                print(f"TTT WARNING: {len(changed)} frozen backbone parameter(s) changed during TTT: {changed[:5]}")\n            else:\n                print("TTT: verified frozen backbone parameters are unchanged after a TTT run")\n            self._backbone_verified = True\n\n        train_state.model.eval()\n        pretrain.evaluate(config, train_state, eval_loader, eval_metadata, rank=0, world_size=1)\n        tokens = _load_decoded_hrm_predictions(output_dir)\n        return HRMInferenceRunner(self.env)._tokens_to_prediction(task, tokens)\n\n\ndef _top_two_by_vote(candidates: list[Grid]) -> tuple[Grid, Grid]:\n    """Pick the two most-common grids among candidates (majority vote across ensemble views)."""\n\n    if not candidates:\n        return [[0]], [[0]]\n    counts = Counter(tuple(tuple(row) for row in grid) for grid in candidates)\n    ranked = [[list(row) for row in flat] for flat, _ in counts.most_common(2)]\n    if len(ranked) == 1:\n        ranked.append(ranked[0])\n    return ranked[0], ranked[1]\n\n\ndef _load_hrm_config(checkpoint_path: Path, *, dataset_dir: Path, output_dir: Path):  # type: ignore[no-untyped-def]\n    try:\n        import yaml\n        from pretrain import PretrainConfig\n    except Exception as exc:  # pragma: no cover - depends on external HRM deps.\n        raise HRMEnvironmentError(f"failed to import HRM config dependencies: {exc}") from exc\n\n    config_path = checkpoint_path.parent / "all_config.yaml"\n    if not config_path.exists():\n        raise HRMEnvironmentError(f"HRM checkpoint directory is missing all_config.yaml: {config_path}")\n    with config_path.open("r", encoding="utf-8") as handle:\n        config = PretrainConfig(**yaml.safe_load(handle))\n    config.data_path = str(dataset_dir)\n    config.checkpoint_path = str(output_dir)\n    config.eval_save_outputs = ["inputs", "puzzle_identifiers", "logits"]\n    if "HRM_GLOBAL_BATCH_SIZE" in os.environ:\n        config.global_batch_size = int(os.environ["HRM_GLOBAL_BATCH_SIZE"])\n    return config\n\n\ndef _batch_genie_loss(preds: Any, inputs_batch: Any) -> Any:\n    """Average Genie background-consistency loss across a training batch.\n\n    Decodes each example\'s own input tokens (from the batch fed to the model,\n    not something the model needs to report back) to a grid -- no need to\n    match against the original un-augmented ArcTask, since the preserve mask\n    is derived from the decoded grid\'s own dominant color -- and penalizes\n    the model\'s predicted logits for changing cells that should stay\n    background. Returns None (rather than raising) when preds doesn\'t\n    contain what\'s needed, so the caller\'s own try/except only has to guard\n    against genuine failures.\n    """\n    if not preds or "logits" not in preds or inputs_batch is None:\n        return None\n    logits_batch = preds["logits"]\n    if logits_batch is None or logits_batch.shape[0] == 0:\n        return None\n    # logits comes back as [batch, 900, vocab_size] -- a flat HRM token\n    # sequence, not a [H, W, 10] grid (confirmed by a real run: "logits must\n    # have rank 3 or rank 4"). Reshape to the 30x30 canvas, then slice out\n    # just the 10 color-token channels (HRM\'s vocab is PAD=0, EOS=1,\n    # colors=2..11 -- see grid_to_hrm_sequence/hrm_sequence_to_grid, the same\n    # scheme already used to decode this model\'s own predicted output\n    # tokens) since genie_background_consistency_loss compares against plain\n    # 0-9 color indices.\n    per_example_losses = []\n    for example_index in range(logits_batch.shape[0]):\n        input_tokens = inputs_batch[example_index].detach().cpu().tolist()\n        decoded_input = hrm_sequence_to_grid(input_tokens)\n        example_logits = logits_batch[example_index].reshape(ARC_MAX_SIZE, ARC_MAX_SIZE, -1)[:, :, 2:12]\n        per_example_losses.append(genie_background_consistency_loss(example_logits, decoded_input))\n    if not per_example_losses:\n        return None\n    return sum(per_example_losses) / len(per_example_losses)\n\n\ndef _load_hrm_checkpoint_best_effort(model: Any, checkpoint: dict[str, Any]) -> None:\n    """Load the pretrained checkpoint, keeping randomly-initialized weights for any\n    key whose shape doesn\'t match.\n\n    HRM\'s `puzzle_emb` is a per-puzzle lookup table sized to the exact puzzle\n    identifier vocabulary of whatever dataset it was trained on (the public\n    checkpoint: 1,045,829 entries). A dataset built from a different task set\n    (ours: 240 tasks) gets fresh, unrelated identifier indices, so this table\n    can never meaningfully transfer -- there is no "fix" for that mismatch,\n    only whether to keep evaluating with it randomly initialized (this) or\n    fail outright. Every other weight (attention/MLP layers, token/H/L init,\n    LM head) is the real pretrained model and does transfer correctly.\n    """\n    model_state = model.state_dict()\n    filtered: dict[str, Any] = {}\n    skipped: list[str] = []\n    for key, value in checkpoint.items():\n        candidates = (key, f"_orig_mod.{key}", key.removeprefix("_orig_mod."))\n        matched_key = next((name for name in candidates if name in model_state), None)\n        if matched_key is None:\n            skipped.append(f"{key}: not present in model")\n            continue\n        if tuple(model_state[matched_key].shape) != tuple(value.shape):\n            skipped.append(\n                f"{matched_key}: checkpoint={tuple(value.shape)} model={tuple(model_state[matched_key].shape)}"\n            )\n            continue\n        filtered[matched_key] = value\n\n    missing, unexpected = model.load_state_dict(filtered, strict=False, assign=True)\n    print(f"HRM checkpoint: loaded {len(filtered)}/{len(model_state)} weight tensors from the pretrained checkpoint")\n    if skipped:\n        print(f"HRM checkpoint: kept randomly-initialized (shape/name mismatch) for {len(skipped)} key(s): {skipped}")\n    if missing:\n        print(f"HRM checkpoint: {len(missing)} model key(s) had no checkpoint match: {list(missing)}")\n    if unexpected:\n        print(f"HRM checkpoint: {len(unexpected)} checkpoint key(s) were unused: {list(unexpected)}")\n\n\ndef _load_decoded_hrm_predictions(output_dir: Path) -> list[tuple[list[int], list[int]]]:\n    torch = HRMEnvironment._import_torch()\n    pred_files = sorted(output_dir.glob("step_*_all_preds.*"))\n    if not pred_files:\n        raise HRMEnvironmentError(f"HRM evaluation wrote no prediction files in {output_dir}")\n\n    raw = torch.load(pred_files[0], map_location="cpu", weights_only=False)\n    if "logits" not in raw:\n        raise HRMEnvironmentError(\n            f"HRM prediction file {pred_files[0]} is missing logits; keys={sorted(raw)}"\n        )\n    logits = raw["logits"]\n    topk = logits.topk(k=2, dim=-1).indices\n    decoded: list[tuple[list[int], list[int]]] = []\n    for row in range(topk.shape[0]):\n        top1 = topk[row, :, 0].to(dtype=torch.int64).tolist()\n        top2 = topk[row, :, 1].to(dtype=torch.int64).tolist()\n        decoded.append((top1, top2))\n    return decoded\n', 'src/mythos/solvers/pipeline.py': '"""Solver wrapper for the plan-aligned Project Mythos pipeline."""\n\nfrom __future__ import annotations\n\nfrom mythos.arc import ArcTask\nfrom mythos.models import ModelRegistry\nfrom mythos.pipeline import PipelineTrace, PlannedPipeline\nfrom mythos.submission import Prediction\n\n\nclass PlannedPipelineSolver:\n    """Solver that runs every task through the master-plan stage boundaries."""\n\n    def __init__(\n        self,\n        pipeline: PlannedPipeline | None = None,\n        *,\n        model_registry: ModelRegistry | None = None,\n        strict_models: bool = False,\n    ) -> None:\n        self.pipeline = pipeline or PlannedPipeline(\n            model_registry=model_registry,\n            strict_models=strict_models,\n        )\n        self.traces: dict[str, PipelineTrace] = {}\n        self.last_trace: PipelineTrace | None = None\n\n    def solve(self, task: ArcTask) -> Prediction:\n        result = self.pipeline.run(task)\n        self.traces[task.id] = result.trace\n        self.last_trace = result.trace\n        return result.prediction\n', 'src/mythos/solvers/symbolic.py': '"""Verified symbolic solver: symmetry repair, then rigid-transform search.\n\nEvery candidate this solver produces is checked for exact agreement against\n*every* train pair before it is ever applied to a test input -- when this\nsolver returns a prediction, it is either provably correct on the train\ndemonstrations or it doesn\'t fire at all (raises SolverError, letting the\ncaller\'s fallback chain move on to a different solver). This trades recall\nfor precision: it will never guess.\n"""\n\nfrom __future__ import annotations\n\nfrom mythos.arc import ArcTask, Grid, grid_equal\nfrom mythos.object_ops import ALL_OBJECT_TRANSFORM_FINDERS\nfrom mythos.objects import ArcObject\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.solvers.fixture import FixtureSolver\nfrom mythos.submission import Prediction\nfrom mythos.symmetry import (\n    crop,\n    find_occlusion_color_candidates,\n    hole_bbox,\n    hole_cells_for_color,\n    repair_grid,\n)\n\n\nclass SymbolicSolver:\n    """Object/symmetry-aware verified solver; falls back to rigid transforms."""\n\n    def __init__(self) -> None:\n        self.fixture_solver = FixtureSolver()\n\n    def solve(self, task: ArcTask) -> Prediction:\n        prediction = _try_symmetry_repair(task)\n        if prediction is not None:\n            return prediction\n        prediction = _try_object_transforms(task)\n        if prediction is not None:\n            return prediction\n        return self.fixture_solver.solve(task)\n\n\ndef _try_object_transforms(task: ArcTask) -> Prediction | None:\n    for find_transform in ALL_OBJECT_TRANSFORM_FINDERS:\n        transform = find_transform(task)\n        if transform is None:\n            continue\n        try:\n            attempts = [(transform(example.input), transform(example.input)) for example in task.test]\n        except Exception:  # noqa: BLE001 - any failure applying to test just means this candidate doesn\'t fire\n            continue\n        try:\n            return make_prediction(task, attempts)\n        except SolverError:\n            continue\n    return None\n\n\ndef _try_symmetry_repair(task: ArcTask) -> Prediction | None:\n    if any(example.output is None for example in task.train):\n        return None\n\n    # Occlusion marker colors are almost always literally consistent across\n    # a task\'s own examples (it\'s one procedurally-generated task instance),\n    # so try each color that looks like a plausible occlusion block in *any*\n    # train example, as that same literal color everywhere -- more robust\n    # than assuming a stable per-example rank ordering.\n    candidate_colors: set[int] = set()\n    for example in task.train:\n        candidate_colors.update(find_occlusion_color_candidates(example.input))\n\n    for color in candidate_colors:\n        variant = _verify_occlusion_color(task, color)\n        if variant is not None:\n            prediction = _apply_occlusion_plan(task, color, variant)\n            if prediction is not None:\n                return prediction\n    return None\n\n\ndef _verify_occlusion_color(task: ArcTask, color: int) -> str | None:\n    """Return the verified output variant (\'full\' or \'crop\') for this color, or None."""\n\n    variant: str | None = None\n    for example in task.train:\n        hole_cells = hole_cells_for_color(example.input, color)\n        if not hole_cells:\n            return None\n        repaired = repair_grid(example.input, hole_cells)\n        if repaired is None:\n            return None\n\n        example_variant = _matches_variant(repaired, hole_cells, example.output)\n        if example_variant is None:\n            return None\n        if variant is None:\n            variant = example_variant\n        elif variant != example_variant:\n            return None\n    return variant\n\n\ndef _matches_variant(repaired: Grid, hole_cells: set, output: Grid) -> str | None:\n    if grid_equal(repaired, output):\n        return "full"\n    top, left, height, width = hole_bbox(hole_cells)\n    if len(output) == height and len(output[0]) == width and grid_equal(crop(repaired, top, left, height, width), output):\n        return "crop"\n    return None\n\n\ndef _apply_occlusion_plan(task: ArcTask, color: int, variant: str) -> Prediction | None:\n    attempts: list[tuple[Grid, Grid]] = []\n    for example in task.test:\n        hole_cells = hole_cells_for_color(example.input, color)\n        if not hole_cells:\n            return None\n        repaired = repair_grid(example.input, hole_cells)\n        if repaired is None:\n            return None\n        if variant == "full":\n            grid = repaired\n        else:\n            top, left, height, width = hole_bbox(hole_cells)\n            grid = crop(repaired, top, left, height, width)\n        attempts.append((grid, grid))\n    try:\n        return make_prediction(task, attempts)\n    except SolverError:\n        return None\n\n\n__all__ = ["SymbolicSolver", "ArcObject"]\n', 'src/mythos/submission.py': '"""Prediction and submission JSON helpers."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nimport json\nfrom pathlib import Path\nfrom typing import Any, Dict, Iterable, List, Mapping, Tuple\n\nfrom mythos.arc import ArcValidationError, Grid, validate_grid\n\n\n@dataclass(frozen=True)\nclass TestPrediction:\n    __test__ = False\n\n    attempt_1: Grid\n    attempt_2: Grid\n\n\n@dataclass(frozen=True)\nclass Prediction:\n    task_id: str\n    outputs: Tuple[TestPrediction, ...]\n\n\nSubmissionMap = Dict[str, Tuple[TestPrediction, ...]]\n\n\ndef prediction_to_json(prediction: Prediction) -> List[dict[str, Grid]]:\n    return [\n        {"attempt_1": item.attempt_1, "attempt_2": item.attempt_2}\n        for item in prediction.outputs\n    ]\n\n\ndef predictions_to_submission(predictions: Iterable[Prediction]) -> dict[str, List[dict[str, Grid]]]:\n    submission: dict[str, List[dict[str, Grid]]] = {}\n    for prediction in predictions:\n        if prediction.task_id in submission:\n            raise ArcValidationError(f"duplicate prediction for task {prediction.task_id}")\n        submission[prediction.task_id] = prediction_to_json(prediction)\n    if not submission:\n        raise ArcValidationError("submission must contain at least one prediction")\n    return submission\n\n\ndef validate_submission_data(data: Any) -> SubmissionMap:\n    if not isinstance(data, Mapping) or not data:\n        raise ArcValidationError("submission must be a non-empty object keyed by task id")\n    validated: SubmissionMap = {}\n    for task_id, raw_outputs in data.items():\n        if not isinstance(raw_outputs, list) or not raw_outputs:\n            raise ArcValidationError(f"{task_id} must contain a non-empty list of test outputs")\n        outputs: List[TestPrediction] = []\n        for index, raw_output in enumerate(raw_outputs):\n            if not isinstance(raw_output, Mapping):\n                raise ArcValidationError(f"{task_id}[{index}] must be an object")\n            if "attempt_1" not in raw_output or "attempt_2" not in raw_output:\n                raise ArcValidationError(f"{task_id}[{index}] must contain attempt_1 and attempt_2")\n            outputs.append(\n                TestPrediction(\n                    attempt_1=validate_grid(raw_output["attempt_1"], field=f"{task_id}[{index}].attempt_1"),\n                    attempt_2=validate_grid(raw_output["attempt_2"], field=f"{task_id}[{index}].attempt_2"),\n                )\n            )\n        validated[str(task_id)] = tuple(outputs)\n    return validated\n\n\ndef write_submission(predictions: Iterable[Prediction], path: str | Path) -> None:\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    data = predictions_to_submission(predictions)\n    with output_path.open("w", encoding="utf-8") as handle:\n        json.dump(data, handle, indent=2)\n        handle.write("\\n")\n\n\ndef load_submission(path: str | Path) -> SubmissionMap:\n    input_path = Path(path)\n    try:\n        with input_path.open("r", encoding="utf-8") as handle:\n            raw = json.load(handle)\n    except json.JSONDecodeError as exc:\n        raise ArcValidationError(f"{input_path} is not valid JSON: {exc}") from exc\n    return validate_submission_data(raw)\n', 'src/mythos/symmetry.py': '"""Symmetry-repair primitives: recover an occluded region from grid symmetry.\n\nTargets a common ARC-AGI-2 task family (e.g. the diagnosed task 0934a4d8):\na mosaic-like grid has one or more axis-aligned rectangular blocks painted\nover with a single "occlusion" color, and the true content underneath must\nbe recovered from the grid\'s own mirror/rotational/periodic symmetries. The\ntrain examples then either want the full repaired grid back, or just the\nrecovered patch cropped to the occlusion\'s bounding box.\n\nThis is deliberately verification-driven, not heuristic-driven: every\ncandidate (which color is the occlusion, which symmetries hold) is checked\nfor exact consistency against the parts of the grid that are NOT occluded,\nso a wrong guess self-rejects instead of producing a plausible-looking but\nwrong answer.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom typing import Callable\n\nfrom mythos.arc import Grid\nfrom mythos.objects import segment_objects\n\nCell = tuple[int, int]\nSymmetryFn = Callable[[Cell], Cell]\n\n_MIN_OCCLUSION_AREA = 4\n_MAX_OCCLUSION_CANDIDATES = 6\n_MAX_BASE_SYMMETRIES_TO_COMPOSE = 24\n\n\ndef find_occlusion_color_candidates(grid: Grid) -> list[int]:\n    """Colors whose every same-color component is a filled rectangle, ranked by area.\n\n    A real occlusion block is normally a single solid rectangle (or a few of\n    them, all the same fill color); this rejects colors that just happen to\n    be common but scattered (real image content), and trivial single-pixel\n    "rectangles" via _MIN_OCCLUSION_AREA.\n    """\n\n    objects = segment_objects(grid, background=None, connectivity=4, univalued=True)\n    by_color: dict[int, list] = {}\n    for obj in objects:\n        by_color.setdefault(obj.dominant_color, []).append(obj)\n\n    candidates: list[tuple[int, int]] = []\n    for color, objs in by_color.items():\n        total_area = 0\n        max_component_area = 0\n        all_solid_rectangles = True\n        for obj in objs:\n            _, _, height, width = obj.bbox\n            if obj.size != height * width:\n                all_solid_rectangles = False\n                break\n            total_area += obj.size\n            max_component_area = max(max_component_area, obj.size)\n        # Gate on the largest single component, not the summed area: many\n        # scattered same-color single pixels (e.g. a checkerboard) would\n        # otherwise trivially clear a summed floor without being one\n        # genuine occlusion block.\n        if all_solid_rectangles and max_component_area >= _MIN_OCCLUSION_AREA:\n            candidates.append((color, total_area))\n\n    candidates.sort(key=lambda item: item[1], reverse=True)\n    return [color for color, _ in candidates[:_MAX_OCCLUSION_CANDIDATES]]\n\n\ndef _grid_shape(grid: Grid) -> tuple[int, int]:\n    return len(grid), len(grid[0])\n\n\ndef _candidate_symmetries(grid: Grid, hole_cells: set[Cell], min_overlap: int) -> list[SymmetryFn]:\n    """Generate and pre-verify symmetry candidates.\n\n    The symmetry axis is *not* assumed to be the geometric center -- a grid\n    can be a window onto a larger symmetric pattern, so the true axis can\n    sit anywhere. Every axis position is tried and only those consistent\n    with at least `min_overlap` known (non-hole) cell pairs are kept, which\n    both finds off-center axes and rejects coincidental few-cell matches.\n    """\n\n    height, width = _grid_shape(grid)\n\n    row_axes = [\n        axis for axis in range(2 * height - 1)\n        if verify_symmetry(grid, hole_cells, _row_mirror(axis), min_overlap=min_overlap)\n    ]\n    col_axes = [\n        axis for axis in range(2 * width - 1)\n        if verify_symmetry(grid, hole_cells, _col_mirror(axis), min_overlap=min_overlap)\n    ]\n\n    symmetries: list[SymmetryFn] = [_row_mirror(axis) for axis in row_axes]\n    symmetries.extend(_col_mirror(axis) for axis in col_axes)\n    symmetries.extend(_rotate_180(row_axis, col_axis) for row_axis in row_axes for col_axis in col_axes)\n\n    if height == width:\n        symmetries.append(lambda cell: (cell[1], cell[0]))  # transpose\n        symmetries.append(lambda cell: (width - 1 - cell[1], height - 1 - cell[0]))  # anti-transpose\n\n    for period in range(1, width):\n        symmetries.append(_make_periodic(0, period, height, width))\n        symmetries.append(_make_periodic(0, -period, height, width))\n    for period in range(1, height):\n        symmetries.append(_make_periodic(period, 0, height, width))\n        symmetries.append(_make_periodic(-period, 0, height, width))\n\n    verified = [fn for fn in symmetries if verify_symmetry(grid, hole_cells, fn, min_overlap=min_overlap)]\n\n    # Closure under composition: if f and g are each independently verified\n    # symmetries of this grid, g(f(x)) is transitively guaranteed consistent\n    # wherever both hold -- not a heuristic, just equality chaining. This is\n    # what discovers compound symmetries (glide reflections, mirror axes\n    # shifted by a verified translation period) without hand-listing every\n    # possible symmetry family up front. Skipped when the base list is\n    # already large (a highly-symmetric/background-heavy grid), since that\n    # already gives plenty of coverage and O(n^2) composition would be the\n    # dominant cost for no real benefit.\n    if 0 < len(verified) <= _MAX_BASE_SYMMETRIES_TO_COMPOSE:\n        composed = [_compose(outer, inner) for inner in verified for outer in verified]\n        verified.extend(fn for fn in composed if verify_symmetry(grid, hole_cells, fn, min_overlap=min_overlap))\n\n    return verified\n\n\ndef _compose(outer: SymmetryFn, inner: SymmetryFn) -> SymmetryFn:\n    return lambda cell: outer(inner(cell))\n\n\ndef _row_mirror(axis_sum: int) -> SymmetryFn:\n    return lambda cell: (axis_sum - cell[0], cell[1])\n\n\ndef _col_mirror(axis_sum: int) -> SymmetryFn:\n    return lambda cell: (cell[0], axis_sum - cell[1])\n\n\ndef _rotate_180(row_axis: int, col_axis: int) -> SymmetryFn:\n    return lambda cell: (row_axis - cell[0], col_axis - cell[1])\n\n\ndef _make_periodic(dr: int, dc: int, height: int, width: int) -> SymmetryFn:\n    def shift(cell: Cell) -> Cell:\n        return (cell[0] + dr, cell[1] + dc)\n\n    return shift\n\n\ndef _in_bounds(cell: Cell, height: int, width: int) -> bool:\n    r, c = cell\n    return 0 <= r < height and 0 <= c < width\n\n\ndef verify_symmetry(grid: Grid, hole_cells: set[Cell], symmetry_fn: SymmetryFn, *, min_overlap: int = 1) -> bool:\n    height, width = _grid_shape(grid)\n    checked = 0\n    for r in range(height):\n        for c in range(width):\n            if (r, c) in hole_cells:\n                continue\n            mapped = symmetry_fn((r, c))\n            if not _in_bounds(mapped, height, width) or mapped in hole_cells:\n                continue\n            checked += 1\n            if grid[r][c] != grid[mapped[0]][mapped[1]]:\n                return False\n    return checked >= min_overlap\n\n\ndef _default_min_overlap(height: int, width: int) -> int:\n    return max(8, (height * width) // 10)\n\n\ndef repair_grid(grid: Grid, hole_cells: set[Cell]) -> Grid | None:\n    """Fill hole_cells using every symmetry that verifies against the rest of the grid.\n\n    Iterates to a fixed point since one symmetry\'s target may itself be an\n    unresolved hole cell that a *different* symmetry (or the same one, after\n    another cell resolves) can fill.\n    """\n\n    if not hole_cells:\n        return [row[:] for row in grid]\n\n    height, width = _grid_shape(grid)\n    min_overlap = _default_min_overlap(height, width)\n    verified = _candidate_symmetries(grid, hole_cells, min_overlap)\n    if not verified:\n        return None\n\n    repaired = [row[:] for row in grid]\n    remaining = set(hole_cells)\n    progressed = True\n    while remaining and progressed:\n        progressed = False\n        for cell in list(remaining):\n            for fn in verified:\n                mapped = fn(cell)\n                if not _in_bounds(mapped, height, width) or mapped in remaining:\n                    continue\n                repaired[cell[0]][cell[1]] = repaired[mapped[0]][mapped[1]]\n                remaining.discard(cell)\n                progressed = True\n                break\n\n    return None if remaining else repaired\n\n\ndef hole_bbox(hole_cells: set[Cell]) -> tuple[int, int, int, int]:\n    rows = [r for r, _ in hole_cells]\n    cols = [c for _, c in hole_cells]\n    top, bottom = min(rows), max(rows)\n    left, right = min(cols), max(cols)\n    return top, left, bottom - top + 1, right - left + 1\n\n\ndef crop(grid: Grid, top: int, left: int, height: int, width: int) -> Grid:\n    return [row[left : left + width] for row in grid[top : top + height]]\n\n\ndef hole_cells_for_color(grid: Grid, color: int) -> set[Cell]:\n    return {(r, c) for r, row in enumerate(grid) for c, cell in enumerate(row) if cell == color}\n', 'src/mythos/text_reasoning.py': '"""Optional HRM-Text rule generation for ARC tasks."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom typing import Sequence\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.features import DEFAULT_RULE_DIM\n\n\nclass HRMTextError(RuntimeError):\n    """Raised when optional HRM-Text inference cannot run."""\n\n\n@dataclass(frozen=True)\nclass HRMTextRuleResult:\n    description: str\n    vector: tuple[float, ...]\n    model_root: str\n\n\ndef generate_hrm_text_rule(\n    task: ArcTask,\n    model_path: str | Path,\n    *,\n    max_new_tokens: int = 96,\n    device: str | None = None,\n    vector_dim: int = DEFAULT_RULE_DIM,\n) -> HRMTextRuleResult:\n    """Run a Hugging Face text model to produce a rule description/vector."""\n\n    try:\n        import torch\n        from transformers import AutoModelForCausalLM, AutoTokenizer\n    except Exception as exc:  # pragma: no cover - optional dependency.\n        raise HRMTextError("transformers and torch are required for HRM-Text inference") from exc\n\n    root = _model_root(model_path)\n    selected_device = device or ("cuda" if torch.cuda.is_available() else "cpu")\n    try:\n        tokenizer = AutoTokenizer.from_pretrained(root, trust_remote_code=True)\n        model = AutoModelForCausalLM.from_pretrained(\n            root,\n            trust_remote_code=True,\n            torch_dtype=torch.float16 if selected_device == "cuda" else torch.float32,\n        ).to(selected_device)\n    except Exception as exc:  # pragma: no cover - depends on external model files.\n        raise HRMTextError(f"failed to load HRM-Text model from {root}: {exc}") from exc\n\n    if tokenizer.pad_token_id is None and tokenizer.eos_token_id is not None:\n        tokenizer.pad_token = tokenizer.eos_token\n\n    prompt = format_arc_rule_prompt(task)\n    encoded = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=4096).to(selected_device)\n    try:\n        with torch.no_grad():\n            generated = model.generate(\n                **encoded,\n                do_sample=False,\n                max_new_tokens=max_new_tokens,\n                pad_token_id=tokenizer.pad_token_id,\n            )\n            forward = model(\n                generated,\n                output_hidden_states=True,\n                use_cache=False,\n            )\n    except Exception as exc:  # pragma: no cover - depends on external model behavior.\n        raise HRMTextError(f"HRM-Text forward/generation failed: {exc}") from exc\n\n    generated_text = tokenizer.decode(generated[0], skip_special_tokens=True)\n    description = _extract_answer(prompt, generated_text)\n    vector = _hidden_state_to_vector(forward.hidden_states[-1][0], vector_dim)\n    return HRMTextRuleResult(description=description, vector=vector, model_root=str(root))\n\n\ndef format_arc_rule_prompt(task: ArcTask) -> str:\n    examples = []\n    for index, example in enumerate(task.train):\n        examples.append(\n            f"Train {index} input:\\n{_grid_text(example.input)}\\n"\n            f"Train {index} output:\\n{_grid_text(example.output or example.input)}"\n        )\n    tests = []\n    for index, example in enumerate(task.test):\n        tests.append(f"Test {index} input:\\n{_grid_text(example.input)}")\n    return (\n        "You are solving an ARC abstract reasoning task. Infer the rule from "\n        "the train pairs and describe the transformation in one concise sentence.\\n\\n"\n        + "\\n\\n".join(examples)\n        + "\\n\\n"\n        + "\\n\\n".join(tests)\n        + "\\n\\nRule:"\n    )\n\n\ndef _model_root(path: str | Path) -> Path:\n    candidate = Path(path)\n    return candidate.parent if candidate.is_file() else candidate\n\n\ndef _grid_text(grid: Grid) -> str:\n    return "\\n".join(" ".join(str(cell) for cell in row) for row in grid)\n\n\ndef _extract_answer(prompt: str, generated_text: str) -> str:\n    if generated_text.startswith(prompt):\n        generated_text = generated_text[len(prompt) :]\n    answer = generated_text.strip()\n    return answer or "No HRM-Text rule text generated."\n\n\ndef _hidden_state_to_vector(hidden_state, dim: int) -> tuple[float, ...]:  # type: ignore[no-untyped-def]\n    if dim <= 0:\n        raise ValueError("vector_dim must be positive")\n    pooled = hidden_state.float().mean(dim=0).detach().cpu()\n    if pooled.numel() < dim:\n        values = pooled.tolist() + [0.0 for _ in range(dim - pooled.numel())]\n    else:\n        chunk_size = max(1, pooled.numel() // dim)\n        values = []\n        for index in range(dim):\n            start = index * chunk_size\n            end = pooled.numel() if index == dim - 1 else min(pooled.numel(), start + chunk_size)\n            values.append(float(pooled[start:end].mean()))\n    return _normalize(values[:dim])\n\n\ndef _normalize(values: Sequence[float]) -> tuple[float, ...]:\n    norm = sum(value * value for value in values) ** 0.5\n    if norm == 0.0:\n        return tuple(round(value, 6) for value in values)\n    return tuple(round(value / norm, 6) for value in values)\n', 'src/mythos/train_projection.py': '"""CLI for training the JEPA-to-HRM projection checkpoint."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.arc import ArcValidationError, attach_solutions, load_challenges, load_solutions\nfrom mythos.training import JepaProjectionConfig, train_jepa_projection\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Train the Mythos I-JEPA projection checkpoint.")\n    parser.add_argument("--challenges", required=True)\n    parser.add_argument("--solutions")\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--steps", type=int, default=200)\n    parser.add_argument("--lr", type=float, default=1e-3)\n    parser.add_argument("--input-dim", type=int, default=1280)\n    parser.add_argument("--output-dim", type=int, default=768)\n    parser.add_argument("--device")\n    parser.add_argument("--include-test-solutions", action="store_true")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.challenges)\n        if args.solutions:\n            tasks = attach_solutions(tasks, load_solutions(args.solutions))\n        result = train_jepa_projection(\n            tasks.values(),\n            checkpoint_path=args.out,\n            config=JepaProjectionConfig(input_dim=args.input_dim, output_dim=args.output_dim),\n            steps=args.steps,\n            lr=args.lr,\n            device=args.device,\n            include_test_solutions=args.include_test_solutions,\n        )\n    except (ArcValidationError, RuntimeError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/train_world_model.py': '"""CLI for training the Mythos world-model transition checkpoint."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.arc import ArcValidationError, attach_solutions, load_challenges, load_solutions\nfrom mythos.training import WorldModelConfig, train_world_model\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Train the Mythos world-model checkpoint.")\n    parser.add_argument("--challenges", required=True)\n    parser.add_argument("--solutions")\n    parser.add_argument("--out", required=True)\n    parser.add_argument("--steps", type=int, default=300)\n    parser.add_argument("--lr", type=float, default=1e-3)\n    parser.add_argument("--z-dim", type=int, default=768)\n    parser.add_argument("--rule-dim", type=int, default=4)\n    parser.add_argument("--hidden-dim", type=int, default=3072)\n    parser.add_argument("--device")\n    parser.add_argument("--include-test-solutions", action="store_true")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.challenges)\n        if args.solutions:\n            tasks = attach_solutions(tasks, load_solutions(args.solutions))\n        result = train_world_model(\n            tasks.values(),\n            checkpoint_path=args.out,\n            config=WorldModelConfig(\n                z_dim=args.z_dim,\n                rule_dim=args.rule_dim,\n                hidden_dim=args.hidden_dim,\n            ),\n            steps=args.steps,\n            lr=args.lr,\n            device=args.device,\n            include_test_solutions=args.include_test_solutions,\n        )\n    except (ArcValidationError, RuntimeError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/training.py': '"""Training helpers for Project Mythos planned stages."""\n\nfrom __future__ import annotations\n\nfrom dataclasses import asdict, dataclass\nfrom pathlib import Path\nimport time\nfrom typing import Iterable\n\nfrom mythos.arc import ArcTask\nfrom mythos.features import (\n    DEFAULT_HRM_FEATURE_DIM,\n    DEFAULT_JEPA_FEATURE_DIM,\n    DEFAULT_RULE_DIM,\n    grid_to_feature_vector,\n    iter_all_supervised_grid_pairs,\n    iter_train_grid_pairs,\n    max_pairwise_cosine,\n    task_rule_vector,\n)\nfrom mythos.lora import (\n    changed_frozen_parameters,\n    inject_lora_adapters,\n    lora_parameters,\n    save_lora_checkpoint,\n    snapshot_frozen_parameters,\n)\n\n\ntry:  # Keep imports cheap for CLI validation paths that do not train.\n    from torch import nn as _OPTIONAL_NN\nexcept Exception:  # pragma: no cover - depends on optional torch install.\n    _OPTIONAL_NN = None\n\n\ndef _torch():\n    try:\n        import torch\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for Mythos training helpers") from exc\n    return torch\n\n\ndef _nn():\n    try:\n        from torch import nn\n    except Exception as exc:  # pragma: no cover - depends on optional torch install.\n        raise RuntimeError("PyTorch is required for Mythos training helpers") from exc\n    return nn\n\n\n@dataclass(frozen=True)\nclass JepaProjectionConfig:\n    input_dim: int = DEFAULT_JEPA_FEATURE_DIM\n    output_dim: int = DEFAULT_HRM_FEATURE_DIM\n    seed: int = 7\n\n\n@dataclass(frozen=True)\nclass WorldModelConfig:\n    z_dim: int = DEFAULT_HRM_FEATURE_DIM\n    rule_dim: int = DEFAULT_RULE_DIM\n    hidden_dim: int = 3072\n    seed: int = 11\n\n\n@dataclass(frozen=True)\nclass TrainingResult:\n    stage: str\n    checkpoint_path: str | None\n    steps: int\n    examples: int\n    initial_loss: float\n    final_loss: float\n    elapsed_seconds: float\n    extra: dict[str, object]\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "stage": self.stage,\n            "checkpoint_path": self.checkpoint_path,\n            "steps": self.steps,\n            "examples": self.examples,\n            "initial_loss": self.initial_loss,\n            "final_loss": self.final_loss,\n            "elapsed_seconds": self.elapsed_seconds,\n            "extra": self.extra,\n        }\n\n\n@dataclass(frozen=True)\nclass TTTSmokeResult:\n    steps: int\n    rank: int\n    injected_modules: tuple[str, ...]\n    initial_loss: float\n    final_loss: float\n    first_backward_seconds: float\n    frozen_parameter_changes: tuple[str, ...]\n    checkpoint_path: str | None\n\n    @property\n    def backbone_frozen(self) -> bool:\n        return not self.frozen_parameter_changes\n\n    def to_dict(self) -> dict[str, object]:\n        return {\n            "steps": self.steps,\n            "rank": self.rank,\n            "injected_modules": list(self.injected_modules),\n            "initial_loss": self.initial_loss,\n            "final_loss": self.final_loss,\n            "first_backward_seconds": self.first_backward_seconds,\n            "frozen_parameter_changes": list(self.frozen_parameter_changes),\n            "backbone_frozen": self.backbone_frozen,\n            "checkpoint_path": self.checkpoint_path,\n        }\n\n\n_BASE_MODULE = _OPTIONAL_NN.Module if _OPTIONAL_NN is not None else object\n\n\nclass JepaProjection(_BASE_MODULE):\n    """Projection layer from I-JEPA/ARC features into the HRM feature width."""\n\n    def __init__(self, config: JepaProjectionConfig) -> None:\n        nn = _nn()\n        super().__init__()\n        self.config = config\n        self.norm = nn.LayerNorm(config.input_dim, elementwise_affine=False)\n        self.proj = nn.Linear(config.input_dim, config.output_dim)\n\n    def forward(self, inputs):  # type: ignore[no-untyped-def]\n        return self.proj(self.norm(inputs))\n\n\nclass WorldModelMLP(_BASE_MODULE):\n    """Two-layer transition model f(z_input, v_rule) -> z_output."""\n\n    def __init__(self, config: WorldModelConfig) -> None:\n        nn = _nn()\n        super().__init__()\n        self.config = config\n        self.net = nn.Sequential(\n            nn.Linear(config.z_dim + config.rule_dim, config.hidden_dim),\n            nn.GELU(),\n            nn.Linear(config.hidden_dim, config.z_dim),\n        )\n\n    def forward(self, z_input, rule):  # type: ignore[no-untyped-def]\n        torch = _torch()\n        return self.net(torch.cat([z_input, rule], dim=-1))\n\n\ndef train_jepa_projection(\n    tasks: Iterable[ArcTask],\n    *,\n    checkpoint_path: str | Path | None = None,\n    config: JepaProjectionConfig | None = None,\n    steps: int = 200,\n    lr: float = 1e-3,\n    device: str | None = None,\n    include_test_solutions: bool = False,\n) -> TrainingResult:\n    """Train only the ARC-to-HRM projection checkpoint."""\n\n    torch = _torch()\n    nn = _nn()\n    cfg = config or JepaProjectionConfig()\n    torch.manual_seed(cfg.seed)\n    started = time.perf_counter()\n\n    pairs = list(\n        iter_all_supervised_grid_pairs(tasks) if include_test_solutions else iter_train_grid_pairs(tasks)\n    )\n    if not pairs:\n        raise ValueError("no supervised ARC grid pairs available for projection training")\n\n    selected_device = _select_device(device)\n    x = torch.tensor(\n        [grid_to_feature_vector(pair.input, cfg.input_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n    y = torch.tensor(\n        [grid_to_feature_vector(pair.output, cfg.output_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n\n    model = JepaProjection(cfg).to(selected_device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n    loss_fn = nn.MSELoss()\n\n    with torch.no_grad():\n        initial_loss = float(loss_fn(model(x), y).detach().cpu())\n    for _ in range(max(0, steps)):\n        optimizer.zero_grad(set_to_none=True)\n        loss = loss_fn(model(x), y)\n        loss.backward()\n        optimizer.step()\n    with torch.no_grad():\n        final_loss = float(loss_fn(model(x), y).detach().cpu())\n\n    output = None\n    if checkpoint_path is not None:\n        output = save_projection_checkpoint(model, checkpoint_path)\n\n    diversity = max_pairwise_cosine(\n        [grid_to_feature_vector(pair.input, cfg.input_dim) for pair in pairs[: min(10, len(pairs))]]\n    )\n    return TrainingResult(\n        stage="jepa_projection",\n        checkpoint_path=str(output) if output is not None else None,\n        steps=steps,\n        examples=len(pairs),\n        initial_loss=round(initial_loss, 8),\n        final_loss=round(final_loss, 8),\n        elapsed_seconds=round(time.perf_counter() - started, 3),\n        extra={\n            "config": asdict(cfg),\n            "device": selected_device,\n            "max_pairwise_input_cosine": round(diversity, 6),\n            "trainable_parameters": sum(parameter.numel() for parameter in model.parameters()),\n        },\n    )\n\n\ndef save_projection_checkpoint(model: JepaProjection, path: str | Path) -> Path:\n    torch = _torch()\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "kind": "mythos_jepa_projection",\n            "config": asdict(model.config),\n            "state_dict": model.state_dict(),\n        },\n        output_path,\n    )\n    return output_path\n\n\ndef load_projection_checkpoint(path: str | Path, *, device: str | None = None) -> JepaProjection:\n    torch = _torch()\n    selected_device = _select_device(device)\n    raw = torch.load(path, map_location=selected_device, weights_only=False)\n    config = JepaProjectionConfig(**raw["config"])\n    model = JepaProjection(config).to(selected_device)\n    model.load_state_dict(raw["state_dict"])\n    model.eval()\n    return model\n\n\ndef train_world_model(\n    tasks: Iterable[ArcTask],\n    *,\n    checkpoint_path: str | Path | None = None,\n    config: WorldModelConfig | None = None,\n    steps: int = 300,\n    lr: float = 1e-3,\n    device: str | None = None,\n    include_test_solutions: bool = False,\n) -> TrainingResult:\n    """Train the two-layer transition world model from ARC before/after pairs."""\n\n    torch = _torch()\n    nn = _nn()\n    cfg = config or WorldModelConfig()\n    torch.manual_seed(cfg.seed)\n    started = time.perf_counter()\n\n    task_list = list(tasks)\n    pairs = list(\n        iter_all_supervised_grid_pairs(task_list) if include_test_solutions else iter_train_grid_pairs(task_list)\n    )\n    task_by_id = {task.id: task for task in task_list}\n    if not pairs:\n        raise ValueError("no supervised ARC grid pairs available for world-model training")\n\n    selected_device = _select_device(device)\n    z_input = torch.tensor(\n        [grid_to_feature_vector(pair.input, cfg.z_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n    rules = torch.tensor(\n        [task_rule_vector(task_by_id[pair.task_id], cfg.rule_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n    z_target = torch.tensor(\n        [grid_to_feature_vector(pair.output, cfg.z_dim) for pair in pairs],\n        dtype=torch.float32,\n        device=selected_device,\n    )\n\n    model = WorldModelMLP(cfg).to(selected_device)\n    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)\n    loss_fn = nn.MSELoss()\n\n    with torch.no_grad():\n        initial_loss = float(loss_fn(model(z_input, rules), z_target).detach().cpu())\n    for _ in range(max(0, steps)):\n        optimizer.zero_grad(set_to_none=True)\n        loss = loss_fn(model(z_input, rules), z_target)\n        loss.backward()\n        optimizer.step()\n    with torch.no_grad():\n        final_loss = float(loss_fn(model(z_input, rules), z_target).detach().cpu())\n\n    output = None\n    if checkpoint_path is not None:\n        output = save_world_model_checkpoint(model, checkpoint_path)\n\n    return TrainingResult(\n        stage="world_model",\n        checkpoint_path=str(output) if output is not None else None,\n        steps=steps,\n        examples=len(pairs),\n        initial_loss=round(initial_loss, 8),\n        final_loss=round(final_loss, 8),\n        elapsed_seconds=round(time.perf_counter() - started, 3),\n        extra={\n            "config": asdict(cfg),\n            "device": selected_device,\n            "trainable_parameters": sum(parameter.numel() for parameter in model.parameters()),\n        },\n    )\n\n\ndef save_world_model_checkpoint(model: WorldModelMLP, path: str | Path) -> Path:\n    torch = _torch()\n    output_path = Path(path)\n    output_path.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(\n        {\n            "kind": "mythos_world_model",\n            "config": asdict(model.config),\n            "state_dict": model.state_dict(),\n        },\n        output_path,\n    )\n    return output_path\n\n\ndef load_world_model_checkpoint(path: str | Path, *, device: str | None = None) -> WorldModelMLP:\n    torch = _torch()\n    selected_device = _select_device(device)\n    raw = torch.load(path, map_location=selected_device, weights_only=False)\n    config = WorldModelConfig(**raw["config"])\n    model = WorldModelMLP(config).to(selected_device)\n    model.load_state_dict(raw["state_dict"])\n    model.eval()\n    return model\n\n\ndef run_ttt_lora_smoke(\n    *,\n    rank: int = 16,\n    steps: int = 50,\n    dim: int = 32,\n    batch_size: int = 8,\n    lr: float = 1e-2,\n    device: str | None = None,\n    checkpoint_path: str | Path | None = None,\n) -> TTTSmokeResult:\n    """Run a small LoRA-only optimization to validate the TTT mechanics."""\n\n    torch = _torch()\n    nn = _nn()\n    selected_device = _select_device(device)\n    torch.manual_seed(23)\n\n    class TinyAttentionModel(nn.Module):\n        def __init__(self) -> None:\n            super().__init__()\n            self.attention_q = nn.Linear(dim, dim)\n            self.attention_out = nn.Linear(dim, dim)\n            self.mlp = nn.Sequential(nn.GELU(), nn.Linear(dim, dim))\n\n        def forward(self, x):  # type: ignore[no-untyped-def]\n            return self.mlp(self.attention_out(torch.tanh(self.attention_q(x))))\n\n    model = TinyAttentionModel().to(selected_device)\n    report = inject_lora_adapters(\n        model,\n        rank=rank,\n        target_patterns=("attention",),\n        fallback_to_all_linear=False,\n        freeze_backbone=True,\n    )\n    snapshot = snapshot_frozen_parameters(model)\n    optimizer = torch.optim.AdamW(lora_parameters(model), lr=lr)\n    loss_fn = nn.MSELoss()\n    inputs = torch.randn(batch_size, dim, device=selected_device)\n    targets = torch.flip(inputs, dims=(-1,))\n\n    with torch.no_grad():\n        initial_loss = float(loss_fn(model(inputs), targets).detach().cpu())\n\n    first_backward_seconds = 0.0\n    for step_index in range(max(0, steps)):\n        optimizer.zero_grad(set_to_none=True)\n        outputs = model(inputs)\n        loss = loss_fn(outputs, targets)\n        started = time.perf_counter()\n        loss.backward()\n        if step_index == 0:\n            first_backward_seconds = time.perf_counter() - started\n        optimizer.step()\n\n    with torch.no_grad():\n        final_loss = float(loss_fn(model(inputs), targets).detach().cpu())\n    changed = changed_frozen_parameters(model, snapshot)\n\n    output = None\n    if checkpoint_path is not None:\n        output = save_lora_checkpoint(\n            model,\n            checkpoint_path,\n            metadata={\n                "rank": rank,\n                "steps": steps,\n                "dim": dim,\n                "batch_size": batch_size,\n                "smoke": True,\n            },\n        )\n\n    return TTTSmokeResult(\n        steps=steps,\n        rank=rank,\n        injected_modules=report.injected_modules,\n        initial_loss=round(initial_loss, 8),\n        final_loss=round(final_loss, 8),\n        first_backward_seconds=round(first_backward_seconds, 6),\n        frozen_parameter_changes=changed,\n        checkpoint_path=str(output) if output is not None else None,\n    )\n\n\ndef adaptive_ttt_should_stop(\n    losses: Iterable[float],\n    *,\n    min_delta: float = 1e-4,\n    patience: int = 5,\n) -> bool:\n    """Return True when recent TTT loss improvements have flattened."""\n\n    collected = list(losses)\n    if len(collected) <= patience:\n        return False\n    recent = collected[-(patience + 1) :]\n    improvements = [recent[index] - recent[index + 1] for index in range(len(recent) - 1)]\n    return all(improvement < min_delta for improvement in improvements)\n\n\ndef _select_device(device: str | None) -> str:\n    if device:\n        return device\n    torch = _torch()\n    return "cuda" if torch.cuda.is_available() else "cpu"\n', 'src/mythos/ttt_smoke.py': '"""CLI for validating LoRA-only test-time-training mechanics."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport json\nimport sys\n\nfrom mythos.training import run_ttt_lora_smoke\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Run a Mythos LoRA/TTT smoke test.")\n    parser.add_argument("--out")\n    parser.add_argument("--rank", type=int, default=16)\n    parser.add_argument("--steps", type=int, default=50)\n    parser.add_argument("--dim", type=int, default=32)\n    parser.add_argument("--batch-size", type=int, default=8)\n    parser.add_argument("--lr", type=float, default=1e-2)\n    parser.add_argument("--device")\n    args = parser.parse_args(argv)\n\n    try:\n        result = run_ttt_lora_smoke(\n            rank=args.rank,\n            steps=args.steps,\n            dim=args.dim,\n            batch_size=args.batch_size,\n            lr=args.lr,\n            device=args.device,\n            checkpoint_path=args.out,\n        )\n    except (RuntimeError, ValueError) as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n\n    print(json.dumps(result.to_dict(), indent=2, sort_keys=True))\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'src/mythos/validate.py': '"""CLI for ARC challenge validation."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport sys\n\nfrom mythos.arc import ArcValidationError, load_challenges\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description="Validate an ARC challenges.json file.")\n    parser.add_argument("path", help="Path to ARC-style challenges JSON.")\n    args = parser.parse_args(argv)\n\n    try:\n        tasks = load_challenges(args.path)\n    except ArcValidationError as exc:\n        print(f"ERROR: {exc}", file=sys.stderr)\n        return 2\n    train_count = sum(len(task.train) for task in tasks.values())\n    test_count = sum(len(task.test) for task in tasks.values())\n    print(f"OK: {len(tasks)} tasks, {train_count} train examples, {test_count} test items")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n', 'third_party/compress_arc/arc_compressor.py': 'import numpy as np\nimport torch\n\nimport initializers\nimport layers\n\n\nnp.random.seed(0)\ntorch.manual_seed(0)\ntorch.set_default_dtype(torch.float32)\ntorch.set_default_device(\'cuda\')\n\n\nclass ARCCompressor:\n    """\n    The main model class for the VAE Decoder in our solution to ARC.\n    """\n\n    # Define the channel dimensions that all the layers use\n    n_layers = 4\n    share_up_dim = 16\n    share_down_dim = 8\n    decoding_dim = 4\n    softmax_dim = 2\n    cummax_dim = 4\n    shift_dim = 4\n    nonlinear_dim = 16\n\n    # This function gives the channel dimension of the residual stream depending on\n    # which dimensions are present, for every tensor in the multitensor.\n    def channel_dim_fn(self, dims):\n        return 16 if dims[2] == 0 else 8\n\n    def __init__(self, task):\n        """\n        Create a model that is tailored to the given task, and initialize all the weights.\n        The weights are symmetrized such that swapping the x and y dimension ordering should\n        make the output\'s dimension ordering also swapped, for the same weights. This may not\n        be exactly correct since symmetrizing all operations is difficult.\n        Args:\n            task (preprocessing.Task): The task which the model is to be made for solving.\n        """\n        self.multitensor_system = task.multitensor_system\n\n        # Initialize weights\n        initializer = initializers.Initializer(self.multitensor_system, self.channel_dim_fn)\n\n        self.multiposteriors = initializer.initialize_multiposterior(self.decoding_dim)\n        self.decode_weights = initializer.initialize_multilinear([self.decoding_dim, self.channel_dim_fn])\n        initializer.symmetrize_xy(self.decode_weights)\n        self.target_capacities = initializer.initialize_multizeros([self.decoding_dim])\n\n        self.share_up_weights = []\n        self.share_down_weights = []\n        self.softmax_weights = []\n        self.cummax_weights = []\n        self.shift_weights = []\n        self.direction_share_weights = []\n        self.nonlinear_weights = []\n\n        for layer_num in range(self.n_layers):\n            self.share_up_weights.append(initializer.initialize_multiresidual(self.share_up_dim, self.share_up_dim))\n            self.share_down_weights.append(initializer.initialize_multiresidual(self.share_down_dim, self.share_down_dim))\n            output_scaling_fn = lambda dims: self.softmax_dim * (2 ** (dims[1] + dims[2] + dims[3] + dims[4]) - 1)\n            self.softmax_weights.append(initializer.initialize_multiresidual(self.softmax_dim, output_scaling_fn))\n            self.cummax_weights.append(initializer.initialize_multiresidual(self.cummax_dim, self.cummax_dim))\n            self.shift_weights.append(initializer.initialize_multiresidual(self.shift_dim, self.shift_dim))\n            self.direction_share_weights.append(initializer.initialize_multidirection_share())\n            self.nonlinear_weights.append(initializer.initialize_multiresidual(self.nonlinear_dim, self.nonlinear_dim))\n\n        self.head_weights = initializer.initialize_head()\n        self.mask_weights = initializer.initialize_linear(\n            [1, 0, 0, 1, 0], [self.channel_dim_fn([1, 0, 0, 1, 0]), 2]\n        )\n\n        # Symmetrize weights so that their behavior is equivariant to swapping x and y dimension ordering\n        for weight_list in [\n            self.share_up_weights,\n            self.share_down_weights,\n            self.softmax_weights,\n            self.cummax_weights,\n            self.shift_weights,\n            self.nonlinear_weights,\n        ]:\n            for layer_num in range(self.n_layers):\n                initializer.symmetrize_xy(weight_list[layer_num])\n\n        for layer_num in range(self.n_layers):\n            initializer.symmetrize_direction_sharing(self.direction_share_weights[layer_num])\n\n        self.weights_list = initializer.weights_list\n\n\n    def forward(self):\n        """\n        Compute the forward pass of the VAE decoder. Start by using internally stored latents,\n        and process from there. Output an [example, color, x, y, channel] tensor for the colors,\n        and an [example, x, channel] and [example, y, channel] tensor for the masks.\n        Returns:\n            Tensor: An [example, color, x, y, channel] tensor, where for every example,\n                    input/output (picked by channel dimension), and every pixel (picked\n                    by x and y dimensions), we have a vector full of logits for that\n                    pixel being each possible color.\n            Tensor: An [example, x, channel] tensor, where for every example, input/output\n                    (picked by channel dimension), and every x, we assign a score that\n                    contributes to the likelihood that that index of the x dimension is not\n                    masked out in the prediction.\n            Tensor: An [example, y, channel] tensor, used in the same way as above.\n            list[Tensor]: A list of tensors indicating the amount of KL contributed by each component\n                    tensor in the layers.decode_latents() step.\n            list[str]: A list of tensor names that correspond to each tensor in the aforementioned output.\n        """\n        # Decoding layer\n        x, KL_amounts, KL_names = layers.decode_latents(\n            self.target_capacities, self.decode_weights, self.multiposteriors\n        )\n\n        for layer_num in range(self.n_layers):\n            # Multitensor communication layer\n            x = layers.share_up(x, self.share_up_weights[layer_num])\n\n            # Softmax layer\n            x = layers.softmax(x, self.softmax_weights[layer_num], pre_norm=True, post_norm=False, use_bias=False)\n\n            # Directional layers\n            x = layers.cummax(\n                x, self.cummax_weights[layer_num], self.multitensor_system.task.masks,\n                pre_norm=False, post_norm=True, use_bias=False\n            )\n            x = layers.shift(\n                x, self.shift_weights[layer_num], self.multitensor_system.task.masks,\n                pre_norm=False, post_norm=True, use_bias=False\n            )\n\n            # Directional communication layer\n            x = layers.direction_share(x, self.direction_share_weights[layer_num], pre_norm=True, use_bias=False)\n\n            # Nonlinear layer\n            x = layers.nonlinear(x, self.nonlinear_weights[layer_num], pre_norm=True, post_norm=False, use_bias=False)\n\n            # Multitensor communication layer\n            x = layers.share_down(x, self.share_down_weights[layer_num])\n\n            # Normalization layer\n            x = layers.normalize(x)\n\n        # Linear Heads\n        output = (\n            layers.affine(x[[1, 1, 0, 1, 1]], self.head_weights, use_bias=False)\n            + 100 * self.head_weights[1]\n        )\n        x_mask = layers.affine(x[[1, 0, 0, 1, 0]], self.mask_weights, use_bias=True)\n        y_mask = layers.affine(x[[1, 0, 0, 0, 1]], self.mask_weights, use_bias=True)\n\n        # Postprocessing\n        x_mask, y_mask = layers.postprocess_mask(self.multitensor_system.task, x_mask, y_mask)\n\n        return output, x_mask, y_mask, KL_amounts, KL_names\n\n', 'third_party/compress_arc/initializers.py': 'import numpy as np\nimport torch\nimport multitensor_systems\n\n\nnp.random.seed(0)\ntorch.manual_seed(0)\n\n\nclass Initializer:\n    def __init__(self, multitensor_system, channel_dim_fn):\n        """\n        Initializes weight tensors for a multitensor system.\n        Args:\n            multitensor_system (MultiTensorSystem): The multitensor system that we want to use\n                    for initializing weights.\n            channel_dim_fn (function): A function that takes in a dims list of type list[int], and\n                    returns an int representing the channel dimension size.\n        """\n        self.multitensor_system = multitensor_system\n        self.channel_dim_fn = channel_dim_fn\n        self.weights_list = []\n\n    def initialize_zeros(self, dims, shape):\n        """Initializes a weight tensor with zeros."""\n        if callable(shape):\n            shape = shape(dims)\n        zeros = torch.zeros(shape, requires_grad=True)\n        self.weights_list.append(zeros)\n        return zeros\n\n    def initialize_linear(self, dims, shape):\n        """Initializes a linear transformation."""\n        if callable(shape):\n            shape = shape(dims)\n        n_in, n_out = shape\n\n        if callable(n_in):\n            n_in = n_in(dims)\n        if callable(n_out):\n            n_out = n_out(dims)\n\n        scale = 1 / np.sqrt(n_in)\n        weight = scale * torch.randn(n_in, n_out)\n        bias = scale * torch.randn(n_out)\n        weight.requires_grad = True\n        bias.requires_grad = True\n\n        self.weights_list.extend([weight, bias])\n        return [weight, bias]\n\n    def initialize_residual(self, dims, n_in, n_out):\n        """Initializes two linear layers that map to and from the residual stream."""\n        linear_1 = self.initialize_linear(dims, [self.channel_dim_fn, n_in])\n        linear_2 = self.initialize_linear(dims, [n_out, self.channel_dim_fn])\n        return [linear_1, linear_2]\n\n    def initialize_posterior(self, dims, channel_dim):\n        """Initializes a posterior z distribution for the decoding layer."""\n        if callable(channel_dim):\n            channel_dim = channel_dim(dims)\n\n        shape = self.multitensor_system.shape(dims, channel_dim)\n        mean = 0.01 * torch.randn(shape)\n        mean.requires_grad=True\n        local_capacity_adjustment = self.initialize_zeros(dims, shape)\n\n        self.weights_list.append(mean)\n        return [mean, local_capacity_adjustment]\n\n    def initialize_direction_share(self, dims, _):\n        """\n        Initializes linear maps for the directional communication layer. Symmetrization\n        is to be performed later by symmetrize_direction_sharing().\n        """\n        channel_dim_fn = self.channel_dim_fn\n        return [[self.initialize_linear(dims, [channel_dim_fn, channel_dim_fn]) for _ in range(8)] for _ in range(8)]\n\n    def initialize_head(self):\n        """Initializes the linear head while ensuring symmetry wrt swapping x and y."""\n        dims = [1, 1, 0, 1, 1]\n        head_weights = self.initialize_linear(dims, [self.channel_dim_fn(dims), 2])\n\n        # Ensure symmetry\n        head_weights[0].requires_grad = False\n        head_weights[0] = torch.stack([head_weights[0][..., 0]] * 2, dim=-1)\n        head_weights[0].requires_grad = True\n\n        # Maintain correct weight list order\n        self.weights_list[-2] = head_weights[0]\n        return head_weights\n\n    # The functions below serve to perform the initializations once per tensor\n    # in the multitensor. Functions can also be fed in as arguments instead,\n    # and they will be run with dims as an argument, to produce a different\n    # argument for every tensor in the multitensor.\n    def initialize_multizeros(self, shape):\n        return multitensor_systems.multify(self.initialize_zeros)(\n            self.multitensor_system.make_multitensor(default=shape)\n        )\n\n    def initialize_multilinear(self, shape):\n        return multitensor_systems.multify(self.initialize_linear)(\n            self.multitensor_system.make_multitensor(default=shape)\n        )\n\n    def initialize_multiresidual(self, n_in, n_out):\n        return multitensor_systems.multify(self.initialize_residual)(\n            n_in, self.multitensor_system.make_multitensor(default=n_out)\n        )\n\n    def initialize_multiposterior(self, decoding_dim):\n        return multitensor_systems.multify(self.initialize_posterior)(\n            self.multitensor_system.make_multitensor(default=decoding_dim)\n        )\n\n    def initialize_multidirection_share(self):\n        return multitensor_systems.multify(self.initialize_direction_share)(\n            self.multitensor_system.make_multitensor()\n        )\n\n    def symmetrize_xy(self, multiweights):\n        """Ensures xy swap symmetry for weights by enforcing shared values."""\n        for dims in self.multitensor_system:\n            if dims[3] == 0 and dims[4] == 1:\n                multiweights[dims] = multiweights[dims[:3] + [1, 0]]\n\n    def symmetrize_direction_sharing(self, multiweights):\n        """\n        Ensures xy swap symmetry for weights by enforcing shared values.\n        Enforcement of shared values is more complicated since the direction axis\n        is involved, which has individual indices assigned to individual directions.\n        """\n\n        # For every directional communication linear map, identify one linear map\n        # that will serve as the representative map for all reachable maps under\n        # the equivariance transformation. Always use that representative map.\n        for dims in self.multitensor_system:\n            for dir1 in range(8):\n                for dir2 in range(8):\n                    from_dims = dims\n                    from_dir1, from_dir2 = dir1, dir2\n\n                    # Apply the transformations under certain conditions to reduce a map\n                    # to the representative map.\n                    if dims[3] + dims[4] == 1:\n                        from_dims = dims[:3] + [1, 0]\n                        if dims[4] == 1:\n                            from_dir1 = (2 + from_dir1) % 8\n                            from_dir2 = (2 + from_dir2) % 8\n\n                        if from_dir1 > 4 or (from_dir1 in {0, 4} and from_dir2 > 4):\n                            from_dir1 = (8 - from_dir1) % 8\n                            from_dir2 = (8 - from_dir2) % 8\n\n                        if 2 < from_dir1 < 6 or (from_dir1 in {2, 6} and 2 < from_dir2 < 6):\n                            from_dir1 = (4 - from_dir1) % 8\n                            from_dir2 = (4 - from_dir2) % 8\n                    else:\n                        rotation = (from_dir1 // 2) * 2\n                        from_dir1 = (from_dir1 - rotation) % 8\n                        from_dir2 = (from_dir2 - rotation) % 8\n\n                        if (from_dir2 - from_dir1) % 8 > 4:\n                            from_dir2 = (8 + 2 * from_dir1 - from_dir2) % 8\n\n                    # Copy down the representative map for later use.\n                    multiweights[dims][dir1][dir2] = multiweights[from_dims][from_dir1][from_dir2]\n\n', 'third_party/compress_arc/layers.py': 'import itertools\n\nimport numpy as np\nimport torch\n\nimport multitensor_systems\n\n"""\nThis file contains all of the layers of our network. The architecture which puts the layers\ntogether is found in arc_compressor.py.\n"""\n\nnp.random.seed(0)\ntorch.manual_seed(0)\n\n\n@multitensor_systems.multify\ndef normalize(dims, x, debias=True):\n    """\n    Normalize the tensor to have variance one, for every index along the channel dimension.\n    Args:\n        dims (list[int]): Tells you which tensor in the multitensor system we\'re normalizing\n        x (Tensor): Tensor to normalize.\n    Returns:\n        Tensor: Normalized tensor.\n    """\n    all_but_last = list(range(len(x.shape)-1))\n    if debias:\n        x = x - torch.mean(x, dim=all_but_last)\n    x = x / torch.sqrt(1e-8+torch.mean(x**2, dim=all_but_last))\n    return x\n\n@multitensor_systems.multify\ndef affine(dims, x, weight, use_bias=False):\n    """\n    Apply a linear layer to a tensor, along the channel dimension.\n    Args:\n        dims (list[int]): Tells you which tensor in the multitensor system we\'re normalizing\n        x (Tensor): Input to the linear layer.\n        weight (list[Tensor]): A weight matrix and a bias vector.\n    Returns:\n        Tensor: Output of the linear layer.\n    """\n    x = torch.matmul(x, weight[0])\n    if use_bias:\n        x = x + weight[1]\n    return x\n\ndef add_residual(layer):\n    """\n    Surround a layer/operation with a residual connection, up and down projections,\n    and pre/post-norms.\n    Args:\n        layer (Callable): The layer/operation to modify.\n    Returns:\n        Callable: Another layer/operation that applies the original layer with the\n                above modifications.\n    """\n    def layer_with_residual(dims, x, residual_weights, *args,\n                            use_bias=False, pre_norm=False, post_norm=False, **kwargs):\n        if pre_norm:\n            z = normalize(x)\n        z = affine(x, residual_weights[0], use_bias=use_bias)\n        z = layer(dims, z, *args, **kwargs)\n        if post_norm:\n            z = normalize(z)\n        z = affine(z, residual_weights[1], use_bias=use_bias)\n        return x + z\n    return layer_with_residual\n\ndef channel_layer(target_capacity, posterior):\n    """\n    Assume that z comes from some prior distribution, measure the KL divergence to the\n    posterior, and give a sample z from the posterior.\n    Args:\n        target_capacity (Tensor): Rough attempted KL capacity (reparameterized).\n        posterior (tuple[Tensor]): Consists of mean and local_capacity_adjustment. mean\n                parameterizes the mean of the posterior, and local_capacity_adjustment\n                gives the attempted KL capacity to use for each element in the tensor, in\n                log space.\n    """\n    mean, local_capacity_adjustment = posterior\n\n    all_but_last_dim = tuple(range(len(mean.shape)-1))\n    dimensionality = 1  # figure out how many elements there are in the tensor\n    for axis_length in mean.shape:\n        dimensionality *= axis_length\n    min_capacity = 0.5\n    init_capacity = 10000\n    min_capacity = torch.tensor(min_capacity)\n    init_capacity = torch.tensor(init_capacity)\n\n    target_capacity = 10*target_capacity  # this reparameterization is for faster learning\n\n    # Compute some rudimentary post-scaling of z. This output scaling leaks a bit of information that isn\'t\n    # measured by the KL, but luckily the scaling parameter is one-dimensional and probably doesn\'t have\n    # that much information in it.\n    # The output is scaled by the sigmoid of a signal-to-noise ratio, where the signal-to-noise ratio is the one\n    # that an AWGN channel would use to achieve a channel capacity equal to the desired_global_capacity below.\n    # A numerically stable formula for the sigmoid of this signal-to-noise ratio is used to compute output_scaling.\n    desired_global_capacity = torch.exp(target_capacity)*init_capacity + min_capacity\n    output_scaling = 1-torch.exp(-desired_global_capacity / dimensionality * 2)\n\n    # We make local adjustments to the desired_global_capacity in order to allow different elements to have\n    # different variances.\n    local_capacity_adjustment = (target_capacity + \n                                 local_capacity_adjustment - \n                                 torch.mean(local_capacity_adjustment, dim=all_but_last_dim))\n    desired_local_capacity = torch.exp(local_capacity_adjustment)*init_capacity + min_capacity\n\n    # Figure out what signal-to-noise ratio is required to achieve desired_local_capacity, and compute how much\n    # signal and how much noise for them to sum to one. Numerically stable formulae for these are used below.\n    noise_std = torch.exp(-desired_local_capacity / dimensionality)\n    noise_var = noise_std**2\n    stable_sqrt1memx = lambda x: torch.where(x>20, 1, torch.sqrt(1-torch.exp(-x)))\n    signal_std = stable_sqrt1memx(desired_local_capacity / dimensionality * 2)\n    signal_var = 1-noise_var\n\n    # Don\'t actually send a signal of variance equal to signal. Instead, normalize the means tensor and send that instead.\n    normalized_mean = mean - torch.mean(mean, dim=all_but_last_dim)\n    normalized_mean = normalized_mean / torch.sqrt(torch.mean(normalized_mean**2+1e-8, dim=all_but_last_dim))\n\n    # Now we can have a sample of z.\n    z = signal_std*normalized_mean + noise_std*torch.randn(normalized_mean.shape)\n    z = output_scaling*z  # leaks a tiny bit of unmeasured information, see comment above\n\n    # Calculate the KL directly instead of using the AWGN channel capacity formula, because we didn\'t\n    # actually send a signal of variance equal to signal, so the AWGN channel capacity formula would be wrong\n    # here.\n    KL = 0.5*(noise_var + signal_var*normalized_mean**2 - 1) + desired_local_capacity/dimensionality\n    return z, KL\n\ndef decode_latents(target_capacities, decode_weights, multiposteriors):\n    """\n    Decode the latents z, and give the KL loss for the VAE-like setup. Break the KL down into\n    its components for possible analysis later. Apply a linear layer afterwards.\n    Args:\n        target_capacities (MultiTensor[Tensor]): Rough attempted KL capacities (reparameterized).\n        decode_weights (MultiTensor[list[Tensor]]): A set of linear layer weights to apply to the decoded\n                outputs for every tensor in the multitensor output of the decoding layer.\n        multiposteriors (MultiTensor[tuple[Tensor]]): Consists of mean and local_capacity_adjustment. mean\n                parameterizes the mean of the posterior, and local_capacity_adjustment\n                gives the attempted KL capacity to use for each element in the tensor, in\n                log space. One (mean, local_capacity_adjustment) tuple for every tensor in the multitensor\n                system.\n    Returns:\n        MultiTensor[Tensor]: The output of the decoding layer.\n        list[Tensor]: Individual KL components that contribute to the total KL.\n        list[str]: Names for individual KL components contributing to the total KL.\n    """\n\n    KL_amounts = []\n    KL_names = []\n\n    @multitensor_systems.multify\n    def decode_latents_(dims, target_capacity, decode_weight, posterior):\n        z, KL = channel_layer(target_capacity, posterior)\n        x = affine(z, decode_weight, use_bias=True)\n        KL_amounts.append(KL)\n        KL_names.append(str(dims))\n        return x\n    x = decode_latents_(target_capacities, decode_weights, multiposteriors)\n    return x, KL_amounts, KL_names\n\n\ndef share_direction(residual, share_weights, direction):\n    """\n    Apply the multitensor communication layer.\n    Args:\n        residual (MultiTensor[Tensor]): The residual stream.\n        share_weights (Multitensor[list[list[Tensor]]]): Multiresidual projection weights.\n        direction (int): 1 for up, -1 for down.\n    Returns:\n        MultiTensor[Tensor]: The output of the multitensor communication layer.\n    """\n    \n    # Split the multiresidual into two multilinears\n    down_project_weights = multitensor_systems.multify(lambda dims, weights: weights[0])(share_weights)\n    up_project_weights = multitensor_systems.multify(lambda dims, weights: weights[1])(share_weights)\n\n    multitensor_system = residual.multitensor_system\n\n    x = affine(residual, down_project_weights, use_bias=False)  # down-project\n\n    # Define a different communication method depending on which way we\'re communicating.\n    if direction == 1:  # share up\n        def share(dims, _):\n            lower_xs = []\n            for lower_dims in multitensor_system:  # get information from all lower tensors\n                # check that lower_dims lower than dims in all indices\n                if all([lower_naxes <= naxes for lower_naxes, naxes in zip(lower_dims, dims)]):\n                    lower_x = x[lower_dims]\n                    # unsqueeze all the dimensions of lower_x until it\'s the same rank as x\n                    for dim, (lower_naxes, naxes) in enumerate(zip(lower_dims, dims)):\n                        if lower_naxes < naxes:\n                            axis = sum(dims[:dim], 0)\n                            lower_x = torch.unsqueeze(lower_x, axis)\n                    lower_xs.append(lower_x)\n            return sum(lower_xs)\n    else:  # share down\n        def share(dims, _):\n            higher_xs = []\n            for higher_dims in multitensor_system:  # get information from all higher tensors\n                # check that higher_dims higher than dims in all indices\n                if all([higher_naxes >= naxes for higher_naxes, naxes in zip(higher_dims, dims)]):\n                    higher_x = x[higher_dims]\n                    # aggregate all the dimensions of higher_x until it\'s the same rank as x\n                    for dim, (higher_naxes, naxes) in reversed(list(enumerate(zip(higher_dims, dims)))):\n                        if higher_naxes > naxes:\n                            axis = sum(higher_dims[:dim], 0)\n                            if (x.multitensor_system.task.in_out_same_size or x.multitensor_system.task.all_out_same_size) and dim==3:  # be careful aggregating the x axis\n                                # expand/contract masks to make the dims the same as higher_x\n                                masks = x.multitensor_system.task.masks\n                                masks = 1-(1-masks[...,0])*(1-masks[...,1])\n                                for i in range(sum(higher_dims[1:3])):  # insert color and direction dims\n                                    masks = masks[:,None,...]\n                                if dims[4] == 0:  # remove y dim\n                                    masks = masks[...,0]\n                                masks = masks[...,None]  # add channel dim\n                                higher_x = torch.sum(higher_x*masks, dim=axis) / (torch.sum(masks, dim=axis)+1e-4)\n                            elif (x.multitensor_system.task.in_out_same_size or x.multitensor_system.task.all_out_same_size) and dim==4:  # be careful aggregating the y axis\n                                # expand/contract masks to make the dims the same as higher_x\n                                masks = x.multitensor_system.task.masks\n                                masks = 1-(1-masks[...,0])*(1-masks[...,1])\n                                for i in range(sum(higher_dims[1:3])):  # insert color and direction dims\n                                    masks = masks[:,None,...]\n                                if higher_dims[3] == 0:  # remove x dim\n                                    masks = masks[...,0,:]\n                                masks = masks[...,None]  # add channel dim\n                                higher_x = torch.sum(higher_x*masks, dim=axis) / (torch.sum(masks, dim=axis)+1e-4)\n                            else:\n                                higher_x = torch.mean(higher_x, dim=axis)\n                    higher_xs.append(higher_x)\n            return sum(higher_xs)\n    x = multitensor_systems.multify(share)(x)  # perform the cross-tensor communication\n    x = normalize(x)  # post-norm\n    x = affine(x, up_project_weights, use_bias=False)  # up-project\n    residual = multitensor_systems.multify(lambda dims, x, y: x+y)(residual, x)  # add residual\n    return residual\n\ndef share_up(residual, share_up_weights):\n    """\n    Apply the multitensor communication layer, upwards.\n    Args:\n        residual (MultiTensor[Tensor]): The residual stream.\n        share_up_weights (Multitensor[list[list[Tensor]]]): Multiresidual projection weights.\n    Returns:\n        MultiTensor[Tensor]: The output of the multitensor communication layer.\n    """\n    return share_direction(residual, share_up_weights, 1)\n\ndef share_down(residual, share_down_weights):\n    """\n    Apply the multitensor communication layer, downwards.\n    Args:\n        residual (MultiTensor[Tensor]): The residual stream.\n        share_down_weights (Multitensor[list[list[Tensor]]]): Multiresidual projection weights.\n    Returns:\n        MultiTensor[Tensor]: The output of the multitensor communication layer.\n    """\n    return share_direction(residual, share_down_weights, -1)\n\n\ndef only_do_for_certain_shapes(*shapes):\n    """\n    Decorator which takes a function that is applied to every tensor in a multitensor,\n    and replaces that function with the identity for select tensors in the multitensor.\n    Args:\n        *shapes (list[list[int]]): A list of MultiTensor dims, for which the function\n                should be applied. Don\'t do the function if the dims for the tensor isn\'t\n                in the list.\n    """\n    def decorator(fn):\n        def filtered_fn(dims, x, *args, **kwargs):\n            if tuple(dims) in shapes:\n                return fn(dims, x, *args, **kwargs)\n            else:\n                return x\n        return filtered_fn\n    return decorator\n\n\n@multitensor_systems.multify\n@add_residual\ndef softmax(dims, x):\n    """\n    Apply the softmax layer. Take softmax over all combinations of dims, but never include\n    the channel dim nor the example dim.\n    Args:\n        dims (list[int]): Ignore this argument. It will be filled in by the multify decorator.\n        x (MultiTensor[Tensor]): The input to the softmax layer.\n    Returns:\n        MultiTensor[Tensor]: The output of the softmax layer.\n    """\n    axes = list(range(sum(dims)))\n    if dims[0]==1:\n        axes.pop(0)  # don\'t softmax over examples\n    subsets_of_axes = []\n    for subset_size in range(1, len(axes)+1):\n        subsets_of_axes = subsets_of_axes + list(itertools.combinations(axes, subset_size))\n    softmaxxes = []\n    for subset in subsets_of_axes:\n        offsets = torch.amax(x, dim=subset, keepdim=True)\n        softmax = torch.exp(x-offsets)\n        softmax = softmax / torch.sum(softmax, dim=subset, keepdim=True)\n        softmaxxes.append(softmax)\n    return torch.cat(softmaxxes, dim=-1)\n\n\ndef make_directional_layer(fn, diagonal_fn):\n    """\n    Take a directional function (one version made for cardinal directions and another for diagonal)\n    and use it to create a directional layer that works on tensors that have a direction\n    dimension.\n    Args:\n        fn (Callable): A directional function that takes a tensor and a dim argument.\n        diagonal_fn (Callable): A directional function that takes a tensor and two dim arguments.\n    Returns:\n        Callable: A function that takes a tensor with a direction dimension and applies fn and\n                diagonal_fn in a different direction for each slice of the tensor along the\n                direction dimension.\n    """\n    def directional_layer(dims, x, masks):\n        """\n        Args:\n            dims (list[int]): Ignore this argument. It will be filled in by the multify decorator.\n            x (MultiTensor[Tensor]): The input to the directional layer.\n            masks (Tensor): A (example, x, y, in/out) tensor of zeros and ones telling you which pixels are in-bounds.\n        Returns:\n            MultiTensor[Tensor]: The output of the directional layer.\n        """\n\n        # rearrange mask to fit same shape as x\n        masks = 1-(1-masks[...,0])*(1-masks[...,1])\n        if dims[4]==0:\n            masks = masks[:,:,0]\n        if dims[3]==0:\n            masks = masks[:,0,...]\n        for i in range(sum(dims[1:3])):\n            masks = masks[:,None,...]\n        masks = masks[...,None]\n        # mask out x\n        x = x*masks\n\n        # figure out which dimension the direction dimension is\n        n_directions = dims[3]+dims[4]\n        direction_dim = sum(dims[:2])\n\n        # make a default output tensor in case we try to do cumulative ops on a dimension that\n        # is not present in the tensor x\n        zero_tensor = torch.zeros_like(torch.select(x, direction_dim, 0))\n\n        # split the channel dimension into two.\n        # split the direction dimension into two.\n        # for each half of the direction dimension, each index of the direction dimension corresponds\n        # to either x or y, and we accumulate in those respective dimensions.\n        # do the other half of the channel dimension in the reverse direction.\n        # do the other half of the direction dimension in the reverse direction.\n        result_tensors = []\n        for channel_split in range(2):  # forward, backward\n            result_list = []\n            for direction_split in range(2):  # forward, backward\n                for direction_ind in range(4):  # x, x+y, y, y-x\n                    if direction_ind % 2 == 0:  # cardinal direction\n                        cardinal_direction_ind = int(direction_ind//2)\n                        if dims[3+cardinal_direction_ind]>0:\n                            x_slice = torch.select(x, direction_dim, 4*direction_split+direction_ind)\n                            x_slice = x_slice[...,channel_split::2]\n                            masks_flipped = torch.select(masks, direction_dim, 0)\n                            if direction_split + channel_split == 1:\n                                # below: decrement index to account for slicing, increment index to go from direction to x\n                                x_slice = torch.flip(x_slice, [direction_dim+cardinal_direction_ind])\n                                masks_flipped = torch.flip(masks_flipped, [direction_dim+cardinal_direction_ind])\n                            result = fn(x_slice, direction_dim+cardinal_direction_ind, masks_flipped)\n                            if direction_split + channel_split == 1:\n                                result = torch.flip(result, [direction_dim+cardinal_direction_ind])\n                        else:\n                            result = zero_tensor\n                    else:  # diagonal direction\n                        if dims[3] == 1 and dims[4] == 1:\n                            diagonal_direction_ind = int(direction_ind//2)  # 0 for x+y, 1 for y-x\n                            x_slice = torch.select(x, direction_dim, 4*direction_split+direction_ind)\n                            x_slice = x_slice[...,channel_split::2]\n                            masks_flipped = torch.select(masks, direction_dim, 0)\n                            if (direction_split + channel_split + diagonal_direction_ind) % 2 == 1:\n                                # below: decrement index to account for slicing, increment index to go from direction to x\n                                x_slice = torch.flip(x_slice, [direction_dim])\n                                masks_flipped = torch.flip(masks_flipped, [direction_dim])\n                            if direction_split + channel_split == 1:\n                                x_slice = torch.flip(x_slice, [direction_dim+1])\n                                masks_flipped = torch.flip(masks_flipped, [direction_dim+1])\n                            result = diagonal_fn(x_slice, direction_dim, direction_dim+1, masks_flipped)\n                            if (direction_split + channel_split + diagonal_direction_ind) % 2 == 1:\n                                result = torch.flip(result, [direction_dim])\n                            if direction_split + channel_split == 1:\n                                result = torch.flip(result, [direction_dim+1])\n                        else:\n                            result = zero_tensor\n                    result_list.append(result)\n            result_list = torch.stack(result_list, dim=direction_dim)  # stack direction dim together\n            result_tensors.append(result_list)\n        return torch.cat(result_tensors, dim=-1)  # cat channel dim together\n    return directional_layer\n\n"""\nFunction cummax\n\nApply the directional cummax layer.\nArgs:\n    x (MultiTensor[Tensor]): The input to the cummax layer.\n    weights (MultiTensor[list[list[Tensor]]]): Multiresidual projection weights surrounding the cummax operations.\n            Implicitly introduced by the add_residual decorator.\n    masks (Tensor): A (example, x, y, in/out) tensor of zeros and ones telling you which pixels are in-bounds.\n    Other boolean kwargs such as pre_norm, post_norm, use_bias, introduced by the add_residual decorator.\nReturns:\n    MultiTensor[Tensor]: The output of the cummax layer.\n"""\ndef cummax_(x, dim, masks):\n    masks = 1e3*(1-masks)\n    max_ = torch.max(x-masks, dim=dim, keepdim=True)[0] + masks + 1e-3\n    min_ = torch.min(x+masks, dim=dim, keepdim=True)[0] - masks - 1e-3\n    x = torch.cummax(x-masks, dim=dim)[0] + masks\n    return (x - min_) / (max_-min_) * 2 - 1\ndef diagonal_cummax_(x, dim1, dim2, masks):\n    masks_ = 1e3*(1-masks)\n    min_dim = min(x.shape[dim1], x.shape[dim2])\n    n_iters = int(np.ceil(np.log2(min_dim)))\n    # compute the cummax and max via forward+backward associative scan\n    max_x = x - masks_\n    for sign in (1, -1):\n        for i in range(n_iters):\n            shift_amount = sign*2**i\n            shifted_x = diagonal_shift_(max_x, dim1, dim2, masks_, shift_amount=shift_amount, pad_value=-1e3)\n            max_x = torch.max(max_x, shifted_x)\n        if sign == 1:  # save the cummax after the forward associative scan\n            cummax_x = max_x + masks_\n    max_x = max_x + masks_\n    # compute the min via forward+backward associative scan\n    min_x = x + masks_\n    for sign in (1, -1):\n        for i in range(n_iters):\n            shift_amount = sign*2**i\n            shifted_x = diagonal_shift_(min_x, dim1, dim2, masks_, shift_amount=shift_amount, pad_value=1e3)\n            min_x = torch.min(min_x, shifted_x)\n    min_x = min_x - masks_\n    return ((cummax_x - min_x) / (max_x-min_x+1e-5) * 2 - 1)*masks  # rescale the cummax to fit the max and min\ncummax = multitensor_systems.multify(  # apply decorators\n         only_do_for_certain_shapes((1,1,1,1,1), (1,0,1,1,1))(\n         add_residual(\n         make_directional_layer(\n         cummax_, diagonal_cummax_\n         ))))\n\n"""\nFunction shift\n\nApply the directional shift layer.\nArgs:\n    x (MultiTensor[Tensor]): The input to the shift layer.\n    weights (MultiTensor[list[list[Tensor]]]): Multiresidual projection weights surrounding the shift operations.\n            Implicitly introduced by the add_residual decorator.\n    masks (Tensor): A (example, x, y, in/out) tensor of zeros and ones telling you which pixels are in-bounds.\n    Other boolean kwargs such as pre_norm, post_norm, use_bias, introduced by the add_residual decorator.\nReturns:\n    MultiTensor[Tensor]: The output of the shift layer.\n"""\ndef shift_(x, dim, masks):\n    padding = torch.zeros_like(torch.narrow(x, dim, 0, 1))\n    narrowed = torch.narrow(x, dim, 0, x.shape[dim]-1)\n    return torch.cat([padding, narrowed], dim=dim)\ndef diagonal_shift_(x, dim1, dim2, masks, shift_amount=1, pad_value=0):\n    for dim in (dim1, dim2):\n        padding = pad_value+torch.zeros_like(torch.narrow(x, dim, 0, abs(shift_amount)))\n        if shift_amount >= 0:\n            narrowed = torch.narrow(x, dim, 0, x.shape[dim]-shift_amount)\n            x = torch.cat([padding, narrowed], dim=dim)\n        else:\n            narrowed = torch.narrow(x, dim, -shift_amount, x.shape[dim]+shift_amount)\n            x = torch.cat([narrowed, padding], dim=dim)\n    return x\nshift = multitensor_systems.multify(  # apply decorators\n        only_do_for_certain_shapes((1,1,1,1,1), (1,0,1,1,1))(\n        add_residual(\n        make_directional_layer(\n        shift_, diagonal_shift_\n        ))))\n\ndirectional_dims = [(i,j,1,k,l) for i in range(2) for j in range(2) for k in range(2) for l in range(2)]\n@multitensor_systems.multify\n@only_do_for_certain_shapes(*directional_dims)\ndef direction_share(dims, x, weights, pre_norm=True, use_bias=False):\n    """\n    Apply the directional communication layer.\n    Args:\n        dims (list[int]): Ignore this argument. It will be filled in by the multify decorator.\n        x (MultiTensor[Tensor]): The input to the directional communication layer.\n        weights (MultiTensor[list[list[list[Tensor]]]]): A multitensor full of linear layer weights\n                for every pair of directions.\n    Returns:\n        MultiTensor[Tensor]: The output of the directional communication layer.\n    """\n    # Optionally normalize the input\n    z = normalize(x) if pre_norm else x\n\n    n_directions = dims[3] + dims[4]\n    direction_dim = -2 - n_directions\n\n    # Unbind x and z along the direction dimension to avoid repeated slicing.\n    x_list = list(torch.unbind(x, dim=direction_dim))\n    z_list = list(torch.unbind(z, dim=direction_dim))\n\n    # Precomputed coefficients for the directional shift.\n    coefficients = [1, 0.2, 0.4, 0.2, 1, 0.2, 0.4, 0.2]\n\n    # Loop over all pairs of directions.\n    for d1 in range(8):\n        for d2 in range(8):\n            # Determine the appropriate coefficient.\n            c = coefficients[(d2 - d1) % 8]\n            # Apply the affine transformation for this pair and accumulate.\n            x_list[d1] = x_list[d1] + c * affine(z_list[d2], weights[d1][d2], use_bias=use_bias)\n\n    # Reassemble the tensor along the original direction dimension.\n    return torch.stack(x_list, dim=direction_dim)\n\n@multitensor_systems.multify\n@add_residual\ndef nonlinear(dims, x):\n    """\n    Apply the nonlinear layer.\n    Args:\n        dims (list[int]): Ignore this argument. It will be filled in by the multify decorator.\n        x (MultiTensor[Tensor]): The input to the nonlinear layer.\n        weights (MultiTensor[list[list[Tensor]]]): Multiresidual projection weights surrounding the nonlinear operations.\n                Implicitly introduced by the add_residual decorator.\n        Other boolean kwargs such as pre_norm, post_norm, use_bias, introduced by the add_residual decorator.\n    Returns:\n        MultiTensor[Tensor]: The output of the nonlinear layer.\n    """\n    return torch.nn.functional.silu(x)\n\ndef postprocess_mask(task, x_mask, y_mask):\n    """\n    Apply postprocessing to the masks outputted by the network. If masks are already determined\n    by the task because the output shapes follow a known hardcoded structure, then enforce the\n    known structure.\n    Args:\n        task (Task): The task that is being solved by the network.\n        x_mask (Tensor): The x mask that is outputted by the network that we must modify to fit\n                the task\'s structure.\n        y_mask (Tensor): The y mask that is outputted by the network that we must modify to fit\n                the task\'s structure.\n    Returns:\n        Tensor: Modified x mask that fits the task\'s structure.\n        Tensor: Modified y mask that fits the task\'s structure.\n    """\n\n    # Make an additive modifier mask that has large negative values for out of bounds pixels.\n    x_mask_modifier = np.zeros([task.n_examples, task.n_x, 2])\n    y_mask_modifier = np.zeros([task.n_examples, task.n_y, 2])\n    for example_num in range(task.n_examples):\n        max_length = max(task.shapes[example_num][0][0], task.shapes[example_num][1][0])\n        for in_out_mode in range(2):\n            x_mask_modifier[example_num,max_length:,in_out_mode] = -1000\n        max_length = max(task.shapes[example_num][0][1], task.shapes[example_num][1][1])\n        for in_out_mode in range(2):\n            y_mask_modifier[example_num,max_length:,in_out_mode] = -1000\n    x_mask = x_mask+torch.from_numpy(x_mask_modifier).to(x_mask.device).to(x_mask.dtype)\n    y_mask = y_mask+torch.from_numpy(y_mask_modifier).to(y_mask.device).to(y_mask.dtype)\n    return x_mask, y_mask\n', 'third_party/compress_arc/LICENSE': 'MIT License\n\nCopyright (c) 2025 Isaac Liao\n\nPermission is hereby granted, free of charge, to any person obtaining a copy\nof this software and associated documentation files (the "Software"), to deal\nin the Software without restriction, including without limitation the rights\nto use, copy, modify, merge, publish, distribute, sublicense, and/or sell\ncopies of the Software, and to permit persons to whom the Software is\nfurnished to do so, subject to the following conditions:\n\nThe above copyright notice and this permission notice shall be included in all\ncopies or substantial portions of the Software.\n\nTHE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR\nIMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,\nFITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE\nAUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER\nLIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,\nOUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE\nSOFTWARE.\n', 'third_party/compress_arc/multitensor_systems.py': 'import numpy as np\nimport torch\n\n\nnp.random.seed(0)\ntorch.manual_seed(0)\n\n\nNUM_DIMENSIONS = 5  # We have 5 dimensions: examples, colors, directions, x, y\n\nclass MultiTensorSystem:\n    """\n    A system for handling multi-dimensional configurations of \'examples\',\n    \'colors\', \'directions\', and (x, y) positions. This class can generate\n    and iterate through valid dimension combinations.\n    """\n    def __init__(self, n_examples, n_colors, n_x, n_y, task):\n        """\n        Args:\n            n_examples (int): Number of examples.\n            n_colors (int): Number of colors.\n            n_x (int): Size of the X dimension.\n            n_y (int): Size of the Y dimension.\n            task: ARC task that the multitensor system is xreated for\n        """\n        self.n_examples = n_examples\n        self.n_colors = n_colors\n        self.n_directions = 8\n        self.n_x = n_x\n        self.n_y = n_y\n        self.task = task\n        self.dim_lengths = [self.n_examples, self.n_colors,\n                            self.n_directions, self.n_x, self.n_y]\n    def dims_valid(self, dims):\n        """\n        Checks whether a given dimension combination is valid.\n        Validity rules:\n        1. If any of x/y is set (dims[3] or dims[4]), then examples (dims[0]) must also be set.\n        2. Sum of dims[1:] cannot be zero (i.e., at least color, direction, or x/y must be set).\n        Args:\n            dims (list[int]): A list of 0/1 flags indicating which dimensions are included.\n        Returns:\n            bool: Whether the dimension combination is valid.\n        """\n        # If x or y is set, then examples must also be set.\n        if (dims[3] or dims[4]) and not dims[0]:\n            return False\n        # At least one of [color, direction, x, y] must be set.\n        if sum(dims[1:]) == 0:\n            return False\n        return True\n\n    def shape(self, dims, extra_dim=None):\n        """\n        Creates a shape tuple for PyTorch or NumPy based on which dimensions are used.\n        Args:\n            dims (list[int]): A list of 0/1 flags for each dimension.\n            extra_dim (int, optional): An additional dimension to be appended at the end.\n        Returns:\n            list[int]: The computed shape.\n        """\n        shape = []\n        for dim_index, length in enumerate(self.dim_lengths):\n            if dims[dim_index]:\n                shape.append(length)\n        if extra_dim is not None:\n            shape.append(extra_dim)\n        return shape\n\n    def _generate_dims_combinations(self):\n        """Generate all possible 5-bit dimension combinations (from 0..31)."""\n        for i in range(2 ** NUM_DIMENSIONS):\n            # For each of the 5 bits in i, compute dims array\n            dims = [(i >> bit) & 1 for bit in range(NUM_DIMENSIONS)]\n            yield dims\n\n    def __iter__(self):\n        """\n        Yields valid dims.\n        """\n        for dims in self._generate_dims_combinations():\n            if self.dims_valid(dims):\n                yield dims\n\n    def _make_multitensor(self, default, index):\n        """\n        Recursively creates a nested list (tree-like) of shape [2 x 2 x 2 x 2 x 2]\n        (depth = NUM_DIMENSIONS) if `index < NUM_DIMENSIONS`.\n        Once index == NUM_DIMENSIONS, returns `default`.\n        Args:\n            default (Any): The value to return at the leaf of the recursion.\n            index (int): Current depth.\n        Returns:\n            list or default: A nested list structure or the default object if at depth.\n        """\n        if index == NUM_DIMENSIONS:\n            return default\n        return [self._make_multitensor(default, index+1) for _ in range(2)]\n\n    def make_multitensor(self, default=None):\n        """\n        Create a multitensor with a default object to place at every index.\n        Args:\n            default (Any): The default value to place at all leaves. Default: None\n        Returns:\n            MultiTensor: A multitensor with the default object at every index.\n        """\n        return MultiTensor(self._make_multitensor(default, 0), self)\n\n\nclass MultiTensor:\n    """\n    Wrapper for a nested data structure that can be indexed by a 5-element dims array.\n    """\n\n    def __init__(self, data, multitensor_system):\n        """\n        Args:\n            data (nested list): The nested list holding the actual data.\n            multitensor_system (MultiTensorSystem): The system this MultiTensor belongs to.\n        """\n        self.data = data\n        self.multitensor_system = multitensor_system\n\n    def __getitem__(self, dims):\n        """\n        Retrieve the data at a specific 5-dimensional index.\n        Args:\n            dims (list[int]): 5-element array (0 or 1) indicating path in nested lists.\n        Returns:\n            Any: The data stored at that nested location.\n        """\n        d = self.data\n        for dim_val in dims:\n            d = d[dim_val]\n        return d\n\n    def __setitem__(self, dims, value):\n        """\n        Set the data at a specific 5-dimensional index.\n        Args:\n            dims (list[int]): 5-element array (0 or 1) indicating path in nested lists.\n            value (Any): The value to store.\n        """\n        d = self.data\n        for dim_val in dims[:-1]:\n            d = d[dim_val]\n        d[dims[-1]] = value\n\n\ndef multify(fn):\n    """\n    Decorator that applies a function to all valid dimension combinations\n    if any arguments are MultiTensor instances.\n    """\n\n    def wrapper(*args, **kwargs):\n\n        # Check if we should perform multi-mode or not\n        multitensor_system = None\n        multi_mode = False\n\n        # Identify if any arg or kwarg is a MultiTensor\n        for arg in args:\n            if isinstance(arg, MultiTensor):\n                multi_mode = True\n                multitensor_system = arg.multitensor_system\n        if not multi_mode:\n            for value in kwargs.values():\n                if isinstance(value, MultiTensor):\n                    multi_mode = True\n                    multitensor_system = value.multitensor_system\n                    break\n\n        # If none of the args/kwargs are MultiTensor, just call the function directly\n        if not multi_mode:\n            return fn(None, *args, **kwargs)\n\n        # We do have MultiTensor arguments, so let\'s build a new MultiTensor result\n        # of the same shape and fill it by iterating over valid dimension combos.\n        def iterate_and_assign(multitensor_system, result_data):\n            """Helper to iterate over dims and assign function outputs."""\n\n            for dims in multitensor_system:\n                # Build per-dims argument list\n                new_args = []\n                for arg in args:\n                    if isinstance(arg, MultiTensor):\n                        new_args.append(arg[dims])\n                    else:\n                        new_args.append(arg)\n                # Build per-dims kwargs\n                new_kwargs = {}\n                for key, value in kwargs.items():\n                    if isinstance(value, MultiTensor):\n                        new_kwargs[key] = value[dims]\n                    else:\n                        new_kwargs[key] = value\n                # Call the user function on these "scalar" values\n                output = fn(dims, *new_args, **new_kwargs)\n                # Assign back to the result MultiTensor\n                # This goes step by step into result_data\n                result_data[dims] = output\n\n        # Create an empty nested list structure\n        result_data = multitensor_system.make_multitensor()\n        iterate_and_assign(multitensor_system, result_data)\n\n        # Return a MultiTensor wrapping the nested result\n        return result_data\n\n    return wrapper\n', 'third_party/compress_arc/NOTICE.md': 'Vendored from https://github.com/iliao2345/CompressARC (MIT License, see LICENSE in\nthis directory), commit as of 2026-07-27. Unmodified except where noted in\n`src/mythos/solvers/compress_arc.py`, which is our own integration wrapper, not\npart of the vendored code.\n\nPaper / write-up: https://iliao2345.github.io/blog_posts/arc_agi_without_pretraining/arc_agi_without_pretraining.html\n', 'third_party/compress_arc/preprocessing.py': 'import json\nimport numpy as np\nimport torch\nimport multitensor_systems\n\nnp.random.seed(0)\ntorch.manual_seed(0)\n\nclass Task:\n    """\n    A class that helps deal with task-specific operations such as preprocessing,\n    grid shape handling, solution processing, etc. Sets up the task-specific\n    multitensor system to be used to construct the network.\n    """\n    def __init__(self, task_name, problem, solution):\n        self.task_name = task_name\n        self.n_train = len(problem[\'train\'])\n        self.n_test = len(problem[\'test\'])\n        self.n_examples = self.n_train + self.n_test\n        self.unprocessed_problem = problem\n\n        self.shapes = self._collect_problem_shapes(problem)\n        self._predict_solution_shapes()\n        self._construct_multitensor_system(problem)\n        self._compute_mask()\n        self._create_problem_tensor(problem)\n\n        self.solution = self._create_solution_tensor(solution) if solution else None\n        if solution is None:\n            self.solution_hash = None\n\n    def _collect_problem_shapes(self, problem):\n        """\n        Extract input/output shapes for each example.\n        """\n        shapes = []\n        for split_name in [\'train\', \'test\']:\n            for example in problem[split_name]:\n                in_shape = list(np.array(example[\'input\']).shape)\n                out_shape = list(np.array(example[\'output\']).shape) if \'output\' in example else None\n                shapes.append([in_shape, out_shape])\n        return shapes\n\n    def _predict_solution_shapes(self):\n        """\n        Predict output shapes when not explicitly provided.\n        """\n        self.in_out_same_size = all(tuple(inp) == tuple(out) for inp, out in self.shapes[:self.n_train])\n        self.all_in_same_size = len({tuple(shape[0]) for shape in self.shapes}) == 1\n        self.all_out_same_size = len({tuple(shape[1]) for shape in self.shapes if shape[1]}) == 1\n\n        if self.in_out_same_size:\n            for shape in self.shapes[self.n_train:]:\n                shape[1] = shape[0]\n        elif self.all_out_same_size:\n            default_shape = self.shapes[0][1]\n            for shape in self.shapes[self.n_train:]:\n                shape[1] = default_shape\n        else:\n            max_x, max_y = self._get_max_dimensions()\n            for shape in self.shapes[self.n_train:]:\n                shape[1] = [max_x, max_y]\n\n    def _get_max_dimensions(self):\n        max_x, max_y = 0, 0\n        for in_out_pair in self.shapes:\n            for shape in in_out_pair:\n                if shape:\n                    max_x = max(max_x, shape[0])\n                    max_y = max(max_y, shape[1])\n        return max_x, max_y\n\n    def _construct_multitensor_system(self, problem):\n        """\n        Build tensor system with appropriate sizes.\n        """\n        self.n_x = max(shape[i][0] for shape in self.shapes for i in range(2))\n        self.n_y = max(shape[i][1] for shape in self.shapes for i in range(2))\n\n        colors = {color \n                  for split in [\'train\', \'test\']\n                  for example in problem[split] \n                  for grid in [example[\'input\'], example.get(\'output\', [])]\n                  for row in grid\n                  for color in row}\n        colors.add(0)  # Always include black as background\n\n        self.colors = list(sorted(colors))\n        self.n_colors = len(self.colors) - 1\n\n        self.multitensor_system = multitensor_systems.MultiTensorSystem(\n            self.n_examples, self.n_colors, self.n_x, self.n_y, self\n        )\n\n    def _create_problem_tensor(self, problem):\n        """\n        Convert input/output grids to tensors.\n        """\n        self.problem = np.zeros((self.n_examples, self.n_colors + 1, self.n_x, self.n_y, 2))\n        \n        for subsplit, n_examples in [(\'train\', self.n_train), (\'test\', self.n_test)]:\n            for example_num, example in enumerate(problem[subsplit]):\n                new_example_num = example_num if subsplit == \'train\' else self.n_train + example_num\n\n                for mode in (\'input\', \'output\'):\n                    if subsplit == \'test\' and mode == \'output\':\n                        continue\n\n                    grid = self._create_grid_tensor(\n                        example.get(mode, np.zeros(self.shapes[new_example_num][1]))\n                    )\n                    mode_num = 0 if mode == \'input\' else 1\n                    self.problem[new_example_num, :, :grid.shape[1], :grid.shape[2], mode_num] = grid\n\n        self.problem = torch.from_numpy(np.argmax(self.problem, axis=1)).to(torch.get_default_device())\n\n    def _create_grid_tensor(self, grid):\n        return np.array([\n            [[1 if self.colors.index(color) == ref_color else 0\n              for color in row]\n             for row in grid]\n            for ref_color in range(self.n_colors + 1)\n        ])\n\n    def _create_solution_tensor(self, solution):\n        """\n        Convert solution grids to tensors for crossentropy evaluation.\n        """\n        solution_tensor = np.zeros((self.n_test, self.n_colors + 1, self.n_x, self.n_y))\n        solution_tuple = ()\n\n        for example_num, grid in enumerate(solution):\n            solution_tuple += (tuple(map(tuple, grid)),)\n            grid_tensor = self._create_grid_tensor(grid)\n            # unfortunately sometimes the solution tensor will be bigger than (n_x, n_y), and in these cases\n            # we\'ll never get the solution.\n            min_x, min_y = min(grid_tensor.shape[1], self.n_x), min(grid_tensor.shape[2], self.n_y)\n            solution_tensor[example_num, :, :min_x, :min_y] = grid_tensor[:, :min_x, :min_y]\n\n        self.solution_hash = hash(solution_tuple)\n        return torch.from_numpy(np.argmax(solution_tensor, axis=1)).to(torch.get_default_device())\n\n    def _compute_mask(self):\n        """\n        Compute masks for activations and cross-entropies.\n        """\n        self.masks = np.zeros((self.n_examples, self.n_x, self.n_y, 2))\n\n        for example_num, (in_shape, out_shape) in enumerate(self.shapes):\n            for mode_num, shape in enumerate([in_shape, out_shape]):\n                if shape:\n                    x_mask = np.arange(self.n_x) < shape[0]\n                    y_mask = np.arange(self.n_y) < shape[1]\n                    self.masks[example_num, :, :, mode_num] = np.outer(x_mask, y_mask)\n\n        self.masks = torch.from_numpy(self.masks).to(torch.get_default_dtype()).to(torch.get_default_device())\n\n\ndef preprocess_tasks(split, task_nums_or_task_names):\n    """\n    Preprocess tasks by loading problems and solutions.\n    """\n    with open(f\'dataset/arc-agi_{split}_challenges.json\', \'r\') as f:\n        problems = json.load(f)\n\n    solutions = None if split == "test" else json.load(open(f\'dataset/arc-agi_{split}_solutions.json\'))\n    \n    task_names = list(problems.keys())\n    \n    return [Task(task_name,\n                 problems[task_name],\n                 solutions.get(task_name) if solutions else None)\n            for task_name in task_names\n            if task_name in task_nums_or_task_names or task_names.index(task_name) in task_nums_or_task_names]\n', 'third_party/compress_arc/solution_selection.py': 'import matplotlib.pyplot as plt\nimport numpy as np\nimport torch\n\nnp.random.seed(0)\ntorch.manual_seed(0)\n\nclass Logger:\n    """\n    This class contains functionalities relating to the recording of model outputs, postprocessing,\n    selection of most frequently sampled/highest scoring solutions, accuracy computations, and more.\n    """\n    ema_decay = 0.97\n\n    def __init__(self, task):\n        self.task = task\n        self.KL_curves = {}\n        self.total_KL_curve = []\n        self.reconstruction_error_curve = []\n        self.loss_curve = []\n\n        n_test, n_colors, n_x, n_y = task.n_test, task.n_colors, task.n_x, task.n_y\n        shape = (n_test, n_colors + 1, n_x, n_y)\n\n        self.current_logits = torch.zeros(shape)\n        self.current_x_mask = torch.zeros((n_test, n_x))\n        self.current_y_mask = torch.zeros((n_test, n_y))\n\n        self.ema_logits = torch.zeros(shape)\n        self.ema_x_mask = torch.zeros((n_test, n_x))\n        self.ema_y_mask = torch.zeros((n_test, n_y))\n\n        self.solution_hashes_count = {}\n        self.solution_most_frequent = None\n        self.solution_second_most_frequent = None\n\n        self.solution_contributions_log = []\n        self.solution_picks_history = []\n\n    def log(self, train_step, logits, x_mask, y_mask, KL_amounts, KL_names, total_KL, reconstruction_error, loss):\n        """Logs training progress and tracks solutions from one forward pass."""\n        if train_step == 0:\n            self.KL_curves = {KL_name: [] for KL_name in KL_names}\n\n        for KL_amount, KL_name in zip(KL_amounts, KL_names):\n            self.KL_curves[KL_name].append(float(KL_amount.detach().sum().cpu().numpy()))\n\n        self.total_KL_curve.append(float(total_KL.detach().cpu().numpy()))\n        self.reconstruction_error_curve.append(float(reconstruction_error.detach().cpu().numpy()))\n        self.loss_curve.append(float(loss.detach().cpu().numpy()))\n\n        self._track_solution(train_step, logits.detach(), x_mask.detach(), y_mask.detach())\n\n    def _track_solution(self, train_step, logits, x_mask, y_mask):\n        """Postprocess and score solutions and keep track of the top two solutions with highest scores."""\n        self.current_logits = logits[self.task.n_train:, :, :, :, 1]  # example, color, x, y\n        self.current_x_mask = x_mask[self.task.n_train:, :, 1]  # example, x\n        self.current_y_mask = y_mask[self.task.n_train:, :, 1]  # example, y\n\n        self.ema_logits = self.ema_decay * self.ema_logits + (1 - self.ema_decay) * self.current_logits\n        self.ema_x_mask = self.ema_decay * self.ema_x_mask + (1 - self.ema_decay) * self.current_x_mask\n        self.ema_y_mask = self.ema_decay * self.ema_y_mask + (1 - self.ema_decay) * self.current_y_mask\n\n        solution_contributions = []\n        for logits, x_mask_set, y_mask_set in [  # Add two potential solutions: sample and mean.\n            (self.current_logits, self.current_x_mask, self.current_y_mask),\n            (self.ema_logits, self.ema_x_mask, self.ema_y_mask)\n        ]:\n\n            # Get the solution and the score.\n            solution, uncertainty = self._postprocess_solution(logits, x_mask_set, y_mask_set)\n            hashed_solution = hash(solution)\n            score = -10*uncertainty\n            if train_step < 150:\n                score = score - 10\n            if logits is self.ema_logits:\n                score = score - 4\n\n            # Accumulate scores for solutions.\n            solution_contributions.append((hashed_solution, score))\n            self.solution_hashes_count[hashed_solution] = float(np.logaddexp(\n                self.solution_hashes_count.get(hashed_solution, -np.inf), score))\n\n            self._update_most_frequent_solutions(hashed_solution, solution)\n\n        self.solution_contributions_log.append(solution_contributions)\n        self.solution_picks_history.append([hash(sol) for sol in [\n            self.solution_most_frequent, self.solution_second_most_frequent]])\n\n    def _update_most_frequent_solutions(self, hashed, solution):\n        """Keeps track of the top two solutions with highest scores."""\n        if self.solution_most_frequent is None:\n            self.solution_most_frequent = solution\n        if self.solution_second_most_frequent is None:\n            self.solution_second_most_frequent = solution\n\n        if hashed != hash(self.solution_most_frequent):\n            if self.solution_hashes_count[hashed] >= self.solution_hashes_count.get(\n                    hash(self.solution_second_most_frequent), -np.inf):\n                self.solution_second_most_frequent = solution\n                if self.solution_hashes_count[hashed] >= self.solution_hashes_count.get(\n                        hash(self.solution_most_frequent), -np.inf):\n                    self.solution_second_most_frequent = self.solution_most_frequent\n                    self.solution_most_frequent = solution\n\n    def best_crop(self, prediction, x_mask, x_length, y_mask, y_length):\n        x_start, x_end = self._best_slice_point(x_mask, x_length)\n        y_start, y_end = self._best_slice_point(y_mask, y_length)\n        return prediction[..., x_start:x_end, y_start:y_end]\n\n    def _best_slice_point(self, mask, length):\n        if self.task.in_out_same_size or self.task.all_out_same_size:\n            search_lengths = [length]\n        else:\n            search_lengths = list(range(1, mask.shape[0]+1))\n        max_logprob, best_slice_start, best_slice_end = None, None, None\n\n        for length in search_lengths:\n            logprobs = torch.stack([\n                -torch.sum(mask[:offset]) + torch.sum(mask[offset:offset + length]) - torch.sum(mask[offset + length:])\n                for offset in range(mask.shape[0] - length + 1)\n            ])\n            if max_logprob is None or torch.max(logprobs) > max_logprob:\n                max_logprob = torch.max(logprobs)\n                best_slice_start = torch.argmax(logprobs).item()\n                best_slice_end = best_slice_start + length\n\n        return best_slice_start, best_slice_end\n\n    def _postprocess_solution(self, prediction, x_mask, y_mask):  # prediction must be example, color, x, y\n        """Postprocess a solution and compute some variables that are used to calculate the score."""\n        colors = torch.argmax(prediction, dim=1)  # example, x, y\n        uncertainties = torch.logsumexp(prediction, dim=1) - torch.amax(prediction, dim=1)  # example, x, y\n        solution_slices, uncertainty_values = [], []  # example, x, y; example\n\n        for example_num in range(self.task.n_test):\n            x_length = None\n            y_length = None\n            if self.task.in_out_same_size or self.task.all_out_same_size:\n                x_length = self.task.shapes[self.task.n_train+example_num][1][0]\n                y_length = self.task.shapes[self.task.n_train+example_num][1][1]\n            solution_slice = self.best_crop(colors[example_num],\n                                            x_mask[example_num],\n                                            x_length,\n                                            y_mask[example_num],\n                                            y_length)  # x, y\n            uncertainty_slice = self.best_crop(uncertainties[example_num],\n                                               x_mask[example_num],\n                                               x_length,\n                                               y_mask[example_num],\n                                               y_length)  # x, y\n\n            solution_slices.append(solution_slice.cpu().numpy().tolist())\n            uncertainty_values.append(float(np.mean(uncertainty_slice.cpu().numpy())))\n\n        for example in solution_slices:\n            for row in example:\n                for i, val in enumerate(row):\n                    row[i] = self.task.colors[val]\n\n        solution_slices = tuple(tuple(tuple(row) for row in example) for example in solution_slices)\n        return solution_slices, np.mean(uncertainty_values)\n\n\ndef save_predictions(loggers, fname=\'predictions.npz\'):\n    """Saves solution score contributions and history of chosen solutions."""\n    np.savez(fname,\n             solution_contribution_logs=[logger.solution_contributions_log for logger in loggers],\n             solution_picks_histories=[logger.solution_picks_history for logger in loggers])\n\n\ndef plot_accuracy(true_solution_hashes, fname=\'predictions.npz\'):\n    """Plots accuracy curve over training iterations."""\n    stored_data = np.load(fname, allow_pickle=True)\n    solution_picks_histories = stored_data[\'solution_picks_histories\']\n\n    n_tasks = len(solution_picks_histories)\n    n_iterations = len(solution_picks_histories[0])\n\n    correct = np.array([[\n        int(any(hash_ == true_solution_hashes[task_num] for hash_ in solution_pair))\n        for solution_pair in task_history\n    ] for task_num, task_history in enumerate(solution_picks_histories)])\n\n    accuracy_curve = correct.mean(axis=0)\n\n    plt.figure()\n    plt.plot(np.arange(n_iterations), accuracy_curve, \'k-\')\n    plt.savefig(\'accuracy_curve.pdf\', bbox_inches=\'tight\')\n    plt.close()\n', 'third_party/compress_arc/solve_task.py': 'import os\nimport sys\nimport time\nimport json\nimport importlib\nimport gc\nimport multiprocessing\nimport tqdm\nimport traceback\n\nimport numpy as np\nimport torch\n\nimport preprocessing\nimport train\nimport arc_compressor\nimport initializers\nimport multitensor_systems\nimport layers\nimport solution_selection\nimport visualization\n\n"""\nA script that solves one puzzle, to be imported and used with parallel_train.py and multiprocessing.\n"""\n\ndef solve_task(task_name, split, time_limit, n_train_iterations, gpu_id, memory_dict, solutions_dict, error_queue):\n    """\n    Solves a puzzle.\n    Args:\n        task_name (str): The name of the puzzle to solve.\n        split (str): \'training\', \'evaluation\', or \'test\'\n        time_limit (float): An end time that will cause training to exit early if reached.\n        n_train_iterations (int): The number of iterations to train for.\n        gpu_id (int): The GPU number to run the solver on.\n        memory_dict (multiprocessing.Dict[str, int]): An inter-process shared dict that we\n            can store the amount of memory taken by this job in.\n        solutions_dict (multiprocessing.Dict[str, list[Dict[str, list[list[int]]]]]): An\n            inter-process shared dict that we can store the solution in.\n        error_queue (multiprocessing.Queue[Exception]): An inter-process shared queue to\n            put errors in when an exception occurs.\n    """\n\n    try:  # Error catching block that puts errors on the error_queue\n\n        torch.set_default_device(\'cuda\')\n        torch.cuda.set_device(gpu_id)\n        torch.cuda.reset_peak_memory_stats()  # Measure the memory used.\n\n        # Get the task\n        with open(f\'dataset/arc-agi_{split}_challenges.json\', \'r\') as f:\n            problems = json.load(f)\n        task = preprocessing.Task(task_name, problems[task_name], None)\n        del problems\n\n        # Set up the training\n        model = arc_compressor.ARCCompressor(task)\n        optimizer = torch.optim.Adam(model.weights_list, lr=0.01, betas=(0.5, 0.9))\n        train_history_logger = solution_selection.Logger(task)\n        train_history_logger.solution_most_frequent = tuple(((0, 0), (0, 0)) for example_num in range(task.n_test))\n        train_history_logger.solution_second_most_frequent = tuple(((0, 0), (0, 0)) for example_num in range(task.n_test))\n\n        # Training loop\n        for train_step in range(n_train_iterations):\n            train.take_step(task, model, optimizer, train_step, train_history_logger)\n            if time.time() > time_limit:\n                break\n\n        # Get the solution\n        example_list = []\n        for example_num in range(task.n_test):\n            attempt_1 = [list(row) for row in train_history_logger.solution_most_frequent[example_num]]\n            attempt_2 = [list(row) for row in train_history_logger.solution_second_most_frequent[example_num]]\n            example_list.append({\'attempt_1\': attempt_1, \'attempt_2\': attempt_2})\n        del task\n        del model\n        del optimizer\n        del train_history_logger\n        torch.cuda.empty_cache()\n        gc.collect()\n\n        # Store the result\n        memory_dict[task_name] = torch.cuda.max_memory_allocated()\n        solutions_dict[task_name] = example_list\n\n    except Exception as e:  # If error, write to the error queue\n        error_queue.put(traceback.format_exc())\n', 'third_party/compress_arc/train.py': 'import time\n\nimport numpy as np\nimport torch\n\nimport preprocessing\nimport arc_compressor\nimport initializers\nimport multitensor_systems\nimport layers\nimport solution_selection\nimport visualization\n\n\n"""\nThis file trains a model for every ARC-AGI task in a split.\n"""\n\nnp.random.seed(0)\ntorch.manual_seed(0)\n\n\ndef mask_select_logprobs(mask, length):\n    """\n    Figure out the unnormalized log probability of taking each slice given the output mask.\n    """\n    logprobs = []\n    for offset in range(mask.shape[0]-length+1):\n        logprob = -torch.sum(mask[:offset])\n        logprob = logprob + torch.sum(mask[offset:offset+length])\n        logprob = logprob - torch.sum(mask[offset+length:])\n        logprobs.append(logprob)\n    logprobs = torch.stack(logprobs, dim=0)\n    log_partition = torch.logsumexp(logprobs, dim=0)\n    return log_partition, logprobs\n\ndef take_step(task, model, optimizer, train_step, train_history_logger):\n    """\n    Runs a forward pass of the model on the ARC-AGI task.\n    Args:\n        task (Task): The ARC-AGI task containing the problem.\n        model (ArcCompressor): The VAE decoder model to run the forward pass with.\n        optimizer (torch.optim.Optimizer): The optimizer used to take the step on the model weights.\n        train_step (int): The training iteration number.\n        train_history_logger (Logger): A logger object used for logging the forward pass outputs\n                of the model, as well as accuracy and other things.\n    """\n\n    optimizer.zero_grad()\n    logits, x_mask, y_mask, KL_amounts, KL_names, = model.forward()\n    logits = torch.cat([torch.zeros_like(logits[:,:1,:,:]), logits], dim=1)  # add black color to logits\n\n    # Compute the total KL loss\n    total_KL = 0\n    for KL_amount in KL_amounts:\n        total_KL = total_KL + torch.sum(KL_amount)\n\n    # Compute the reconstruction error\n    reconstruction_error = 0\n    for example_num in range(task.n_examples):  # sum over examples\n        for in_out_mode in range(2):  # sum over in/out grid per example\n            if example_num >= task.n_train and in_out_mode == 1:\n                continue\n\n            # Determine whether the grid size is already known.\n            # If not, there is an extra term in the reconstruction error, corresponding to\n            # the probability of reconstructing the correct grid size.\n            grid_size_uncertain = not (task.in_out_same_size or task.all_out_same_size and in_out_mode==1 or task.all_in_same_size and in_out_mode==0)\n            if grid_size_uncertain:\n                coefficient = 0.01**max(0, 1-train_step/100)\n            else:\n                coefficient = 1\n            logits_slice = logits[example_num,:,:,:,in_out_mode]  # color, x, y\n            problem_slice = task.problem[example_num,:,:,in_out_mode]  # x, y\n            output_shape = task.shapes[example_num][in_out_mode]\n            x_log_partition, x_logprobs = mask_select_logprobs(coefficient*x_mask[example_num,:,in_out_mode], output_shape[0])\n            y_log_partition, y_logprobs = mask_select_logprobs(coefficient*y_mask[example_num,:,in_out_mode], output_shape[1])\n            # Account for probability of getting right grid size, if grid size is not known\n            if grid_size_uncertain:\n                x_log_partitions = []\n                y_log_partitions = []\n                for length in range(1, x_mask.shape[1]+1):\n                    x_log_partitions.append(mask_select_logprobs(coefficient*x_mask[example_num,:,in_out_mode], length)[0])\n                for length in range(1, y_mask.shape[1]+1):\n                    y_log_partitions.append(mask_select_logprobs(coefficient*y_mask[example_num,:,in_out_mode], length)[0])\n                x_log_partition = torch.logsumexp(torch.stack(x_log_partitions, dim=0), dim=0)\n                y_log_partition = torch.logsumexp(torch.stack(y_log_partitions, dim=0), dim=0)\n\n            # Given that we have the correct grid size, get the reconstruction error of getting the colors right\n            logprobs = [[] for x_offset in range(x_logprobs.shape[0])]  # x, y\n            for x_offset in range(x_logprobs.shape[0]):\n                for y_offset in range(y_logprobs.shape[0]):\n                    logprob = x_logprobs[x_offset] - x_log_partition + y_logprobs[y_offset] - y_log_partition  # given the correct grid size,\n                    logits_crop = logits_slice[:,x_offset:x_offset+output_shape[0],y_offset:y_offset+output_shape[1]]  # c, x, y\n                    target_crop = problem_slice[:output_shape[0],:output_shape[1]]  # x, y\n                    logprob = logprob - torch.nn.functional.cross_entropy(logits_crop[None,...], target_crop[None,...], reduction=\'sum\')  # calculate the error for the colors.\n                    logprobs[x_offset].append(logprob)\n            logprobs = torch.stack([torch.stack(logprobs_, dim=0) for logprobs_ in logprobs], dim=0)  # x, y\n            if grid_size_uncertain:\n                coefficient = 0.1**max(0, 1-train_step/100)\n            else:\n                coefficient = 1\n            logprob = torch.logsumexp(coefficient*logprobs, dim=(0,1))/coefficient  # Aggregate for all possible grid sizes\n            reconstruction_error = reconstruction_error - logprob\n\n    loss = total_KL + 10*reconstruction_error\n    loss.backward()\n    optimizer.step()\n    optimizer.zero_grad()\n\n    # Performance recording\n    train_history_logger.log(train_step,\n                             logits,\n                             x_mask,\n                             y_mask,\n                             KL_amounts,\n                             KL_names,\n                             total_KL,\n                             reconstruction_error,\n                             loss)\n\n\nif __name__ == "__main__":\n    start_time = time.time()\n\n    task_nums = list(range(400))\n    split = "training"  # "training", "evaluation, or "test"\n\n    # Preprocess all tasks, make models, optimizers, and loggers. Make plots.\n    tasks = preprocessing.preprocess_tasks(split, task_nums)\n    models = []\n    optimizers = []\n    train_history_loggers = []\n    for task in tasks:\n        model = arc_compressor.ARCCompressor(task)\n        models.append(model)\n        optimizer = torch.optim.Adam(model.weights_list, lr=0.01, betas=(0.5, 0.9))\n        optimizers.append(optimizer)\n        train_history_logger = solution_selection.Logger(task)\n        visualization.plot_problem(train_history_logger)\n        train_history_loggers.append(train_history_logger)\n\n    # Get the solution hashes so that we can check for correctness\n    true_solution_hashes = [task.solution_hash for task in tasks]\n\n    # Train the models one by one\n    for i, (task, model, optimizer, train_history_logger) in enumerate(zip(tasks, models, optimizers, train_history_loggers)):\n        n_iterations = 2000\n        for train_step in range(n_iterations):\n            take_step(task, model, optimizer, train_step, train_history_logger)\n        visualization.plot_solution(train_history_logger)\n        solution_selection.save_predictions(train_history_loggers[:i+1])\n        solution_selection.plot_accuracy(true_solution_hashes)\n\n    # Write down how long it all took\n    with open(\'timing_result.txt\', \'w\') as f:\n        f.write("Time elapsed in seconds: " + str(time.time() - start_time))\n', 'third_party/compress_arc/visualization.py': 'import os\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport torch\n\n\n"""\nThis file trains a model for every ARC-AGI task in a split.\n"""\n\nnp.random.seed(0)\ntorch.manual_seed(0)\n\n\ncolor_list = np.array([\n    [0, 0, 0],  # black\n    [30, 147, 255],  # blue\n    [249, 60, 49],  # red\n    [79, 204, 48],  # green\n    [255, 220, 0],  # yellow\n    [153, 153, 153],  # gray\n    [229, 58, 163],  # magenta\n    [255, 133, 27],  # orange\n    [135, 216, 241],  # light blue\n    [146, 18, 49],  # brown\n])\n\ndef convert_color(grid):  # grid dims must end in c\n    return np.clip(np.matmul(grid, color_list), 0, 255).astype(np.uint8)\n\ndef plot_problem(logger):\n    """\n    Draw a plot of an ARC-AGI problem, and save it in plots/\n    Args:\n        logger (Logger): A logger object used to log model outputs for the ARC-AGI task.\n    """\n\n    # Put all the grids beside one another on one grid\n    n_train = logger.task.n_train\n    n_test = logger.task.n_test\n    n_examples = logger.task.n_examples\n    n_x = logger.task.n_x\n    n_y = logger.task.n_y\n    pixels = 255+np.zeros([n_train+n_test, 2*n_x+2, 2, 2*n_y+8, 3], dtype=np.uint8)\n    for example_num in range(n_examples):\n        if example_num < n_train:\n            subsplit = \'train\'\n            subsplit_example_num = example_num\n        else:\n            subsplit = \'test\'\n            subsplit_example_num = example_num - n_train\n        for mode_num, mode in enumerate((\'input\', \'output\')):\n            if subsplit == \'test\' and mode == \'output\':\n                continue\n            grid = np.array(logger.task.unprocessed_problem[subsplit][subsplit_example_num][mode])  # x, y\n            grid = (np.arange(10)==grid[:,:,None]).astype(np.float32)  # x, y, c\n            grid = convert_color(grid)  # x, y, c\n            repeat_grid = np.repeat(grid, 2, axis=0)\n            repeat_grid = np.repeat(repeat_grid, 2, axis=1)\n            pixels[example_num,n_x+1-grid.shape[0]:n_x+1+grid.shape[0],mode_num,n_y+4-grid.shape[1]:n_y+4+grid.shape[1],:] = repeat_grid\n    pixels = pixels.reshape([(n_train+n_test)*(2*n_x+2), 2*(2*n_y+8), 3])\n    \n    os.makedirs("plots/", exist_ok=True)\n\n    # Plot the combined grid and make gray dividers between the grid cells, arrows, and a question mark for unsolved examples.\n    fig, ax = plt.subplots()\n    ax.imshow(pixels, aspect=\'equal\', interpolation=\'none\')\n    for example_num in range(n_examples):\n        for mode_num, mode in enumerate((\'input\', \'output\')):\n            if example_num < n_train:\n                subsplit = \'train\'\n                subsplit_example_num = example_num\n            else:\n                subsplit = \'test\'\n                subsplit_example_num = example_num - n_train\n            ax.arrow((2*n_y+8)-3-0.5, (2*n_x+2)*example_num+1+n_x-0.5, 6, 0, width=0.5, fc=\'k\', ec=\'k\', length_includes_head=True)\n            if subsplit == \'test\' and mode == \'output\':\n                ax.text((2*n_y+8)+4+n_y-0.5, (2*n_x+2)*example_num+1+n_x-0.5, \'?\', size=\'xx-large\', ha=\'center\', va=\'center\')\n                continue\n            grid = np.array(logger.task.unprocessed_problem[subsplit][subsplit_example_num][mode])  # x, y\n            for xline in range(grid.shape[0]+1):\n                ax.plot(((2*n_y+8)*mode_num+4+n_y-grid.shape[1]-0.5, (2*n_y+8)*mode_num+4+n_y+grid.shape[1]-0.5),\n                        ((2*n_x+2)*example_num+1+n_x-grid.shape[0]+2*xline-0.5,)*2,\n                        color=(59/255, 59/255, 59/255),\n                        linewidth=0.3)\n            for yline in range(grid.shape[1]+1):\n                ax.plot(((2*n_y+8)*mode_num+4+n_y-grid.shape[1]+2*yline-0.5,)*2,\n                        ((2*n_x+2)*example_num+1+n_x-grid.shape[0]-0.5, (2*n_x+2)*example_num+1+n_x+grid.shape[0]-0.5),\n                        color=(59/255, 59/255, 59/255),\n                        linewidth=0.3)\n    plt.axis(\'off\')\n    plt.savefig(\'plots/\' + logger.task.task_name + \'_problem.png\', bbox_inches=\'tight\', pad_inches=0)\n    plt.close()\n\ndef plot_solution(logger, fname=None):\n    """\n    Draw a plot of a model\'s solution to an ARC-AGI problem, and save it in plots/\n    Draws four plots: A model output sample, the mean of samples, and the top two most common samples.\n    Args:\n        logger (Logger): A logger object used to log model outputs for the ARC-AGI task.\n    """\n    n_train = logger.task.n_train\n    n_test = logger.task.n_test\n    n_examples = logger.task.n_examples\n    n_x = logger.task.n_x\n    n_y = logger.task.n_y\n\n    # Four plotted solutions\n    solutions_list = [\n            torch.softmax(logger.current_logits, dim=1).cpu().numpy(),\n            torch.softmax(logger.ema_logits, dim=1).cpu().numpy(),\n            logger.solution_most_frequent,\n            logger.solution_second_most_frequent,\n            ]\n    masks_list = [\n            (logger.current_x_mask, logger.current_y_mask),\n            (logger.ema_x_mask, logger.ema_y_mask),\n            None,\n            None,\n            ]\n    solutions_labels = [\n            \'sample\',\n            \'sample average\',\n            \'guess 1\',\n            \'guess 2\',\n            ]\n    n_plotted_solutions = len(solutions_list)\n\n    # Put all the grids beside one another on one grid\n    pixels = 255+np.zeros([n_test, 2*n_x+2, n_plotted_solutions, 2*n_y+8, 3], dtype=np.uint8)\n    shapes = []\n    for subsplit_example_num in range(n_test):\n        subsplit = \'test\'\n        example_num = subsplit_example_num + n_train\n        shapes.append([])\n\n        for solution_num, (solution, masks, label) in enumerate(zip(solutions_list, masks_list, solutions_labels)):\n            grid = np.array(solution[subsplit_example_num])  # c, x, y if \'sample\' in label else x, y, c\n            if \'sample\' in label:\n                grid = np.einsum(\'dxy,dc->xyc\', grid, color_list[logger.task.colors])  # x, y, c\n                if logger.task.in_out_same_size or logger.task.all_out_same_size:\n                    x_length = logger.task.shapes[example_num][1][0]\n                    y_length = logger.task.shapes[example_num][1][1]\n                else:\n                    x_length = None\n                    y_length = None\n                x_start, x_end = logger._best_slice_point(masks[0][subsplit_example_num,:], x_length)\n                y_start, y_end = logger._best_slice_point(masks[1][subsplit_example_num,:], y_length)\n                grid = grid[x_start:x_end,y_start:y_end,:]  # x, y, c\n                grid = np.clip(grid, 0, 255).astype(np.uint8)\n            else:\n                grid = (np.arange(10)==grid[:,:,None]).astype(np.float32)  # x, y, c\n                grid = convert_color(grid)  # x, y, c\n\n            shapes[subsplit_example_num].append((grid.shape[0], grid.shape[1]))\n            repeat_grid = np.repeat(grid, 2, axis=0)\n            repeat_grid = np.repeat(repeat_grid, 2, axis=1)\n            pixels[subsplit_example_num,n_x+1-grid.shape[0]:n_x+1+grid.shape[0],solution_num,n_y+4-grid.shape[1]:n_y+4+grid.shape[1],:] = repeat_grid\n\n    pixels = pixels.reshape([n_test*(2*n_x+2), n_plotted_solutions*(2*n_y+8), 3])\n    \n    # Plot the combined grid and make gray dividers between the grid cells, and labels.\n    fig, ax = plt.subplots()\n    ax.imshow(pixels, aspect=\'equal\', interpolation=\'none\')\n    for subsplit_example_num in range(n_test):\n        for solution_num in range(n_plotted_solutions):\n            subsplit = \'test\'\n            grid = np.array(solutions_list[solution_num][subsplit_example_num])  # x, y\n            shape = shapes[subsplit_example_num][solution_num]\n            for xline in range(shape[0]+1):\n                ax.plot(((2*n_y+8)*solution_num+4+n_y-shape[1]-0.5, (2*n_y+8)*solution_num+4+n_y+shape[1]-0.5),\n                        ((2*n_x+2)*subsplit_example_num+1+n_x-shape[0]+2*xline-0.5,)*2,\n                        color=(59/255, 59/255, 59/255),\n                        linewidth=0.3)\n            for yline in range(shape[1]+1):\n                ax.plot(((2*n_y+8)*solution_num+4+n_y-shape[1]+2*yline-0.5,)*2,\n                        ((2*n_x+2)*subsplit_example_num+1+n_x-shape[0]-0.5, (2*n_x+2)*subsplit_example_num+1+n_x+shape[0]-0.5),\n                        color=(59/255, 59/255, 59/255),\n                        linewidth=0.3)\n    for solution_num, solution_label in enumerate(solutions_labels):\n        ax.text((2*n_y+8)*solution_num+4+n_y-0.5, -3, solution_label, size=\'xx-small\', ha=\'center\', va=\'center\')\n    plt.axis(\'off\')\n    if fname is None:\n        fname = \'plots/\' + logger.task.task_name + \'_solutions.pdf\'\n    plt.savefig(fname, bbox_inches=\'tight\', pad_inches=0)\n    plt.close()\n\n\n', 'agentic_repl/__init__.py': '"""Agentic program-synthesis & REPL-refinement ARC solver.\n\nKept as a top-level package, separate from the shelved src/mythos neural\npath (HRM/TTT/LoRA/CompressARC), but built on top of mythos\'s data model\n(ArcTask/Grid) and solver contract (mythos.solvers.base.Solver) and reusing\nmythos\'s verified grid-primitive modules (objects/object_ops/symmetry/\naugment) as the DSL surface offered to a code-generating LLM.\n"""\n', 'agentic_repl/augment_vote.py': '"""Augmentation + majority voting: turn a pool of verified programs into the\ntwo grids for attempt_1/attempt_2.\n\nEvery verified program already reproduces every train pair exactly (see\nsolver.py\'s _verify_on_train) -- this stage isn\'t re-checking correctness,\nit\'s a robustness/agreement signal, mirroring mythos.augment\'s existing\ninference-time-ensembling rationale (see src/mythos/augment.py\'s module\ndocstring) applied to program *outputs* instead of model logits. There is no\nmajority-voting/candidate-ranking logic anywhere in mythos.score, so it has\nto live here, before make_prediction ever sees the two chosen grids.\n"""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom typing import Callable\n\nfrom mythos.arc import ArcTask, Grid\nfrom mythos.augment import NUM_TRANSFORMS, forward_transform, inverse_transform\nfrom mythos.solvers.base import SolverError\n\nfrom agentic_repl.repl import ExecutionResult\n\nRunCandidate = Callable[..., ExecutionResult]\nGridKey = tuple[tuple[int, ...], ...]\n\n\ndef _grid_key(grid: Grid) -> GridKey:\n    return tuple(tuple(row) for row in grid)\n\n\ndef _predict_one(\n    code: str,\n    input_grid: Grid,\n    *,\n    run_candidate: RunCandidate,\n    timeout_s: float,\n) -> Grid | None:\n    """Run one verified program across all D4 views of input_grid and self-vote.\n\n    Reverse-transforming each augmented view\'s output back to the original\n    frame and voting across them catches programs that are only *coincidentally*\n    correct on train in one orientation -- a genuine rule agrees with itself\n    across every view once un-rotated; a lucky one usually doesn\'t.\n    """\n\n    votes: Counter[GridKey] = Counter()\n    grids_by_key: dict[GridKey, Grid] = {}\n    for transform_index in range(NUM_TRANSFORMS):\n        augmented_input = forward_transform(transform_index, input_grid)\n        result = run_candidate(code, augmented_input, timeout_s=timeout_s)\n        if not result.ok or result.output is None:\n            continue\n        try:\n            restored = inverse_transform(transform_index, result.output)\n        except Exception:  # noqa: BLE001 - a malformed candidate output shouldn\'t crash voting\n            continue\n        key = _grid_key(restored)\n        votes[key] += 1\n        grids_by_key[key] = restored\n\n    if not votes:\n        return None\n    best_key, _ = votes.most_common(1)[0]\n    return grids_by_key[best_key]\n\n\ndef vote_predictions(\n    verified_codes: list[str],\n    task: ArcTask,\n    *,\n    run_candidate: RunCandidate,\n    timeout_s: float,\n) -> list[tuple[Grid, Grid]]:\n    """Return one (attempt_1, attempt_2) pair per task.test item.\n\n    For each test item, every verified program casts one (augmentation\n    self-voted) prediction; those per-program predictions are then\n    majority-voted across programs. The top two distinct results become\n    attempt_1/attempt_2 (attempt_2 repeats attempt_1 if only one distinct\n    grid was produced at all).\n    """\n\n    attempts: list[tuple[Grid, Grid]] = []\n    for test_example in task.test:\n        program_votes: Counter[GridKey] = Counter()\n        grids_by_key: dict[GridKey, Grid] = {}\n        for code in verified_codes:\n            predicted = _predict_one(code, test_example.input, run_candidate=run_candidate, timeout_s=timeout_s)\n            if predicted is None:\n                continue\n            key = _grid_key(predicted)\n            program_votes[key] += 1\n            grids_by_key[key] = predicted\n\n        ranked = program_votes.most_common(2)\n        if not ranked:\n            raise SolverError(f"{task.id}: no verified program produced output for a test item")\n        attempt_1 = grids_by_key[ranked[0][0]]\n        attempt_2 = grids_by_key[ranked[1][0]] if len(ranked) > 1 else attempt_1\n        attempts.append((attempt_1, attempt_2))\n    return attempts\n', 'agentic_repl/dsl/__init__.py': '"""DSL surface exposed to the code-generating LLM.\n\nRe-exports mythos\'s verified grid-primitive modules (mythos.objects,\nmythos.symmetry, mythos.augment) as a curated, flat namespace: these\nfunctions already carry docstrings written for a human reader (see each\nmodule\'s docstring in src/mythos/), so this module doesn\'t re-document them\n-- it just picks the stable subset worth exposing and collects them into\nDSL_FUNCTIONS/DSL_CLASSES so agentic_repl.dsl.catalog can generate the\nLLM-facing reference directly from the real callables (never a hand-copied,\ndriftable description) and agentic_repl.repl can build the restricted exec()\nnamespace from the same list.\n"""\n\nfrom __future__ import annotations\n\nfrom mythos.arc import copy_grid, grid_shape\nfrom mythos.augment import NUM_TRANSFORMS, forward_transform, inverse_transform, transform_name\nfrom mythos.objects import (\n    ArcObject,\n    crop_to_object,\n    d4_signature_variants,\n    dominant_grid_color,\n    segment_objects,\n)\nfrom mythos.symmetry import (\n    crop,\n    find_occlusion_color_candidates,\n    hole_bbox,\n    hole_cells_for_color,\n    repair_grid,\n    verify_symmetry,\n)\n\nDSL_FUNCTIONS = (\n    segment_objects,\n    crop_to_object,\n    dominant_grid_color,\n    d4_signature_variants,\n    find_occlusion_color_candidates,\n    repair_grid,\n    hole_bbox,\n    crop,\n    hole_cells_for_color,\n    verify_symmetry,\n    forward_transform,\n    inverse_transform,\n    transform_name,\n    grid_shape,\n    copy_grid,\n)\n\nDSL_CLASSES = (ArcObject,)\n\nPUBLIC_NAMESPACE: dict[str, object] = {fn.__name__: fn for fn in DSL_FUNCTIONS}\nPUBLIC_NAMESPACE.update({cls.__name__: cls for cls in DSL_CLASSES})\nPUBLIC_NAMESPACE["NUM_TRANSFORMS"] = NUM_TRANSFORMS\n\n__all__ = [\n    "DSL_FUNCTIONS",\n    "DSL_CLASSES",\n    "PUBLIC_NAMESPACE",\n    *PUBLIC_NAMESPACE.keys(),\n]\n', 'agentic_repl/dsl/catalog.py': '"""Builds the LLM-facing DSL reference text by introspecting the real callables.\n\nGenerating this from inspect.signature()/inspect.getdoc() over\nagentic_repl.dsl.DSL_FUNCTIONS/DSL_CLASSES (rather than hand-writing a\nseparate description) means the prompt catalog can never drift from what\nthe sandboxed exec() namespace in agentic_repl.repl actually exposes.\n"""\n\nfrom __future__ import annotations\n\nimport inspect\n\nfrom agentic_repl.dsl import DSL_CLASSES, DSL_FUNCTIONS\n\n\ndef _first_line(doc: str | None) -> str:\n    if not doc:\n        return ""\n    return doc.strip().splitlines()[0].strip()\n\n\ndef _function_entry(fn: object) -> str:\n    signature = inspect.signature(fn)  # type: ignore[arg-type]\n    summary = _first_line(inspect.getdoc(fn))\n    header = f"{fn.__name__}{signature}"  # type: ignore[attr-defined]\n    return f"{header}\\n    {summary}" if summary else header\n\n\ndef _class_entry(cls: type) -> str:\n    lines = [f"class {cls.__name__}:"]\n    class_summary = _first_line(inspect.getdoc(cls))\n    if class_summary:\n        lines.append(f"    {class_summary}")\n    for name, member in inspect.getmembers(cls):\n        if name.startswith("_") or not isinstance(member, property):\n            continue\n        summary = _first_line(inspect.getdoc(member))\n        lines.append(f"    .{name}" + (f"  -- {summary}" if summary else ""))\n    return "\\n".join(lines)\n\n\ndef build_catalog_text() -> str:\n    """Return the DSL reference text to inject into the code-generation prompt."""\n\n    sections = ["Available DSL primitives (already imported into scope, call directly):", ""]\n    for cls in DSL_CLASSES:\n        sections.append(_class_entry(cls))\n        sections.append("")\n    for fn in DSL_FUNCTIONS:\n        sections.append(_function_entry(fn))\n        sections.append("")\n    return "\\n".join(sections).rstrip() + "\\n"\n', 'agentic_repl/llm/__init__.py': '"""LLM client abstraction for agentic-REPL candidate generation."""\n', 'agentic_repl/llm/client.py': '"""LLM client abstraction: the solver depends only on this Protocol, never on\nwhich backend is actually generating code.\n\nFakeLLMClient is a deterministic stand-in for tests (no network, no GPU, no\nreal model -- see tests/test_agentic_repl.py). LlamaCppClient is the real\nbackend: a local quantized code-LLM served via llama-cpp-python, loaded from\na GGUF file staged as a Kaggle Dataset for internet-disabled competition\nreruns (see agentic_repl/models/README.md).\n\nA transformers-native GGUF loader (AutoModelForCausalLM(..., gguf_file=...))\nwas considered first since transformers ships on Kaggle with no install\nneeded at all -- but transformers explicitly does not support the qwen3moe\narchitecture for GGUF loading yet (confirmed: "GGUF model with architecture\nqwen3moe is not supported yet"), so it\'s not viable for this exact model.\nllama-cpp-python needs an offline-staged wheel instead: Kaggle\'s L4 sessions\nhard-enforce no internet regardless of kernel-metadata.json\'s enable_internet\nsetting (confirmed directly -- a live pip install attempt failed with DNS\nresolution errors even with enable_internet=true), and building from source\nrisks the same multi-hour CUDA compile this repo already hit once with\nflash-attn. The fix: a prebuilt CUDA wheel (llama-cpp-python\'s newer releases\nare `py3-none-manylinux_*` -- pure ctypes bindings around a compiled .so, no\nPython-version-specific extension, and CUDA is runtime-backward-compatible)\nstaged as a small Kaggle Dataset and installed offline via `pip install\n--no-index --find-links`.\n"""\n\nfrom __future__ import annotations\n\nimport os\nfrom typing import Protocol\n\n\nclass LLMClient(Protocol):\n    def generate(self, prompt: str, *, n: int, temperature: float = 0.7) -> list[str]:\n        """Return up to n candidate completions for prompt."""\n\n\nclass FakeLLMClient:\n    """Deterministic stand-in: cycles through pre-scripted responses.\n\n    Each generate() call advances a cursor through `responses`, so a\n    multi-round refinement loop sees later scripted responses instead of the\n    first one repeating forever -- lets tests script "wrong code, then the\n    fix" without depending on any real model.\n    """\n\n    def __init__(self, responses: list[str]) -> None:\n        if not responses:\n            raise ValueError("FakeLLMClient needs at least one scripted response")\n        self._responses = list(responses)\n        self._cursor = 0\n\n    def generate(self, prompt: str, *, n: int, temperature: float = 0.7) -> list[str]:\n        del prompt, temperature  # unused: the fake ignores prompt content by design\n        batch = []\n        for _ in range(n):\n            batch.append(self._responses[self._cursor % len(self._responses)])\n            self._cursor += 1\n        return batch\n\n\nclass LlamaCppClient:\n    """Real backend: a local GGUF code model served via llama-cpp-python.\n\n    Model path resolves from the `model_path` argument, else the\n    MYTHOS_AGENTIC_MODEL_PATH env var (mirrors this repo\'s existing\n    env-var-driven checkpoint loading, e.g. HRM_CHECKPOINT_PATH), so the\n    same code runs against a local file during development and a mounted\n    Kaggle Dataset during a real submission. Importing llama_cpp is deferred\n    to __init__ so importing this module -- or selecting any other solver\n    via mythos.solvers.factory -- never requires the dependency installed.\n    """\n\n    def __init__(\n        self,\n        model_path: str | None = None,\n        *,\n        n_gpu_layers: int = -1,\n        n_ctx: int = 32768,\n    ) -> None:\n        try:\n            from llama_cpp import Llama\n        except ImportError as exc:\n            raise RuntimeError(\n                "LlamaCppClient requires the \'llama-cpp-python\' package: "\n                "pip install -e \'.[agentic]\'"\n            ) from exc\n\n        resolved_path = model_path or os.environ.get("MYTHOS_AGENTIC_MODEL_PATH")\n        if not resolved_path:\n            raise RuntimeError(\n                "no model_path given and MYTHOS_AGENTIC_MODEL_PATH is not set "\n                "(see agentic_repl/models/README.md)"\n            )\n        self._llama = Llama(\n            model_path=resolved_path,\n            n_gpu_layers=n_gpu_layers,\n            n_ctx=n_ctx,\n            verbose=False,\n        )\n\n    def generate(self, prompt: str, *, n: int, temperature: float = 0.7) -> list[str]:\n        # create_chat_completion (not create_completion) so llama.cpp applies\n        # the GGUF\'s embedded chat template -- Qwen3-Coder-Instruct was\n        # fine-tuned to expect its chat format, and a first real run against\n        # it via raw create_completion verified 0/30 candidates on real\n        # ARC-AGI-2 training tasks, consistent with the model not being\n        # prompted the way it was tuned to respond to.\n        completions = []\n        for _ in range(n):\n            result = self._llama.create_chat_completion(\n                messages=[{"role": "user", "content": prompt}],\n                max_tokens=1024,\n                temperature=temperature,\n            )\n            completions.append(result["choices"][0]["message"]["content"] or "")\n        return completions\n', 'agentic_repl/llm/prompts.py': '"""Prompt templates for the agentic-REPL code-generation loop."""\n\nfrom __future__ import annotations\n\nfrom mythos.arc import ArcTask, Grid\n\n\ndef render_grid(grid: Grid) -> str:\n    return "\\n".join(" ".join(str(cell) for cell in row) for row in grid)\n\n\n# Concrete usage examples for the DSL primitives, not just their signatures.\n# Added after a real benchmark run (v56, 100 ARC-AGI-2 training tasks)\n# showed the model reaching only for generic per-pixel loops even when a\n# task was exactly the kind mythos.solvers.symbolic\'s hand-written transform\n# finders already solve (object selection, symmetry repair) -- the catalog\'s\n# signatures alone weren\'t enough to make the model reach for them instead.\n_DSL_USAGE_EXAMPLES = """\\\nExample uses of the DSL primitives above (adapt the pattern, don\'t copy verbatim):\n\n# Pattern: output is one selected object, cropped to its bounding box.\ndef solve(grid):\n    objects = segment_objects(grid, background=0, connectivity=4, univalued=True)\n    largest = max(objects, key=lambda obj: obj.size)\n    return crop_to_object(grid, largest, background=0)\n\n# Pattern: a rectangular region was painted over with one color; recover it\n# from the grid\'s own mirror/rotational/periodic symmetry.\ndef solve(grid):\n    occlusion_color = find_occlusion_color_candidates(grid)[0]\n    holes = hole_cells_for_color(grid, occlusion_color)\n    repaired = repair_grid(grid, holes)\n    top, left, height, width = hole_bbox(holes)\n    return crop(repaired, top, left, height, width)\n"""\n\n\ndef build_dsl_reference(dsl_catalog: str) -> str:\n    return f"{dsl_catalog}\\n{_DSL_USAGE_EXAMPLES}"\n\n\ndef build_initial_prompt(task: ArcTask, dsl_catalog: str) -> str:\n    examples = []\n    for index, example in enumerate(task.train):\n        assert example.output is not None  # train examples always have outputs\n        examples.append(\n            f"Example {index + 1} input:\\n{render_grid(example.input)}\\n\\n"\n            f"Example {index + 1} output:\\n{render_grid(example.output)}\\n"\n        )\n    examples_text = "\\n".join(examples)\n    return (\n        "You are solving an ARC-AGI grid transformation puzzle. Write a Python "\n        "function `solve(grid)` that takes a grid (a list of lists of ints, 0-9) "\n        "and returns the transformed grid, reproducing the rule shown by these "\n        "training examples exactly. Prefer the DSL primitives below over "\n        "hand-written pixel loops when a task looks like object selection, "\n        "cropping, or symmetry repair -- they handle edge cases a from-scratch "\n        "loop usually misses.\\n\\n"\n        f"{build_dsl_reference(dsl_catalog)}\\n"\n        f"{examples_text}\\n"\n        "Respond with ONLY a Python code block defining `solve(grid)`. Do not "\n        "import anything -- the DSL primitives above are already in scope.\\n"\n    )\n\n\ndef build_refinement_prompt(\n    task: ArcTask, dsl_catalog: str, previous_code: str, failure_report: str\n) -> str:\n    """Re-includes the full task (not just the failure text).\n\n    Each LLMClient.generate() call is a fresh, stateless completion, not a\n    multi-turn conversation -- without re-rendering the actual train\n    examples here, the model has no way to see what it got wrong beyond the\n    failure summary, and can\'t "remember" example dimensions/content from\n    the now-discarded initial-prompt turn. Confirmed as a real bug via a\n    real benchmark run: refinement rounds kept regenerating the same class\n    of error (e.g. wrong output shape) instead of converging.\n    """\n\n    examples = []\n    for index, example in enumerate(task.train):\n        assert example.output is not None\n        examples.append(\n            f"Example {index + 1} input:\\n{render_grid(example.input)}\\n\\n"\n            f"Example {index + 1} output:\\n{render_grid(example.output)}\\n"\n        )\n    examples_text = "\\n".join(examples)\n    return (\n        "You are solving an ARC-AGI grid transformation puzzle. Your previous "\n        "solve(grid) candidate did not reproduce every training example exactly. "\n        "Fix it -- consider whether one of the DSL primitives below handles this "\n        "more robustly than a hand-written pixel loop.\\n\\n"\n        f"{build_dsl_reference(dsl_catalog)}\\n"\n        f"{examples_text}\\n"\n        f"Previous code:\\n```python\\n{previous_code}\\n```\\n\\n"\n        f"Failure report:\\n{failure_report}\\n\\n"\n        "Respond with ONLY a corrected Python code block defining `solve(grid)`.\\n"\n    )\n\n\ndef extract_code_block(completion: str) -> str:\n    """Pull the first ```-fenced code block out of a completion, else return as-is."""\n\n    if "```" not in completion:\n        return completion.strip()\n    fence_parts = completion.split("```")\n    if len(fence_parts) < 2:\n        return completion.strip()\n    block = fence_parts[1]\n    if block.lstrip().lower().startswith("python"):\n        block = block.lstrip()[len("python"):]\n    return block.strip()\n', 'agentic_repl/repl.py': '"""Sandboxed execution of LLM-generated solve(grid) candidates.\n\nRuns candidates in a subprocess (multiprocessing, spawn context) rather than\nusing signal.alarm: signal.alarm doesn\'t exist on Windows (this repo is\ndeveloped on Windows, deployed on Kaggle/Linux), and a subprocess is also the\nonly way to actually kill code that\'s genuinely hung.\n\nThe worker process is persistent, not spawned fresh per call: a fresh spawn\nhas to re-import this whole module (and mythos/agentic_repl\'s dependency\nchain) before it can run anything, and that reimport cost -- not the\ncandidate code itself -- can exceed a short per-candidate timeout on its own\n(measured in practice: ~2s of pure process-startup overhead when invoked\nfrom a plain script, versus negligible overhead from inside an\nalready-running test process). Reusing one worker across many calls pays\nthat cost once; a fresh `solve` namespace is still built per call (see\n_build_namespace), so candidates never see state left behind by a previous\none. If a candidate hangs, its worker is killed and a replacement is spawned\nlazily on the next call -- this trades a small amount of inter-candidate\nprocess isolation (they share one OS process across a run, not one each) for\nthroughput that\'s necessary given real per-task wall-clock budgets; this is\nfine here since candidates are LLM-generated ARC solve() attempts, not\nadversarial input.\n\nNot thread-safe: callers (solver.py, augment_vote.py) invoke run_candidate\nserially, one call at a time.\n\nexec() runs against a restricted namespace: a small builtins allowlist plus\nthe DSL functions from agentic_repl.dsl, and nothing else -- no imports, no\nfile/network/process access.\n"""\n\nfrom __future__ import annotations\n\nimport atexit\nimport builtins\nfrom dataclasses import dataclass\nimport multiprocessing as mp\nimport queue as queue_module\nimport time\nimport traceback\n\nfrom mythos.arc import ArcValidationError, Grid, validate_grid\n\nfrom agentic_repl.dsl import PUBLIC_NAMESPACE\n\nDEFAULT_TIMEOUT_SECONDS = 2.0\n_WORKER_STARTUP_TIMEOUT_SECONDS = 60.0\n_WORKER_READY = "__agentic_repl_worker_ready__"\n\n_ALLOWED_BUILTIN_NAMES = (\n    "abs", "all", "any", "bool", "dict", "enumerate", "filter", "float",\n    "frozenset", "int", "isinstance", "iter", "len", "list", "map", "max",\n    "min", "next", "range", "reversed", "set", "slice", "sorted", "str",\n    "sum", "tuple", "type", "zip",\n    "Exception", "IndexError", "KeyError", "StopIteration", "TypeError",\n    "ValueError", "ZeroDivisionError",\n)\n\n\n@dataclass\nclass ExecutionResult:\n    ok: bool\n    output: Grid | None\n    error: str | None\n    elapsed_seconds: float\n\n\ndef _build_namespace() -> dict[str, object]:\n    safe_builtins = {name: getattr(builtins, name) for name in _ALLOWED_BUILTIN_NAMES}\n    namespace: dict[str, object] = {"__builtins__": safe_builtins}\n    namespace.update(PUBLIC_NAMESPACE)\n    return namespace\n\n\ndef _execute(code: str, input_grid: Grid) -> tuple[bool, Grid | None, str | None]:\n    """Runs inside the worker process. Never raises -- always returns a report."""\n\n    namespace = _build_namespace()\n    try:\n        exec(compile(code, "<agentic_repl_candidate>", "exec"), namespace)\n    except Exception:  # noqa: BLE001 - report every failure mode, don\'t crash the worker\n        return False, None, f"candidate code failed to compile/exec:\\n{traceback.format_exc()}"\n\n    solve_fn = namespace.get("solve")\n    if not callable(solve_fn):\n        return False, None, "candidate code must define a top-level `solve(grid)` function"\n\n    try:\n        output = solve_fn([row[:] for row in input_grid])\n    except Exception:  # noqa: BLE001\n        return False, None, f"solve(grid) raised:\\n{traceback.format_exc()}"\n\n    try:\n        validated = validate_grid(output, field="solve() output")\n    except ArcValidationError as exc:\n        return False, None, f"solve(grid) returned an invalid grid: {exc}"\n\n    return True, validated, None\n\n\ndef _worker_loop(task_queue: "mp.Queue[object]", result_queue: "mp.Queue[object]") -> None:\n    result_queue.put(_WORKER_READY)\n    while True:\n        message = task_queue.get()\n        if message is None:  # shutdown sentinel\n            return\n        code, input_grid = message\n        result_queue.put(_execute(code, input_grid))\n\n\n_worker_process: "mp.process.BaseProcess | None" = None\n_worker_task_queue: "mp.Queue[object] | None" = None\n_worker_result_queue: "mp.Queue[object] | None" = None\n\n\ndef _spawn_worker() -> tuple["mp.process.BaseProcess", "mp.Queue[object]", "mp.Queue[object]"]:\n    ctx = mp.get_context("spawn")\n    task_queue: "mp.Queue[object]" = ctx.Queue()\n    result_queue: "mp.Queue[object]" = ctx.Queue()\n    process = ctx.Process(target=_worker_loop, args=(task_queue, result_queue), daemon=True)\n    process.start()\n    try:\n        ready = result_queue.get(timeout=_WORKER_STARTUP_TIMEOUT_SECONDS)\n    except queue_module.Empty as exc:\n        process.kill()\n        raise RuntimeError(\n            f"agentic_repl worker failed to start within {_WORKER_STARTUP_TIMEOUT_SECONDS}s"\n        ) from exc\n    if ready != _WORKER_READY:\n        process.kill()\n        raise RuntimeError("agentic_repl worker sent an unexpected startup message")\n    return process, task_queue, result_queue\n\n\ndef _kill_worker() -> None:\n    global _worker_process, _worker_task_queue, _worker_result_queue\n    if _worker_process is not None and _worker_process.is_alive():\n        _worker_process.terminate()\n        _worker_process.join(1.0)\n        if _worker_process.is_alive():\n            _worker_process.kill()\n            _worker_process.join()\n    _worker_process = None\n    _worker_task_queue = None\n    _worker_result_queue = None\n\n\natexit.register(_kill_worker)\n\n\ndef run_candidate(\n    code: str,\n    input_grid: Grid,\n    *,\n    timeout_s: float = DEFAULT_TIMEOUT_SECONDS,\n) -> ExecutionResult:\n    """Execute `code`\'s solve(grid) against input_grid under a strict timeout.\n\n    Always returns an ExecutionResult -- never raises -- so callers (the\n    solver\'s train-pair verification loop, the augmentation/voting stage)\n    can treat every candidate uniformly, whether it\'s a syntax error, a\n    runtime crash, a shape mismatch, or a genuine timeout.\n    """\n\n    global _worker_process, _worker_task_queue, _worker_result_queue\n    if _worker_process is None or not _worker_process.is_alive():\n        _worker_process, _worker_task_queue, _worker_result_queue = _spawn_worker()\n\n    start = time.perf_counter()\n    _worker_task_queue.put((code, input_grid))\n    try:\n        ok, output, error = _worker_result_queue.get(timeout=timeout_s)\n    except queue_module.Empty:\n        elapsed = time.perf_counter() - start\n        _kill_worker()  # the worker may be genuinely hung; discard it, respawn lazily next call\n        return ExecutionResult(\n            ok=False, output=None, error=f"execution exceeded {timeout_s}s timeout", elapsed_seconds=elapsed\n        )\n\n    elapsed = time.perf_counter() - start\n    return ExecutionResult(ok=ok, output=output, error=error, elapsed_seconds=elapsed)\n', 'agentic_repl/solver.py': '"""Agentic program-synthesis solver: LLM-generated solve(grid) candidates,\nverified against every train pair in a sandboxed REPL, refined on failure,\nand voted across augmented views before producing a two-attempt prediction.\n\nMirrors mythos.solvers.symbolic.SymbolicSolver\'s "prove correctness on train\nor don\'t fire" contract (src/mythos/solvers/symbolic.py): this solver raises\nSolverError rather than ever guessing, so it\'s safe to slot into the same\nsolver-factory fallback chain as the existing solvers.\n"""\n\nfrom __future__ import annotations\n\nimport os\n\nfrom mythos.arc import ArcTask, Grid, grid_equal\nfrom mythos.solvers.base import SolverError, make_prediction\nfrom mythos.submission import Prediction\n\nfrom agentic_repl.augment_vote import vote_predictions\nfrom agentic_repl.dsl.catalog import build_catalog_text\nfrom agentic_repl.llm.client import LLMClient\nfrom agentic_repl.llm.prompts import build_initial_prompt, build_refinement_prompt, extract_code_block\nfrom agentic_repl.repl import ExecutionResult, run_candidate\n\nDEFAULT_NUM_CANDIDATES = 4\nDEFAULT_REFINEMENT_ROUNDS = 2\nDEFAULT_TIMEOUT_SECONDS = 2.0\n\n\ndef _debug_enabled() -> bool:\n    return os.environ.get("MYTHOS_AGENTIC_DEBUG") == "1"\n\n\ndef _debug_log(label: str, text: str, *, limit: int = 800) -> None:\n    if not _debug_enabled():\n        return\n    truncated = text if len(text) <= limit else text[:limit] + f"... [{len(text) - limit} more chars]"\n    print(f"[agentic_repl debug] {label}:\\n{truncated}")\n\n\ndef _failure_report(train_index: int, result: ExecutionResult, expected: Grid) -> str:\n    if not result.ok or result.output is None:\n        return f"Example {train_index + 1}: {result.error}"\n    got_rows, got_cols = len(result.output), len(result.output[0])\n    want_rows, want_cols = len(expected), len(expected[0])\n    if (got_rows, got_cols) != (want_rows, want_cols):\n        return (\n            f"Example {train_index + 1}: got a {got_rows}x{got_cols} grid, "\n            f"expected {want_rows}x{want_cols}."\n        )\n    return f"Example {train_index + 1}: output shape matched but cell values did not."\n\n\ndef _verify_on_train(code: str, task: ArcTask, *, timeout_s: float) -> tuple[bool, str | None]:\n    for index, example in enumerate(task.train):\n        assert example.output is not None  # train examples always have outputs\n        result = run_candidate(code, example.input, timeout_s=timeout_s)\n        if not result.ok or result.output is None or not grid_equal(result.output, example.output):\n            return False, _failure_report(index, result, example.output)\n    return True, None\n\n\nclass AgenticReplSolver:\n    def __init__(\n        self,\n        llm_client: LLMClient,\n        *,\n        num_candidates: int = DEFAULT_NUM_CANDIDATES,\n        refinement_rounds: int = DEFAULT_REFINEMENT_ROUNDS,\n        timeout_s: float = DEFAULT_TIMEOUT_SECONDS,\n    ) -> None:\n        self._llm_client = llm_client\n        self._num_candidates = num_candidates\n        self._refinement_rounds = refinement_rounds\n        self._timeout_s = timeout_s\n        self._dsl_catalog = build_catalog_text()\n\n    def solve(self, task: ArcTask) -> Prediction:\n        verified_codes = self._search_verified_programs(task)\n        if not verified_codes:\n            raise SolverError(f"{task.id}: no agentic-REPL candidate verified against all train pairs")\n\n        attempts = vote_predictions(\n            verified_codes, task, run_candidate=run_candidate, timeout_s=self._timeout_s\n        )\n        return make_prediction(task, attempts)\n\n    def _search_verified_programs(self, task: ArcTask) -> list[str]:\n        prompt = build_initial_prompt(task, self._dsl_catalog)\n        _debug_log(f"{task.id} initial prompt", prompt, limit=1500)\n        raw_completions = self._llm_client.generate(prompt, n=self._num_candidates)\n        candidates = []\n        for index, completion in enumerate(raw_completions):\n            _debug_log(f"{task.id} candidate {index} raw completion", completion)\n            code = extract_code_block(completion)\n            _debug_log(f"{task.id} candidate {index} extracted code", code)\n            candidates.append(code)\n\n        verified: list[str] = []\n        for code in candidates:\n            verified_code = self._refine_until_verified(code, task)\n            if verified_code is not None:\n                verified.append(verified_code)\n        return verified\n\n    def _refine_until_verified(self, code: str, task: ArcTask) -> str | None:\n        current = code\n        attempts_left = self._refinement_rounds + 1\n        while attempts_left > 0:\n            ok, failure = _verify_on_train(current, task, timeout_s=self._timeout_s)\n            _debug_log(f"{task.id} verify result", f"ok={ok} failure={failure}")\n            if ok:\n                return current\n            attempts_left -= 1\n            if attempts_left == 0:\n                return None\n            refinement_prompt = build_refinement_prompt(\n                task, self._dsl_catalog, current, failure or "unknown failure"\n            )\n            completions = self._llm_client.generate(refinement_prompt, n=1)\n            if not completions:\n                return None\n            _debug_log(f"{task.id} refinement raw completion", completions[0])\n            current = extract_code_block(completions[0])\n        return None\n'}

EMBED_ROOT = Path(os.environ.get('MYTHOS_EMBED_ROOT', '/kaggle/working/project_mythos_embedded'))
if not EMBED_ROOT.parent.exists():
    EMBED_ROOT = Path.cwd() / 'project_mythos_embedded'

for relative_path, content in EMBEDDED_FILES.items():
    path = EMBED_ROOT / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding='utf-8')

SRC_DIR = EMBED_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# Vendored third-party code (e.g. CompressARC) keeps its own top-level
# import style (`import arc_compressor`, not a package), so its directory
# goes on sys.path directly rather than under src/.
COMPRESS_ARC_DIR = EMBED_ROOT / 'third_party' / 'compress_arc'
if COMPRESS_ARC_DIR.is_dir() and str(COMPRESS_ARC_DIR) not in sys.path:
    sys.path.insert(0, str(COMPRESS_ARC_DIR))

# agentic_repl is embedded as its own top-level package (agentic_repl/, a
# sibling of src/), so EMBED_ROOT itself -- not EMBED_ROOT/src -- needs to be
# on sys.path for `import agentic_repl` to resolve.
if (EMBED_ROOT / 'agentic_repl').is_dir() and str(EMBED_ROOT) not in sys.path:
    sys.path.insert(0, str(EMBED_ROOT))

# The agentic-REPL code-LLM's weights are pre-staged as a Kaggle Dataset
# (see agentic_repl/models/README.md), never downloaded live -- internet is
# disabled on scored reruns. Autodiscover the mounted .gguf file the same
# way the HRM checkpoint path is set explicitly above, so LlamaCppClient
# just reads MYTHOS_AGENTIC_MODEL_PATH without any further wiring.
if 'MYTHOS_AGENTIC_MODEL_PATH' not in os.environ:
    _kaggle_input = Path('/kaggle/input')
    if _kaggle_input.is_dir():
        # Confirmed directly (not assumed): private dataset inputs mount at
        # /kaggle/input/datasets/<owner>/<slug>/, not the flatter
        # /kaggle/input/<slug>/ this repo's HRM checkpoint paths assume --
        # check both rather than trust either blindly.
        _gguf_candidates = sorted(_kaggle_input.glob('*/*.gguf')) or sorted(
            _kaggle_input.glob('datasets/*/*/*.gguf')
        )
        if _gguf_candidates:
            os.environ['MYTHOS_AGENTIC_MODEL_PATH'] = str(_gguf_candidates[0])

print('Embedded Mythos package written to:', SRC_DIR)
print('Embedded files:', len(EMBEDDED_FILES))


## 2. Configuration

In [ ]:
from pathlib import Path
import json
import os
import time

DATA_DIR = Path('/kaggle/input/competitions/arc-prize-2026-arc-agi-2')
# ARCHITECTURE pass: hyperparameter tuning alone plateaued (v29-v37: stable,
# decreasing loss but 0 exact matches regardless of steps/rank/lr). Testing two
# real architectural changes together -- Genie background-consistency loss (was
# implemented in mythos.losses but never wired into any training loop) and wider
# LoRA target modules (MLP layers, not just attention) -- against the evaluation
# split's known solutions before spending the daily submission quota again.
SPLIT = 'test'  # 'evaluation' was used for diagnostic runs against known solutions (v43-v49); 'test' is the real submission split
# agentic_repl accuracy validation: point at the staged, solution-known public
# ARC-AGI-2 dataset instead of the competition's own split. None = use DATA_DIR/SPLIT above.
BENCHMARK_ARC_AGI_2_SPLIT = 'training'  # 'training' (1000 tasks) or 'evaluation' (120 tasks) or None
BENCHMARK_DATA_DIR = Path('/kaggle/input/agentic-repl-arc-agi-2-data')
os.environ.setdefault('MYTHOS_TTT_STEPS', '100')  # validated in v49: 2-view ensemble at steps=100 each beat steps=200 single-view at roughly matched total TTT compute
os.environ.setdefault('MYTHOS_TTT_ENSEMBLE', '0,4')  # v49-validated: identity + mirror_horizontal views, voted -- real (modest) accuracy gain over single-view
os.environ.setdefault('MYTHOS_TTT_NUM_AUG', '8')
os.environ.setdefault('MYTHOS_TTT_RANK', '16')
os.environ.setdefault('MYTHOS_TTT_GENIE_WEIGHT', '0.01')
os.environ.setdefault('MYTHOS_TTT_LR', '1e-4')
os.environ.setdefault('MYTHOS_TTT_BATCH_SIZE', '2')
SOLVER_NAME = 'agentic_repl'  # 'pipeline', 'baseline', 'fixture', 'hrm', or 'agentic_repl'
# BENCHMARK: agentic_repl (composed with symbolic-first, see section 6) against
# the real, solution-known ARC-AGI-2 training split. Progress log:
#   v53 (tasks 0-30):   0/30 exact  -- raw create_completion() vs an Instruct
#                       model (fixed: create_chat_completion()).
#   v55 (tasks 0-50):   1/50 exact  -- post chat-completion + refinement-
#                       prompt-context fixes; found a context-window overflow
#                       (fixed: n_ctx 8192->32768).
#   v56 (tasks 0-100):  3/100 exact -- all attributable to the symbolic solver;
#                       agentic_repl added zero unique coverage on this slice.
#   v57 (tasks 0-100):   5/100 exact -- symbolic still 3, agentic_repl added 2
#                        genuinely new solves (08ed6ac7, 1d0a4b61) after nudging
#                        prompts to prefer DSL primitives over pixel loops.
#   v58 (tasks 100-300): 13/200 exact -- symbolic 5, agentic_repl added 8 new
#                        unique solves. Cumulative (tasks 0-300): 18/300 = 6.0%,
#                        more than double the symbolic solver's own documented
#                        25/1000 (2.5%) full-training-set rate.
# Continuing to cover fresh, non-overlapping slices -- see MYTHOS_TASK_OFFSET
# in section 4 (MYTHOS_MAX_TASKS alone always truncates to the first N).
os.environ.setdefault('MYTHOS_TASK_OFFSET', '300')
os.environ.setdefault('MYTHOS_MAX_TASKS', '200')
os.environ.setdefault('MYTHOS_AGENTIC_DEBUG', '0')
MODEL_MODE = 'fallback'  # 'fallback' or 'strict' (irrelevant for SOLVER_NAME='hrm', which bypasses ModelRegistry)
# Kaggle competition reruns are internet-disabled, so the HRM repo + checkpoint
# are pre-staged as a Kaggle Dataset (ankitdash24/hrm-arc2-checkpoint) instead of
# downloaded live -- verified working end-to-end first with AUTO_DOWNLOAD+internet
# in a dev run, then pinned here for the actually-submittable configuration.
AUTO_DOWNLOAD_GIT_CODE = False
AUTO_DOWNLOAD_HF_MODELS = False
AUTO_DOWNLOAD_DIRECT_CHECKPOINTS = False
AUTO_DISCOVER_MODELS = True
os.environ.setdefault('HRM_REPO_DIR', '/kaggle/input/hrm-arc2-checkpoint/hrm-repo')
os.environ.setdefault('HRM_CHECKPOINT_PATH', '/kaggle/input/hrm-arc2-checkpoint/hrm-checkpoint/checkpoint')
OUTPUT_PATH = Path('/kaggle/working/submission.json')
RUN_HRM_SMOKE = False

# Training/checkpoint-producing stages. Keep all False for final rerun unless needed.
RUN_TRAINING_STAGES = False
TRAIN_JEPA_PROJECTION = False
TRAIN_WORLD_MODEL = False
RUN_TTT_SMOKE = False
ENABLE_REAL_HRM_INFERENCE = True
ENABLE_REAL_JEPA = False
ENABLE_HRM_TEXT = False
ENABLE_TTT_IN_PIPELINE = True  # drives MYTHOS_ENABLE_TTT for SOLVER_NAME='hrm' too, not just the pipeline solver
CHECKPOINT_DIR = Path('/kaggle/working/mythos_checkpoints')
IJEPA_PROJECTION_OUTPUT = CHECKPOINT_DIR / 'ijepa_projection.pt'
WORLD_MODEL_OUTPUT = CHECKPOINT_DIR / 'world_model.pt'
TTT_LORA_OUTPUT = CHECKPOINT_DIR / 'ttt_lora_smoke.pt'
JEPA_PROJECTION_STEPS = 200
WORLD_MODEL_STEPS = 300
TTT_SMOKE_STEPS = 50

os.environ['MYTHOS_ENABLE_REAL_HRM'] = '1' if ENABLE_REAL_HRM_INFERENCE else '0'
os.environ['MYTHOS_ENABLE_REAL_JEPA'] = '1' if ENABLE_REAL_JEPA else '0'
os.environ['MYTHOS_ENABLE_HRM_TEXT'] = '1' if ENABLE_HRM_TEXT else '0'
os.environ['MYTHOS_ENABLE_TTT'] = '1' if ENABLE_TTT_IN_PIPELINE else '0'
# HRM's own eval path logs to Weights & Biases; without this it can block on an
# interactive API-key prompt in a non-interactive kernel and hang out the session.
os.environ.setdefault('WANDB_MODE', 'offline')
# HRM's public checkpoint was trained for 8-GPU distributed batches; keep eval batches
# small since this pipeline calls HRM's eval loop with world_size=1 on Kaggle's GPU(s).
os.environ.setdefault('HRM_GLOBAL_BATCH_SIZE', '32')

# Verified public defaults looked up from official sources.
os.environ.setdefault('HRM_GIT_REPO_URL', 'https://github.com/sapientinc/HRM.git')
os.environ.setdefault('HRM_HF_REPO_ID', 'sapientinc/HRM-checkpoint-ARC-2')
os.environ.setdefault('HRM_HF_CHECKPOINT_GLOB', 'checkpoint')
os.environ.setdefault('IJEPA_HF_REPO_ID', 'facebook/ijepa_vith14_1k')
os.environ.setdefault('IJEPA_HF_CHECKPOINT_GLOB', 'model.safetensors')
os.environ.setdefault('HRM_TEXT_HF_REPO_ID', 'sapientinc/HRM-Text-1B')
os.environ.setdefault('HRM_TEXT_HF_CHECKPOINT_GLOB', 'model.safetensors')

# Components intentionally left unset because no verified public checkpoint ID was found.
# Train/fine-tune these and add your own dataset/HF IDs when available:
# - IJEPA_PROJECTION_HF_REPO_ID / IJEPA_PROJECTION_CHECKPOINT_PATH
# - WORLD_MODEL_HF_REPO_ID / WORLD_MODEL_CHECKPOINT_PATH
# - TTT_LORA_HF_REPO_ID / TTT_LORA_CHECKPOINT_PATH

# For real transformers I-JEPA, stage the Hugging Face snapshot as a Kaggle Dataset
# and set IJEPA_CHECKPOINT_PATH to any file inside that snapshot, or let autodiscovery find it.

# Optional real-model inputs. Set these if you have additional public/private model repos.
# os.environ['IJEPA_HF_REPO_ID'] = '<org-or-user>/<ijepa-model-repo>'
# os.environ['IJEPA_HF_CHECKPOINT_GLOB'] = '*.pt'
# os.environ['IJEPA_PROJECTION_HF_REPO_ID'] = '<org-or-user>/<projection-repo>'
# os.environ['HRM_TEXT_HF_REPO_ID'] = '<org-or-user>/<hrm-text-model-repo>'
# os.environ['WORLD_MODEL_HF_REPO_ID'] = '<org-or-user>/<world-model-repo>'
# os.environ['TTT_LORA_HF_REPO_ID'] = '<org-or-user>/<lora-repo>'
# os.environ['HRM_HF_REPO_ID'] = '<org-or-user>/<hrm-model-repo>'
# os.environ['HRM_HF_CHECKPOINT_GLOB'] = '*.pt'

# Or set explicit Kaggle input paths when internet/download is unavailable.
# os.environ['IJEPA_CHECKPOINT_PATH'] = '/kaggle/input/<ijepa-hf-snapshot>/model.safetensors'
# os.environ['IJEPA_PROJECTION_CHECKPOINT_PATH'] = '/kaggle/input/<projection>/ijepa_projection.pt'
# os.environ['HRM_TEXT_REPO_DIR'] = '/kaggle/input/<hrm-text-code>/hrm-text'
# os.environ['HRM_TEXT_CHECKPOINT_PATH'] = '/kaggle/input/<hrm-text-checkpoint>/checkpoint.pt'
# os.environ['WORLD_MODEL_CHECKPOINT_PATH'] = '/kaggle/input/<world-model>/world_model.pt'
# os.environ['TTT_LORA_CHECKPOINT_PATH'] = '/kaggle/input/<lora>/lora.pt'
# os.environ['HRM_REPO_DIR'] = '/kaggle/input/<hrm-code>/HRM'
# os.environ['HRM_CHECKPOINT_PATH'] = '/kaggle/input/<hrm-checkpoint>/checkpoint.pt'

# HRM uses flash-attn in its attention path. For final offline reruns, pre-stage a wheel
# built against Kaggle's CUDA/PyTorch image instead of building from source at submission time.

print('DATA_DIR =', DATA_DIR)
print('SPLIT =', SPLIT)
print('SOLVER_NAME =', SOLVER_NAME)
print('MODEL_MODE =', MODEL_MODE)
print('AUTO_DOWNLOAD_GIT_CODE =', AUTO_DOWNLOAD_GIT_CODE)
print('AUTO_DOWNLOAD_HF_MODELS =', AUTO_DOWNLOAD_HF_MODELS)
print('AUTO_DOWNLOAD_DIRECT_CHECKPOINTS =', AUTO_DOWNLOAD_DIRECT_CHECKPOINTS)
print('AUTO_DISCOVER_MODELS =', AUTO_DISCOVER_MODELS)
print('OUTPUT_PATH =', OUTPUT_PATH)
print('RUN_TRAINING_STAGES =', RUN_TRAINING_STAGES)
print('ENABLE_REAL_HRM_INFERENCE =', ENABLE_REAL_HRM_INFERENCE)
print('ENABLE_REAL_JEPA =', ENABLE_REAL_JEPA)
print('ENABLE_HRM_TEXT =', ENABLE_HRM_TEXT)
print('ENABLE_TTT_IN_PIPELINE =', ENABLE_TTT_IN_PIPELINE)
print('MYTHOS_MAX_TASKS =', os.environ.get('MYTHOS_MAX_TASKS'))
print('MYTHOS_TASK_OFFSET =', os.environ.get('MYTHOS_TASK_OFFSET'))
print('MYTHOS_TTT_STEPS =', os.environ.get('MYTHOS_TTT_STEPS'))
print('MYTHOS_TTT_NUM_AUG =', os.environ.get('MYTHOS_TTT_NUM_AUG'))
print('MYTHOS_TTT_RANK =', os.environ.get('MYTHOS_TTT_RANK'))
print('MYTHOS_TTT_LR =', os.environ.get('MYTHOS_TTT_LR'))
print('MYTHOS_TTT_BATCH_SIZE =', os.environ.get('MYTHOS_TTT_BATCH_SIZE'))
print('MYTHOS_TTT_GENIE_WEIGHT =', os.environ.get('MYTHOS_TTT_GENIE_WEIGHT'))
print('MYTHOS_TTT_ENSEMBLE =', os.environ.get('MYTHOS_TTT_ENSEMBLE'))


## 3. Import Mythos Runtime

In [ ]:
import mythos
from mythos.arc import load_challenges
from mythos.kaggle_run import resolve_challenge_path, resolve_solution_path
from mythos.kaggle_models import (
    autodiscover_model_inputs,
    download_direct_checkpoint_inputs,
    download_git_code_repositories,
    download_huggingface_model_inputs,
)
from mythos.arc import load_solutions
from mythos.metrics import score_files, score_submission_data
from mythos.pipeline import PLAN_STAGE_ORDER
from mythos.solvers.factory import make_solver
from mythos.submission import load_submission, write_submission

from mythos.training import (
    JepaProjectionConfig,
    WorldModelConfig,
    run_ttt_lora_smoke,
    train_jepa_projection,
    train_world_model,
)

print('Imported mythos from:', mythos.__file__)
print('PLAN_STAGE_ORDER =', ' -> '.join(PLAN_STAGE_ORDER))


## 4. Load ARC Data

If BENCHMARK_ARC_AGI_2_SPLIT is set, load the staged public ARC-AGI-2 data (with known solutions) instead of the competition's own split -- for measuring real accuracy, not producing a submission.

In [ ]:
if BENCHMARK_ARC_AGI_2_SPLIT:
    _bench_dir = BENCHMARK_DATA_DIR
    if not _bench_dir.is_dir():
        _nested = sorted(Path('/kaggle/input').glob('datasets/*/agentic-repl-arc-agi-2-data'))
        if _nested:
            _bench_dir = _nested[0]
    challenge_path = _bench_dir / f'{BENCHMARK_ARC_AGI_2_SPLIT}_challenges.json'
    solution_path = _bench_dir / f'{BENCHMARK_ARC_AGI_2_SPLIT}_solutions.json'
else:
    challenge_path = resolve_challenge_path(DATA_DIR, SPLIT)
    solution_path = resolve_solution_path(DATA_DIR, SPLIT)
tasks = load_challenges(challenge_path)

_max_tasks = os.environ.get('MYTHOS_MAX_TASKS')
_task_offset = int(os.environ.get('MYTHOS_TASK_OFFSET', '0'))
if _max_tasks or _task_offset:
    _items = list(tasks.items())
    _end = _task_offset + int(_max_tasks) if _max_tasks else len(_items)
    tasks = dict(_items[_task_offset:_end])
    print(f'MYTHOS_TASK_OFFSET={_task_offset} MYTHOS_MAX_TASKS={_max_tasks}: '
          f'truncated to {len(tasks)} task(s) (a non-overlapping slice, not always '
          f'the first N -- lets successive benchmark runs cover fresh tasks instead '
          f'of redundantly re-processing the same early ones)')

train_examples = sum(len(task.train) for task in tasks.values())
test_items = sum(len(task.test) for task in tasks.values())

print('challenge_path =', challenge_path)
print('solution_path =', solution_path)
print('tasks =', len(tasks))
print('train_examples =', train_examples)
print('test_items =', test_items)


## 4b. Install agentic_repl Runtime Dependencies (offline wheel)

`AgenticReplSolver`'s real backend (`LlamaCppClient`) needs `llama-cpp-python`, which Kaggle doesn't preinstall. A live `pip install` doesn't work here at all: Kaggle's L4 sessions hard-enforce no internet regardless of kernel-metadata.json's enable_internet setting (confirmed directly -- a live install attempt failed with DNS resolution errors even with enable_internet=true). A transformers-native GGUF loader was considered as an install-free alternative, but transformers doesn't support the qwen3moe architecture for GGUF loading yet.

Fix: a prebuilt CUDA wheel (`llama_cpp_python-0.3.31-py3-none-manylinux_2_35_x86_64.whl`, cu125 -- CUDA is runtime-backward-compatible, so this runs fine against Kaggle's newer CUDA), staged as the small Kaggle Dataset `agentic-repl-llama-cpp-wheel` and installed offline via `pip install --no-index`. This has to run before `make_solver()` below, not after like the HRM dependency cell, since AgenticReplSolver's constructor loads the model eagerly (unlike HRMSolver, which defers its heavy imports).

In [ ]:
agentic_setup_ok = True

if SOLVER_NAME == 'agentic_repl':
    import subprocess
    try:
        import llama_cpp
        print('llama_cpp already importable:', llama_cpp.__file__)
    except ImportError:
        wheel_dir_candidates = (
            list(Path('/kaggle/input/agentic-repl-llama-cpp-wheel').glob('*'))
            + list(Path('/kaggle/input').glob('datasets/*/agentic-repl-llama-cpp-wheel/*'))
        )
        wheel_files = [p for p in wheel_dir_candidates if p.suffix == '.whl']
        print('found wheel files:', wheel_files)
        if not wheel_files:
            print('WARNING: no staged llama-cpp-python wheel found under /kaggle/input')
            agentic_setup_ok = False
        else:
            install = subprocess.run(
                [sys.executable, '-m', 'pip', 'install', '-v', '--no-index',
                 '--find-links', str(wheel_files[0].parent), 'llama-cpp-python'],
                capture_output=True, text=True, timeout=300,
            )
            print('--- pip stdout (tail) ---')
            print(install.stdout[-3000:])
            if install.returncode != 0:
                print('--- pip stderr (tail) ---')
                print(install.stderr[-3000:])
                agentic_setup_ok = False
            else:
                try:
                    import llama_cpp
                    print('llama_cpp installed and importable:', llama_cpp.__file__)
                except ImportError as exc:
                    print('WARNING: pip install succeeded but import still fails:', repr(exc))
                    agentic_setup_ok = False

    if not agentic_setup_ok:
        print('Disabling agentic_repl for this run; falling back to SOLVER_NAME=pipeline.')
        SOLVER_NAME = 'pipeline'

print('agentic_setup_ok =', agentic_setup_ok)
print('SOLVER_NAME (post agentic_repl setup) =', SOLVER_NAME)


## 5. Download Models, Optionally Train Local Stages, Then Load Solver

In [ ]:
if AUTO_DOWNLOAD_GIT_CODE:
    git_download = download_git_code_repositories(apply=True)
    print('git_code_download =')
    print(json.dumps(git_download, indent=2))

if AUTO_DOWNLOAD_HF_MODELS:
    hf_download = download_huggingface_model_inputs(apply=True)
    print('huggingface_model_download =')
    print(json.dumps(hf_download, indent=2))

if AUTO_DOWNLOAD_DIRECT_CHECKPOINTS:
    direct_download = download_direct_checkpoint_inputs(apply=True)
    print('direct_checkpoint_download =')
    print(json.dumps(direct_download, indent=2))

if AUTO_DISCOVER_MODELS:
    discovery = autodiscover_model_inputs(apply=True)
    print('model_autodiscovery =')
    print(json.dumps(discovery, indent=2))

training_results = {}
if RUN_TRAINING_STAGES:
    CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
    if TRAIN_JEPA_PROJECTION:
        projection_result = train_jepa_projection(
            tasks.values(),
            checkpoint_path=IJEPA_PROJECTION_OUTPUT,
            config=JepaProjectionConfig(input_dim=1280, output_dim=768),
            steps=JEPA_PROJECTION_STEPS,
            lr=1e-3,
        )
        os.environ['IJEPA_PROJECTION_CHECKPOINT_PATH'] = str(IJEPA_PROJECTION_OUTPUT)
        training_results['jepa_projection'] = projection_result.to_dict()
    if TRAIN_WORLD_MODEL:
        world_result = train_world_model(
            tasks.values(),
            checkpoint_path=WORLD_MODEL_OUTPUT,
            config=WorldModelConfig(z_dim=768, rule_dim=4, hidden_dim=3072),
            steps=WORLD_MODEL_STEPS,
            lr=1e-3,
        )
        os.environ['WORLD_MODEL_CHECKPOINT_PATH'] = str(WORLD_MODEL_OUTPUT)
        training_results['world_model'] = world_result.to_dict()
    if RUN_TTT_SMOKE:
        ttt_result = run_ttt_lora_smoke(
            rank=16,
            steps=TTT_SMOKE_STEPS,
            checkpoint_path=TTT_LORA_OUTPUT,
        )
        os.environ['TTT_LORA_CHECKPOINT_PATH'] = str(TTT_LORA_OUTPUT)
        training_results['ttt_lora_smoke'] = ttt_result.to_dict()
else:
    print('RUN_TRAINING_STAGES is False; using downloaded/discovered checkpoints only.')

if training_results:
    print('training_results =')
    print(json.dumps(training_results, indent=2, sort_keys=True))

solver = make_solver(SOLVER_NAME, model_mode=MODEL_MODE)
print('Loaded solver:', solver.__class__.__name__)

if hasattr(solver, 'pipeline'):
    print('model_registry =')
    print(json.dumps(solver.pipeline.model_registry.summary(), indent=2))


## 5b. Install HRM Runtime Dependencies

The external HRM repo needs packages Kaggle's base image doesn't ship (`flash-attn` in particular has no fallback attention path). This cell installs them best-effort; if anything critical fails, it disables real HRM inference and falls back to the deterministic pipeline solver rather than risk a hung or crashed GPU session.

In [ ]:
hrm_setup_ok = True
hrm_setup_log = []

if SOLVER_NAME == 'hrm':
    import subprocess
    import torch as _torch_probe

    print('torch =', _torch_probe.__version__, '| cuda =', _torch_probe.version.cuda,
          '| cuda_available =', _torch_probe.cuda.is_available(),
          '| device_count =', _torch_probe.cuda.device_count())
    if _torch_probe.cuda.is_available():
        print('gpu_name =', _torch_probe.cuda.get_device_name(0))

    hrm_dependencies = [
        'einops', 'tqdm', 'coolname', 'pydantic', 'argdantic', 'wandb',
        'omegaconf', 'hydra-core', 'huggingface_hub', 'pyyaml',
        # adam-atan2's setup.py uses the legacy setup_requires=['setuptools_scm']
        # auto-fetch path, which pulls a setuptools_scm version whose own
        # vcs_versioning dependency doesn't resolve through that legacy mechanism.
        # Installing setuptools_scm normally first lets adam-atan2's build find it
        # already satisfied and skip the broken auto-fetch.
        'setuptools_scm', 'adam-atan2',
    ]
    # Kaggle's scored rerun is internet-disabled, so PyPI is unreachable too --
    # confirmed by a real run: pydantic/pyyaml/etc already ship in Kaggle's base
    # image (installs no-op fine offline), but coolname/argdantic/hydra-core/
    # setuptools_scm/adam-atan2 don't and failed outright with no internet. Their
    # wheels (pure-python only; platform-specific ones like pydantic-core were
    # deliberately excluded since those packages are already present) are
    # pre-staged in the same dataset as the checkpoint.
    wheelhouse = Path('/kaggle/input/hrm-arc2-checkpoint/wheels')
    print('wheelhouse =', wheelhouse, '| is_dir =', wheelhouse.is_dir())
    hrm_dataset_root = Path('/kaggle/input/hrm-arc2-checkpoint')
    if hrm_dataset_root.is_dir():
        print('hrm-arc2-checkpoint dataset contents:', sorted(p.name for p in hrm_dataset_root.iterdir()))
    else:
        print('hrm-arc2-checkpoint dataset root not found; listing /kaggle/input:')
        print(sorted(p.name for p in Path('/kaggle/input').iterdir()) if Path('/kaggle/input').is_dir() else 'no /kaggle/input')
    offline_install_args = ['--no-index', '--find-links', str(wheelhouse)] if wheelhouse.is_dir() else []

    def _resolve_install_target(package):
        # Resolve to the exact staged .whl instead of relying on pip's --find-links
        # name-based matching: confirmed on a real run that Kaggle's pip 24.1.2 fails
        # to match 'adam-atan2' against a staged sdist by name. The glob is restricted
        # to *.whl files specifically -- Kaggle auto-extracts uploaded .tar.gz archives
        # into a same-named bare directory (confirmed: an old 'adam_atan2-0.0.3/' dir
        # from a superseded dataset version was still present and, unfiltered, sorted
        # ahead of the real wheel and got picked instead), so only .whl is ever staged.
        if not wheelhouse.is_dir():
            return package
        normalized = package.replace('-', '_')
        candidates = sorted(wheelhouse.glob(f'{normalized}-*.whl')) or sorted(wheelhouse.glob(f'{package}-*.whl'))
        return str(candidates[0]) if candidates else package

    failed_packages = []
    for package in hrm_dependencies:
        # Install one at a time: a single bad package must not take down the
        # whole batch and hide which of the others would have installed fine.
        try:
            install = subprocess.run(
                # -v: pip swallows the build subprocess's own traceback by default
                # (even without -q) and only shows its own generic wrapper error;
                # -v is what actually surfaces why a legacy setup.py build failed.
                [sys.executable, '-m', 'pip', 'install', '-v', *offline_install_args, _resolve_install_target(package)],
                capture_output=True, text=True, timeout=300,
            )
            hrm_setup_log.append({'step': f'pip_install:{package}', 'returncode': install.returncode})
            if install.returncode != 0:
                failed_packages.append(package)
                print(f'WARNING: failed to install {package}:')
                print('--- stdout (tail) ---')
                print(install.stdout[-4000:])
                print('--- stderr (tail) ---')
                print(install.stderr[-4000:])
        except subprocess.TimeoutExpired:
            failed_packages.append(package)
            hrm_setup_log.append({'step': f'pip_install:{package}', 'error': 'timed out after 300s'})
            print(f'WARNING: installing {package} timed out after 300s')
    if failed_packages:
        hrm_setup_ok = False
        print('WARNING: HRM dependency install failed for:', failed_packages)

    if hrm_setup_ok:
        # flash-attn has no prebuilt wheel matching Kaggle's exact torch build and a
        # from-source compile was observed taking 100+ minutes without finishing (nvcc
        # compiling many CUDA template instantiations, no fast path available). HRM's
        # Attention.forward() only calls flash_attn_func(q=, k=, v=, causal=) with plain
        # [batch, seq_len, heads, head_dim] tensors -- torch's own built-in
        # scaled_dot_product_attention implements the same math and ships with the stock
        # torch Kaggle already has installed, so no extra install/compile is needed at all.
        import types

        def _flash_attn_func(q, k, v, causal=False, softmax_scale=None, dropout_p=0.0, **_ignored):
            q_, k_, v_ = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
            num_heads, num_kv_heads = q_.shape[1], k_.shape[1]
            if num_kv_heads and num_kv_heads != num_heads:
                repeat = num_heads // num_kv_heads
                k_ = k_.repeat_interleave(repeat, dim=1)
                v_ = v_.repeat_interleave(repeat, dim=1)
            out = _torch_probe.nn.functional.scaled_dot_product_attention(
                q_, k_, v_, dropout_p=dropout_p, is_causal=causal, scale=softmax_scale,
            )
            # .contiguous(): .transpose() is a view (stride swap only); real
            # flash_attn_func returns memory already laid out this way, and the
            # model's own code does a plain .view() right after this call, which
            # requires contiguous memory (confirmed failure: 'Cannot view a tensor
            # with shape ... and strides ...' -- that's the exact non-contiguous
            # symptom -- .reshape() would also work but .contiguous() matches what
            # the real function actually hands back).
            return out.transpose(1, 2).contiguous()

        _flash_attn_shim = types.ModuleType('flash_attn')
        _flash_attn_shim.flash_attn_func = _flash_attn_func
        sys.modules['flash_attn'] = _flash_attn_shim
        hrm_setup_log.append({'step': 'flash_attn_shim', 'note': 'using torch.nn.functional.scaled_dot_product_attention, no compile'})
        print('Installed flash_attn shim backed by scaled_dot_product_attention (no compile needed)')

        # adam_atan2's compiled CUDA/C++ backend (adam_atan2_backend) is only built
        # when torch is present at pip-build time; our offline wheel was built without
        # torch available, so it's missing and 'import pretrain' fails outright (it
        # imports AdamATan2 unconditionally to build the optimizer in init_train_state,
        # even though .step() is never called in this eval-only flow). Provide a
        # faithful pure-PyTorch implementation of the same fused update instead of a
        # whole extra Kaggle round-trip just to compile one C extension.
        def _adam_atan2_cuda_impl_(params, grads, exp_avgs, exp_avg_sqs, state_steps, lr, beta1, beta2, weight_decay):
            for param, grad, exp_avg, exp_avg_sq, step in zip(params, grads, exp_avgs, exp_avg_sqs, state_steps):
                if weight_decay != 0:
                    param.mul_(1 - lr * weight_decay)
                exp_avg.mul_(beta1).add_(grad, alpha=1 - beta1)
                exp_avg_sq.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                bias_correction1 = 1 - beta1 ** step.item()
                bias_correction2 = 1 - beta2 ** step.item()
                numerator = exp_avg / bias_correction1
                denominator = (exp_avg_sq / bias_correction2).sqrt()
                param.add_(_torch_probe.atan2(numerator, denominator), alpha=-lr)

        _adam_atan2_backend_shim = types.ModuleType('adam_atan2_backend')
        _adam_atan2_backend_shim.adam_atan2_cuda_impl_ = _adam_atan2_cuda_impl_
        sys.modules['adam_atan2_backend'] = _adam_atan2_backend_shim
        hrm_setup_log.append({'step': 'adam_atan2_backend_shim', 'note': 'pure-PyTorch AdamATan2 update, no compile'})
        print('Installed adam_atan2_backend shim (pure PyTorch, no compile needed)')

    if not hrm_setup_ok:
        print('Disabling real HRM inference for this run; falling back to SOLVER_NAME=pipeline.')
        SOLVER_NAME = 'pipeline'
        ENABLE_REAL_HRM_INFERENCE = False
        os.environ['MYTHOS_ENABLE_REAL_HRM'] = '0'
        solver = make_solver(SOLVER_NAME, model_mode=MODEL_MODE)

print('hrm_setup_ok =', hrm_setup_ok)
print('hrm_setup_log =', json.dumps(hrm_setup_log, indent=2))
print('SOLVER_NAME (post-setup) =', SOLVER_NAME)


## 6. Run Plan-Aligned Pipeline

In [ ]:
from mythos.arc import copy_grid
from mythos.solvers.base import make_prediction
from mythos.solvers.baseline import BaselineSolver

def solve_with_fallback(solver, fallback_solver, task):
    # A Kaggle rerun must always produce a submission.json; one task raising
    # must never abort the loop and discard every already-solved prediction.
    try:
        return solver.solve(task)
    except Exception as exc:
        print(f'WARNING: {task.id} failed with {solver.__class__.__name__}: {exc!r}; using baseline fallback')
    try:
        return fallback_solver.solve(task)
    except Exception as exc:
        print(f'WARNING: {task.id} baseline fallback also failed: {exc!r}; using trivial prediction')
    attempts = [(copy_grid(example.input), [[0]]) for example in task.test]
    return make_prediction(task, attempts)

started = time.perf_counter()
predictions = []
fallback_solver = solver if isinstance(solver, BaselineSolver) else BaselineSolver()

if SOLVER_NAME == 'hrm':
    from mythos.solvers.hrm import HRMEnvironment, HRMInferenceRunner, HRMTTTRunner, TTTConfig
    from mythos.solvers.symbolic import SymbolicSolver
    from mythos.kaggle_run import parse_ensemble_transforms, solve_symbolic_first
    symbolic_predictions, remaining_tasks = solve_symbolic_first(list(tasks.values()))
    print(f'symbolic solver: {len(symbolic_predictions)}/{len(tasks)} tasks solved with a train-verified rule; {len(remaining_tasks)} sent to HRM')
    try:
        env = HRMEnvironment.from_env()
        env.validate(require_cuda=True)
        if os.environ.get('MYTHOS_ENABLE_TTT') == '1':
            hrm_runner = HRMTTTRunner(
                env,
                ttt=TTTConfig(
                    rank=int(os.environ.get('MYTHOS_TTT_RANK', '16')),
                    steps=int(os.environ.get('MYTHOS_TTT_STEPS', '20')),
                    lr=float(os.environ.get('MYTHOS_TTT_LR', '1e-3')),
                    batch_size=int(os.environ.get('MYTHOS_TTT_BATCH_SIZE', '2')),
                    genie_weight=float(os.environ.get('MYTHOS_TTT_GENIE_WEIGHT', '0.1')),
                    ensemble_transforms=parse_ensemble_transforms(os.environ.get('MYTHOS_TTT_ENSEMBLE', '0')),
                ),
                num_aug=int(os.environ.get('MYTHOS_TTT_NUM_AUG', '0')),
            )
        else:
            hrm_runner = HRMInferenceRunner(env)
        hrm_predictions = hrm_runner.solve_tasks(remaining_tasks) if remaining_tasks else []
        print(f'Batched HRM solved {len(hrm_predictions)} task(s)')
    except Exception as exc:
        import traceback
        print(f'WARNING: HRM batch run failed: {exc!r}; using baseline fallback for all tasks')
        print('--- full traceback ---')
        traceback.print_exc()
        if hasattr(exc, 'stdout') and exc.stdout:
            print('--- subprocess stdout (tail) ---')
            print(exc.stdout[-4000:])
        if hasattr(exc, 'stderr') and exc.stderr:
            print('--- subprocess stderr (tail) ---')
            print(exc.stderr[-4000:])
        hrm_predictions = [fallback_solver.solve(task) for task in remaining_tasks]
    hrm_predictions_by_id = {prediction.task_id: prediction for prediction in hrm_predictions}
    predictions = [symbolic_predictions.get(task.id) or hrm_predictions_by_id[task.id] for task in tasks.values()]
elif SOLVER_NAME == 'agentic_repl':
    # Compose, don't compete: symbolic never guesses wrong on train (verify-
    # before-trust), so running it first can only add coverage on top of
    # whatever agentic_repl solves independently, never take away from it --
    # same reasoning as the existing symbolic+HRM composition above. A real
    # benchmark (v55, 50 ARC-AGI-2 training tasks) got agentic_repl alone to
    # 1/50 (2.0%) exact matches, already close to the symbolic solver's own
    # documented 25/1000 (2.5%) -- composing both is the strongest path to
    # beating either alone, since their solved-task sets aren't the same.
    from mythos.kaggle_run import solve_symbolic_first
    symbolic_predictions, remaining_tasks = solve_symbolic_first(list(tasks.values()))
    print(f'symbolic solver: {len(symbolic_predictions)}/{len(tasks)} tasks solved with a train-verified rule; {len(remaining_tasks)} sent to agentic_repl')
    agentic_predictions_by_id = {}
    for index, task in enumerate(remaining_tasks, start=1):
        prediction = solve_with_fallback(solver, fallback_solver, task)
        agentic_predictions_by_id[task.id] = prediction
        if index <= 3 or index == len(remaining_tasks):
            print(f'{index}/{len(remaining_tasks)} (agentic_repl pass) solved: {task.id}')
    predictions = [symbolic_predictions.get(task.id) or agentic_predictions_by_id[task.id] for task in tasks.values()]
else:
    for index, task in enumerate(tasks.values(), start=1):
        prediction = solve_with_fallback(solver, fallback_solver, task)
        predictions.append(prediction)
        if index <= 3 or index == len(tasks):
            print(f'{index}/{len(tasks)} solved: {task.id}')

write_submission(predictions, OUTPUT_PATH)
elapsed = time.perf_counter() - started

print('Wrote submission:', OUTPUT_PATH)
print('tasks_predicted =', len(predictions))
print('elapsed_seconds =', round(elapsed, 3))

if hasattr(solver, 'last_trace') and solver.last_trace is not None:
    print('last_pipeline_trace =')
    print(json.dumps(solver.last_trace.to_dict(), indent=2))


## 7. Validate Submission and Score When Solutions Exist

In [ ]:
submission = load_submission(OUTPUT_PATH)
submission_items = sum(len(outputs) for outputs in submission.values())

print('submission_tasks =', len(submission))
print('submission_test_items =', submission_items)
print('sample_task_id =', next(iter(submission)))

if solution_path is not None:
    if _max_tasks:
        # score_files requires every solved task to be present (correct for a
        # real full run); a MYTHOS_MAX_TASKS test run only solved a subset, so
        # score just that subset instead of letting it raise on the rest.
        all_solutions = load_solutions(solution_path)
        partial_solutions = {task_id: all_solutions[task_id] for task_id in submission if task_id in all_solutions}
        score = score_submission_data(submission, partial_solutions)
        print(f'score (partial: {len(partial_solutions)}/{len(all_solutions)} tasks, MYTHOS_MAX_TASKS set) =')
    else:
        score = score_files(str(OUTPUT_PATH), str(solution_path))
        print('score =')
    print(json.dumps(score.to_dict(), indent=2, sort_keys=True))

    # DIAGNOSTIC: print predicted grids next to the true answer for two
    # representative tasks, to see the real failure mode directly instead of
    # guessing from aggregate metrics alone (cell accuracy alone can't say
    # whether it's a wrong output shape, systematically wrong colors, or
    # something else). One task has train output shapes that disagree
    # (output_shape_hint defers to the model's own EOS markers); one has
    # train output shapes that all agree (shape is never in question, so any
    # remaining failure is purely about predicted grid content).
    solutions_for_diagnostic = partial_solutions if _max_tasks else load_solutions(solution_path)
    candidate_ids = [tid for tid in submission if tid in solutions_for_diagnostic]
    def _train_shapes_agree(task_id):
        shapes = {(len(ex.output), len(ex.output[0])) for ex in tasks[task_id].train if ex.output is not None}
        return len(shapes) == 1
    diagnostic_task_ids = []
    variable_shape_id = next((tid for tid in candidate_ids if not _train_shapes_agree(tid)), None)
    fixed_shape_id = next((tid for tid in candidate_ids if _train_shapes_agree(tid)), None)
    if variable_shape_id is not None:
        diagnostic_task_ids.append(('variable-train-shape', variable_shape_id))
    if fixed_shape_id is not None:
        diagnostic_task_ids.append(('fixed-train-shape', fixed_shape_id))
    for label, diagnostic_task_id in diagnostic_task_ids:
        truth = solutions_for_diagnostic[diagnostic_task_id]
        preds_for_task = submission[diagnostic_task_id]
        source_task = tasks[diagnostic_task_id]
        print(f'--- diagnostic ({label}): task {diagnostic_task_id} ---')
        for demo_index, example in enumerate(source_task.train):
            print(f'train[{demo_index}] input =', example.input)
            print(f'train[{demo_index}] output =', example.output)
        for item_index, (prediction, truth_grid) in enumerate(zip(preds_for_task, truth)):
            print(f'test[{item_index}] input =', source_task.test[item_index].input)
            print(f'test[{item_index}] attempt_1 =', prediction.attempt_1)
            print(f'test[{item_index}] attempt_2 =', prediction.attempt_2)
            print(f'test[{item_index}] true output =', truth_grid)
else:
    print('No solutions file for this split; skipping score.')


## 8. Optional HRM Smoke Test

In [ ]:
if RUN_HRM_SMOKE:
    from mythos.hrm_dataset import prepare_hrm_raw_dataset
    from mythos.solvers.hrm import HRMEnvironment

    env = HRMEnvironment.from_env()
    env.validate(require_cuda=True)
    modules = env.import_modules()
    checkpoint = env.load_checkpoint()
    raw_dir = prepare_hrm_raw_dataset(tasks.values(), Path('/kaggle/working/hrm_smoke/raw/ARC-AGI-2/data'))

    print('HRM repo:', env.repo_dir)
    print('HRM checkpoint:', env.checkpoint_path)
    print('Imported modules:', sorted(modules))
    print('Checkpoint type:', type(checkpoint).__name__)
    print('Prepared HRM raw data:', raw_dir)
else:
    print('RUN_HRM_SMOKE is False; skipping HRM smoke test.')
